<a href="https://colab.research.google.com/github/PedroDiz/TESE_FCUL_2026/blob/main/voice_command_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ─────────────────────────────────────────────────────
# 📖 PROJECT EXPLANATION & MOUNT DRIVE
# ─────────────────────────────────────────────────────

<a id="project"></a>
## <center>Project Description</center>

This is a project to detect the 10 following 1 second voice commands sampled at 16kHz :

yes, no, up, down, left, right, on, off, stop, go

The code also recognizes if there is silence or if the word is not recognized.

<a id="data_provided"></a>
### Data
The training data is provided by the Google Brain challenge named: TensorFlow Speech Recognition Challenge and contains about 23k samples of the 10 voice commands mentioned above and a total of 64721 utterances in total. In addition to the command voices, there are 20 other word utterances to be used as sample other words that might be mentioned and need to be categorized as unknown words. All the data has been divided to 70% for training and 30% to validation in this work.

The data is gathered by thousands of volunteers across the world.

Also, there have been sample noises provided which were randomly selected and added to the train and validation sets to make the train and validation more real world like scenarios.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!ls /content/drive/MyDrive

<a id="mslfb"></a>
## <center>Mel-scaled log filter-bank</center>

<a id="why_mslfb"></a>
### Why MS-LFB

Based on the book  “Automatic Speech Recognition: A Deep Learning Approach.” at reference [1], the most used raw features for speech recognition are Mel Frequency Cepstral Coefficients (MFCC) and Perceptual Linear Predictive (PLP) which both are derived from Mel-Scaled Log Filter-Bank features (MS-LFB). Experiments by this book are showing that MS-LFB can outperform MFCC when tested on a full multiple word speech with a relative Word Error Rate (WER) improvement of 4.4% . So MS-LFB is considered as a good candidate to be used as a feature for this work.

<a id="spectrum_fft"></a>
### Spectrogram of a Signal

To calculate the MS-LFB of a signal we first need to calculate its spectrum. Since frequency content of voices are changing over time and they are not periodic signals with the fixed frequency content, their Discrete Fourier Transform (DFT) can not be calculated directly on the entire signal. So a work around is to consider only a small portion (window of tens of ms) of the voice as stationary with fixed frequencies. i.e we can find the frequency content of each small window and show the frequency content as they are changing over these small windows. One issue with this approach is the edges of these selected windows which will have sharp edges and introduce unwanted high frequency content. To alleviate this problem a filter can be multiplied to each window to reduce the value of the edges of the signal in each window. Lastly, to make sure that there is not lost data in between windows, a portion of the selected windows are overlapped. A good representation of this process can be found in reference [2] and it is shown here:

<a id="mslfb_banks"></a>
### MS-LFB Frequency Filter Banks

The idea of Mel-Scaling is that the human ear can hear different frequency resolutions at different frequencies. In other words, we are more sensitive to the lower range of our hearing range as compared to its higher frequencies. So a scaling method is developed to change the frequencies to Mel_Scale which is based on having the same human ear resolution at all scales. The formula used to convert the frequency in Hz to Mel-Scale are the following:

m = 2959 * log10 ( 1 + freq / 700)

Here is a representation of this scaling which shows the conversion between frequency and Mel Scale for an utterance that has upto 8KHz frequency content.

To find the MS-LFB, a set of filters have been developed that are multiplied with the squared magnitude of the FFT results of each window of the signal. These filters add the frequency energy of the signal at each Mel_Scaled bin and only present the signal at each window with the values at these bins of frequency energy. A full signal would therefore be represented by the number of these filters times the number of windows for the length of the signal.
Here is a set of 40 filters used for a signal that has been sampled at 16Khz and has frequency content up to 8kHz.



<a id="mfcc"></a>
## <center>MFCC (Mel Frequency Cepstral Coefficients)</center>

<a id="mfcc_mslfb"></a>
### Difference Between MFCC and MS-LFB

MFCC is based on MS-LFB with an addition of a Discrete Cosine Transform (DCT) operation at the end to get the spectrum of the MS-LFB coefficients. This is because there are some correlation in between the MS-LFB values and by applying the DCT and keeping only first 12  coefficients (called cepstral coefficients) we will have a compressed representation of the signal.




<a id="ensemble_model"></a>
## <center>Ensemble Model</center>

<a id="combining"></a>
### Combining the Models

To improve the accuracy of the predictions, an ensemble of the two models mentioned above has been used. There are 3 methods to combine the models and create an ensemble which are explained with details in reference [1].

Here is a short description of the three methods:
1. The features obtained from the utterances can be combined together and fed into a model,
2. There could be two models using the features separately and a final portion of the model could combine the results
3. Each model could make its own decision and the results could be looked at with a voting mechanism.

Here the 2nd method has been used where a portion of the models are shared and a final layer will make the last call on what should be predicted.


# ─────────────────────────────────────────────────────
# 📁 0. SETUP & IMPORTS
# ─────────────────────────────────────────────────────

In [ ]:
!mkdir -p /content/dataset
!cp -r /content/drive/MyDrive/speech_commands_v0.01.tar.gz /content/drive/MyDrive/results /content/dataset

In [ ]:
!tar -xvzf /content/dataset/speech_commands_v0.01.tar.gz -C /content/dataset/

In [ ]:
!cp -r /content/drive/MyDrive/{brecq,bitsplit,lapq,pwlq,qdrop,subsetq} /content/

In [ ]:
!pip install scikit-posthocs

In [ ]:
import sys
import os

sys.path.insert(0, '/content/brecq')
sys.path.insert(0, '/content/brecq/linklink')
sys.path.insert(0, '/content')

# ----------------------------------------------------------
# 1. Create __init__.py
# ----------------------------------------------------------
with open("/content/brecq/__init__.py", "w") as f:
    f.write("")

# ----------------------------------------------------------
# 2. Patch quant imports
# ----------------------------------------------------------
quant_dir = "/content/brecq/quant"
for fname in os.listdir(quant_dir):
    if not fname.endswith(".py"):
        continue
    fpath = os.path.join(quant_dir, fname)
    with open(fpath, "r") as f:
        content = f.read()
    new_content = content.replace("from quant.", "from brecq.quant.")
    new_content = new_content.replace("import quant.", "import brecq.quant.")
    if new_content != content:
        with open(fpath, "w") as f:
            f.write(new_content)
        print(f"  Patched imports: {fname}")

# ----------------------------------------------------------
# 3. Patch data_utils.py for tuple inputs
# ----------------------------------------------------------
fpath = "/content/brecq/quant/data_utils.py"
with open(fpath, "r") as f:
    content = f.read()

old = """    for i in range(int(cali_data.size(0) / batch_size)):
        cur_inp, cur_out = get_inp_out(cali_data[i * batch_size:(i + 1) * batch_size])"""
new = """    if isinstance(cali_data, (tuple, list)):
        n_samples = cali_data[0].size(0)
    else:
        n_samples = cali_data.size(0)

    for i in range(int(n_samples / batch_size)):
        if isinstance(cali_data, (tuple, list)):
            batch = tuple(x[i * batch_size:(i + 1) * batch_size] for x in cali_data)
        else:
            batch = cali_data[i * batch_size:(i + 1) * batch_size]
        cur_inp, cur_out = get_inp_out(batch)"""
content = content.replace(old, new)

old2 = """    for i in range(int(cali_data.size(0) / batch_size)):
        cur_grad = get_grad(cali_data[i * batch_size:(i + 1) * batch_size])"""
new2 = """    if isinstance(cali_data, (tuple, list)):
        n_samples = cali_data[0].size(0)
    else:
        n_samples = cali_data.size(0)

    for i in range(int(n_samples / batch_size)):
        if isinstance(cali_data, (tuple, list)):
            batch = tuple(x[i * batch_size:(i + 1) * batch_size] for x in cali_data)
        else:
            batch = cali_data[i * batch_size:(i + 1) * batch_size]
        cur_grad = get_grad(batch)"""
content = content.replace(old2, new2)

old3 = """        with torch.no_grad():
            try:
                _ = self.model(model_input.to(self.device))
            except StopForwardException:
                pass

            if self.asym:
                # Recalculate input with network quantized
                self.data_saver.store_output = False
                self.model.set_quant_state(weight_quant=True, act_quant=self.act_quant)
                try:
                    _ = self.model(model_input.to(self.device))
                except StopForwardException:
                    pass
                self.data_saver.store_output = True"""
new3 = """        def run_model(inp):
            if isinstance(inp, (tuple, list)):
                return self.model(tuple(x.to(self.device) for x in inp))
            return self.model(inp.to(self.device))

        with torch.no_grad():
            try:
                _ = run_model(model_input)
            except StopForwardException:
                pass

            if self.asym:
                self.data_saver.store_output = False
                self.model.set_quant_state(weight_quant=True, act_quant=self.act_quant)
                try:
                    _ = run_model(model_input)
                except StopForwardException:
                    pass
                self.data_saver.store_output = True"""
content = content.replace(old3, new3)

with open(fpath, "w") as f:
    f.write(content)
print("  Patched: data_utils.py")

# ----------------------------------------------------------
# 4. PATCH quant_model.py — disable BN folding
# ----------------------------------------------------------
fpath = "/content/brecq/quant/quant_model.py"
with open(fpath, "r") as f:
    content = f.read()

old = "        search_fold_and_remove_bn(model)"
new = "        pass  # search_fold_and_remove_bn disabled — Conv(+ReLU)->BN order"
content = content.replace(old, new)

with open(fpath, "w") as f:
    f.write(content)
print("  Patched: quant_model.py (BN folding disabled)")

# ----------------------------------------------------------
# 5. PATCH quant_layer.py — fix near-zero delta
# ----------------------------------------------------------
fpath = "/content/brecq/quant/quant_layer.py"
with open(fpath, "r") as f:
    content = f.read()

old = """                if delta < 1e-8:
                    warnings.warn('Quantization range close to zero: [{}, {}]'.format(x_min, x_max))
                    delta = 1e-8"""
new = """                if delta < 1e-8:
                    delta = 1.0
                    zero_point = 0"""
content = content.replace(old, new)

with open(fpath, "w") as f:
    f.write(content)
print("  Patched: quant_layer.py (near-zero delta fixed)")

# ----------------------------------------------------------
# 6. Mock linklink
# ----------------------------------------------------------
from unittest.mock import MagicMock
sys.modules['linklink'] = MagicMock()
print("  Mocked: linklink")

# ----------------------------------------------------------
# 7. Force reload all brecq modules from disk
# ----------------------------------------------------------
brecq_modules = [k for k in sys.modules if 'brecq' in k]
for m in brecq_modules:
    del sys.modules[m]
print(f"  Cleared {len(brecq_modules)} cached brecq modules")

# ----------------------------------------------------------
# 8. Re-import fresh
# ----------------------------------------------------------
from brecq.quant.quant_model import QuantModel
from brecq.quant.quant_block import BaseQuantBlock
from brecq.quant.quant_layer import QuantModule
from brecq.quant.layer_recon import layer_reconstruction

print("\nAll patches applied and modules reloaded ✓")

In [ ]:
import sys
import os

# ==========================================================
# 1. PATHS
# ==========================================================
sys.path.insert(0, '/content/qdrop')
sys.path.insert(0, '/content/qdrop/qdrop')
sys.path.insert(0, '/content')

# ==========================================================
# 2. CREATE __init__.py FILES
# ==========================================================
for path in [
    "/content/qdrop/__init__.py",
    "/content/qdrop/qdrop/__init__.py",
    "/content/qdrop/qdrop/quantization/__init__.py",
    "/content/qdrop/qdrop/solver/__init__.py",
    "/content/qdrop/qdrop/model/__init__.py",
]:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        f.write("")
    print(f"  Created: {path}")

# ==========================================================
# 3. PATCH IMPORTS
# ==========================================================
def patch_imports(base_dir):
    for root, _, files in os.walk(base_dir):
        for fname in files:
            if not fname.endswith(".py"):
                continue

            fpath = os.path.join(root, fname)
            with open(fpath, "r") as f:
                content = f.read()

            new_content = content

            # ---- internal qdrop imports ----
            new_content = new_content.replace(
                "from qdrop.quantization", "from quantization")
            new_content = new_content.replace(
                "import qdrop.quantization", "import quantization")
            new_content = new_content.replace(
                "from qdrop.model", "from model")
            new_content = new_content.replace(
                "from qdrop.solver", "from solver")

            # ---- imagenet_utils → brecq data_utils ----
            new_content = new_content.replace(
                "from imagenet_utils import DataSaverHook, StopForwardException",
                "from brecq.quant.data_utils import DataSaverHook, StopForwardException"
            )

            # ---- recon import in main_imagenet.py ----
            new_content = new_content.replace(
                "from recon import reconstruction",
                "from solver.recon import reconstruction"
            )

            # ---- fold_bn import ----
            new_content = new_content.replace(
                "from fold_bn import search_fold_and_remove_bn, StraightThrough",
                "from solver.fold_bn import search_fold_and_remove_bn, StraightThrough"
            )
            new_content = new_content.replace(
                "import fold_bn",
                "from solver import fold_bn"
            )

            if new_content != content:
                with open(fpath, "w") as f:
                    f.write(new_content)
                print(f"  Patched: {fpath}")

patch_imports("/content/qdrop/qdrop")

# ==========================================================
# 4. PATCH recon.py — tuple input support
# ==========================================================
recon_path = "/content/qdrop/qdrop/solver/recon.py"

with open(recon_path, "r") as f:
    content = f.read()

# ---- patch save_inp_oup_data ----
old = """        for i in range(int(cali_data.size(0) / bs)):
            try:
                _ = model(cali_data[i * bs: (i + 1) * bs].to(device))
            except StopForwardException:
                pass"""

new = """        if isinstance(cali_data, (tuple, list)):
            n_samples = cali_data[0].size(0)
        else:
            n_samples = cali_data.size(0)

        def run_model(inp):
            if isinstance(inp, (tuple, list)):
                return model(tuple(x.to(device) for x in inp))
            return model(inp.to(device))

        for i in range(int(n_samples / bs)):
            if isinstance(cali_data, (tuple, list)):
                batch = tuple(x[i * bs:(i + 1) * bs] for x in cali_data)
            else:
                batch = cali_data[i * bs:(i + 1) * bs]

            try:
                _ = run_model(batch)
            except StopForwardException:
                pass"""

if old in content:
    content = content.replace(old, new)
    print("  Patched: save_inp_oup_data ✓")
else:
    print("  WARNING: save_inp_oup_data patch not found — check manually")

# ---- patch drop input loop ----
old2 = """        if config.drop_prob < 1.0:
            cur_quant_inp = quant_inp[idx].to(device)
            cur_fp_inp = fp_inp[idx].to(device)
            cur_inp = torch.where(torch.rand_like(cur_quant_inp) < config.drop_prob, cur_quant_inp, cur_fp_inp)
        else:
            cur_inp = quant_inp[idx].to(device)"""

new2 = """        if config.drop_prob < 1.0:
            if isinstance(quant_inp, (tuple, list)):
                cur_quant_inp = tuple(x[idx].to(device) for x in quant_inp)
                cur_fp_inp    = tuple(x[idx].to(device) for x in fp_inp)
                cur_inp = tuple(
                    torch.where(torch.rand_like(q) < config.drop_prob, q, f)
                    for q, f in zip(cur_quant_inp, cur_fp_inp)
                )
            else:
                cur_quant_inp = quant_inp[idx].to(device)
                cur_fp_inp    = fp_inp[idx].to(device)
                cur_inp = torch.where(
                    torch.rand_like(cur_quant_inp) < config.drop_prob,
                    cur_quant_inp, cur_fp_inp
                )
        else:
            if isinstance(quant_inp, (tuple, list)):
                cur_inp = tuple(x[idx].to(device) for x in quant_inp)
            else:
                cur_inp = quant_inp[idx].to(device)"""

if old2 in content:
    content = content.replace(old2, new2)
    print("  Patched: drop input loop ✓")
else:
    print("  WARNING: drop input loop patch not found — check manually")

with open(recon_path, "w") as f:
    f.write(content)

# ==========================================================
# 5. VERIFY brecq data_utils exists
# ==========================================================
data_utils_path = '/content/brecq/quant/data_utils.py'
if os.path.exists(data_utils_path):
    print(f"  brecq data_utils found ✓")
else:
    print(f"  WARNING: {data_utils_path} not found!")
    print(f"  Available brecq files:")
    for root, _, files in os.walk('/content/brecq'):
        for f in files:
            print(f"    {os.path.join(root, f)}")

# ==========================================================
# 6. CLEAR CACHED MODULES
# ==========================================================
to_clear = [k for k in sys.modules
            if any(x in k for x in ['qdrop', 'solver', 'quantization',
                                     'fold_bn', 'recon', 'imagenet'])]
for m in to_clear:
    del sys.modules[m]
print(f"  Cleared {len(to_clear)} cached modules ✓")

# ==========================================================
# 7. CLEAN IMPORTS
# ==========================================================
from solver.recon import reconstruction
from solver.fold_bn import search_fold_and_remove_bn, StraightThrough
from quantization.quantized_module import QuantizedLayer, QuantizedBlock
from quantization.state import (enable_calibration_woquantization,
                                 enable_quantization, disable_all)
from quantization.fake_quant import QuantizeBase
from quantization.observer import ObserverBase

print("\nQDrop READY ✓")

# ─────────────────────────────────────────────────────
# 📂 1. DATA PIPELINE
# ─────────────────────────────────────────────────────

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from datetime import datetime
from scipy.io import wavfile
from scipy.fftpack import dct
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import LearningRateScheduler, EarlyStopping, CSVLogger
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Input, Conv2D, Dense, Flatten, Dropout, MaxPooling2D, Concatenate
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.metrics import Precision, Recall
from tqdm import tqdm
from sklearn.utils.class_weight import compute_class_weight

# ---------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------
drive_root = '/content/dataset/'
train_directory = os.path.join(drive_root)
noise_directory = os.path.join(drive_root, '_background_noise_')
main_directory = drive_root
os.makedirs(os.path.join(main_directory, 'results'), exist_ok=True)

noise_coef    = 0.1       # amplitude of noise added to real audio (augmentation)
silence_coef  = 0.1       # amplitude of noise used to synthesize silence samples
drop_out_rate = 0.45      # middle ground between original (0.5) and my fix (0.3)
                         # 0.5 is quite aggressive and can slow convergence
                         # 0.3 is too low for a dataset this size
                         # 0.4 balances regularization and learning speed
num_epochs = 230
num_silences  = 800       # FIX: reduced from 1700 to avoid silence over-representation
audio_length  = 16000
sample_rate   = 16000
batch_size    = 128      # keep your value, smaller batches help generalization
                         # and are safer for Colab RAM

classes = ['down', 'go', 'left', 'no', 'off', 'on',
           'right', 'silence', 'stop', 'unknown', 'up', 'yes']

In [ ]:
def MSLFB(signal_resampled):
    pre_emphasis  = 0.97
    frame_size    = 0.025
    frame_stride  = 0.01
    NFFT          = 512
    nfilt         = 40

    emphasized_signal = np.append(
        signal_resampled[0],
        signal_resampled[1:] - pre_emphasis * signal_resampled[:-1]
    )
    frame_length = int(round(frame_size   * sample_rate))
    frame_step   = int(round(frame_stride * sample_rate))
    signal_length = len(emphasized_signal)
    num_frames    = int(np.ceil(float(np.abs(signal_length - frame_length)) / frame_step))
    pad_signal_length = num_frames * frame_step + frame_length
    z = np.zeros(pad_signal_length - signal_length)
    pad_signal = np.append(emphasized_signal, z)
    indices = (
        np.tile(np.arange(0, frame_length), (num_frames, 1)) +
        np.tile(np.arange(0, num_frames * frame_step, frame_step), (frame_length, 1)).T
    )
    frames = pad_signal[indices.astype(np.int32, copy=False)]
    frames *= np.hamming(frame_length)
    mag_frames = np.absolute(np.fft.rfft(frames, NFFT))
    pow_frames = (1.0 / NFFT) * (mag_frames ** 2)

    low_freq_mel  = 0
    high_freq_mel = 2595 * np.log10(1 + (sample_rate / 2) / 700)
    mel_points    = np.linspace(low_freq_mel, high_freq_mel, nfilt + 2)
    hz_points     = 700 * (10 ** (mel_points / 2595) - 1)
    bin_arr       = np.floor((NFFT + 1) * hz_points / sample_rate)
    fbank         = np.zeros((nfilt, int(np.floor(NFFT / 2 + 1))))
    for m in range(1, nfilt + 1):
        f_m_minus = int(bin_arr[m - 1])
        f_m       = int(bin_arr[m])
        f_m_plus  = int(bin_arr[m + 1])
        for k in range(f_m_minus, f_m):
            fbank[m - 1, k] = (k - bin_arr[m - 1]) / (bin_arr[m] - bin_arr[m - 1])
        for k in range(f_m, f_m_plus):
            fbank[m - 1, k] = (bin_arr[m + 1] - k) / (bin_arr[m + 1] - bin_arr[m])

    filter_banks = np.dot(pow_frames, fbank.T)
    filter_banks = np.where(filter_banks == 0, np.finfo(float).eps, filter_banks)
    filter_banks = 20 * np.log10(filter_banks)
    filter_banks -= (np.mean(filter_banks, axis=0) + 1e-8)
    return abs(filter_banks)


def MFCC(signal_resampled):
    num_ceps   = 12
    filter_banks = MSLFB(signal_resampled)
    mfcc = dct(filter_banks, type=2, axis=1, norm='ortho')[:, 1:(num_ceps + 1)]
    return mfcc


def normalize(arr):
    """FIX: consistent per-sample normalization used in BOTH training and inference."""
    return (arr - np.mean(arr)) / (np.std(arr) + 1e-8)


def step_decay_schedule(initial_lr=1e-3, decay_factor=0.75, step_size=10):
    def schedule(epoch):
        return initial_lr * (decay_factor ** np.floor(epoch / step_size))
    return LearningRateScheduler(schedule)

In [ ]:
# ---------------------------------------------------------
# LOAD NOISE SAMPLES
# ---------------------------------------------------------
noise_samples = []
print("Loading background noise samples...")
for noises in tqdm(sorted(os.listdir(noise_directory))):  # ← sorted
    if not noises.endswith('.wav'):
        continue
    sr, noise = wavfile.read(os.path.join(noise_directory, noises))
    noise_samples.append(noise.astype(np.float32))


# ---------------------------------------------------------
# INDEX FILE PATHS & LABELS
# ---------------------------------------------------------
file_paths = []
labels     = []

known_commands = {'yes', 'no', 'up', 'down', 'left', 'right', 'on', 'off', 'stop', 'go'}

print("Indexing dataset...")
for folder_name in tqdm(sorted(os.listdir(train_directory))):  # ← sorted
    if folder_name == '_background_noise_':
        continue
    folder_path = os.path.join(train_directory, folder_name)
    if not os.path.isdir(folder_path):
        continue
    for wav_file in sorted(os.listdir(folder_path)):  # ← sorted
        if not wav_file.endswith('.wav'):
            continue
        file_paths.append(os.path.join(folder_path, wav_file))
        labels.append(folder_name if folder_name in known_commands else 'unknown')

# Add silence tokens
for _ in range(num_silences):
    file_paths.append('SILENCE')
    labels.append('silence')

In [ ]:
# ---------------------------------------------------------
# DETERMINE FIXED FEATURE SHAPES
# ---------------------------------------------------------
_dummy = np.zeros(audio_length, dtype=np.float32)
MFCC_SHAPE  = MFCC(_dummy).shape    # e.g. (99, 12)
MSLFB_SHAPE = MSLFB(_dummy).shape   # e.g. (99, 40)
print("MFCC_SHAPE:", MFCC_SHAPE)
print("MSLFB_SHAPE:", MSLFB_SHAPE)


In [ ]:
import hashlib

# ---------------------------------------------------------
# PRECOMPUTE FEATURES  (deterministic per file via seed)
# ---------------------------------------------------------
print("Precomputing (normalized) features to disk...")
for fp in tqdm(file_paths):
    if fp == 'SILENCE':
        continue
    mfcc_path  = fp.replace('.wav', '_mfcc.npy')
    mslfb_path = fp.replace('.wav', '_mslfb.npy')
    if os.path.exists(mfcc_path) and os.path.exists(mslfb_path):
        continue
    sr, sample = wavfile.read(fp)
    signal = sample.astype(np.float32)
    if len(signal) < audio_length:
        signal = np.pad(signal, (0, audio_length - len(signal)))
    else:
        signal = signal[:audio_length]

    file_seed   = int(hashlib.md5(fp.encode()).hexdigest(), 16) % (2**32)
    rng         = np.random.RandomState(file_seed)
    noise_idx   = rng.randint(0, len(noise_samples))
    noise_arr   = noise_samples[noise_idx]
    noise_start = rng.randint(0, max(1, len(noise_arr) - audio_length))
    noise_slice = noise_arr[noise_start:noise_start + audio_length]
    if len(noise_slice) < audio_length:
        noise_slice = np.pad(noise_slice, (0, audio_length - len(noise_slice)))
    signal_noisy = signal + noise_coef * noise_slice

    np.save(mfcc_path,  normalize(MFCC(signal_noisy)).astype(np.float32))
    np.save(mslfb_path, normalize(MSLFB(signal_noisy)).astype(np.float32))


# ---------------------------------------------------------
# PRE-GENERATE FIXED SILENCE SAMPLES
# Saved to Drive so they survive session restarts  ← FIXED
# ---------------------------------------------------------
silence_dir = '/content/drive/MyDrive/silence_precomputed'  # ← Drive, not local
os.makedirs(silence_dir, exist_ok=True)

print("Precomputing fixed silence samples...")
for i in tqdm(range(num_silences)):
    mfcc_path  = os.path.join(silence_dir, f'silence_{i}_mfcc.npy')
    mslfb_path = os.path.join(silence_dir, f'silence_{i}_mslfb.npy')
    if os.path.exists(mfcc_path) and os.path.exists(mslfb_path):
        continue  # already on Drive → skip forever
    rng       = np.random.RandomState(i)
    noise_idx = rng.randint(0, len(noise_samples))
    noise_arr = noise_samples[noise_idx]
    start     = rng.randint(0, max(1, len(noise_arr) - audio_length))
    signal    = silence_coef * noise_arr[start:start + audio_length]
    if len(signal) < audio_length:
        signal = np.pad(signal, (0, audio_length - len(signal)))
    np.save(mfcc_path,  normalize(MFCC(signal.astype(np.float32))).astype(np.float32))
    np.save(mslfb_path, normalize(MSLFB(signal.astype(np.float32))).astype(np.float32))

print(f"Fixed silence samples ready: {silence_dir}")

# ---------------------------------------------------------
# ENCODE LABELS
# ---------------------------------------------------------
le          = LabelEncoder()
y_all       = le.fit_transform(labels)
num_classes = len(le.classes_)
print("Classes:", list(le.classes_))

file_paths = np.array(file_paths)
y_all      = np.array(y_all)

# random_state=42 guarantees identical split every session
x_train_files, x_test_files, y_train_labels, y_test_labels = train_test_split(
    file_paths, y_all, test_size=0.3, random_state=42, stratify=y_all
)

# ---------------------------------------------------------
# REPLACE 'SILENCE' IN TEST SET WITH FIXED PATHS
# Points to Drive paths — same files every session  ← FIXED
# ---------------------------------------------------------
silence_counter  = 0
fixed_test_files = []
for fp in x_test_files:
    if fp == 'SILENCE':
        fixed_test_files.append(
            os.path.join(silence_dir, f'silence_{silence_counter}')
        )
        silence_counter += 1
    else:
        fixed_test_files.append(fp)

x_test_files = np.array(fixed_test_files)
print(f"{silence_counter} test silence tokens replaced with fixed Drive paths")


# ---------------------------------------------------------
# CLASS WEIGHTS
# ---------------------------------------------------------
class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_labels),
    y=y_train_labels
)
class_weight_dict = dict(enumerate(class_weights_arr))
print("Class weights:", class_weight_dict)


# ---------------------------------------------------------
# TF.DATA DATASET LOADER
# ---------------------------------------------------------
def load_sample(fp_tensor, label):
    def _load(path_bytes):
        path = path_bytes.decode('utf-8')
        if path == 'SILENCE':
            # Train only — random each epoch for healthy variety
            noise_idx  = np.random.randint(0, len(noise_samples))
            noise_arr  = noise_samples[noise_idx]
            start      = np.random.randint(0, max(1, len(noise_arr) - audio_length))
            signal     = silence_coef * noise_arr[start:start + audio_length]
            if len(signal) < audio_length:
                signal = np.pad(signal, (0, audio_length - len(signal)))
            mfcc_feat  = normalize(MFCC(signal.astype(np.float32))).astype(np.float32)
            mslfb_feat = normalize(MSLFB(signal.astype(np.float32))).astype(np.float32)
        else:
            # Real .wav files → replace extension
            # Fixed silence files (Drive paths) → no .wav extension
            if path.endswith('.wav'):
                mfcc_feat  = np.load(path.replace('.wav', '_mfcc.npy'))
                mslfb_feat = np.load(path.replace('.wav', '_mslfb.npy'))
            else:
                mfcc_feat  = np.load(path + '_mfcc.npy')
                mslfb_feat = np.load(path + '_mslfb.npy')
        return mfcc_feat, mslfb_feat

    mfcc_arr, mslfb_arr = tf.numpy_function(_load, [fp_tensor], [tf.float32, tf.float32])
    mfcc_arr.set_shape(MFCC_SHAPE)
    mslfb_arr.set_shape(MSLFB_SHAPE)

    mfcc_arr  = tf.expand_dims(mfcc_arr,  -1)
    mslfb_arr = tf.expand_dims(mslfb_arr, -1)
    label_oh  = tf.one_hot(label, num_classes)
    return (mfcc_arr, mslfb_arr), label_oh


train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train_files, y_train_labels))
    .shuffle(buffer_size=10000)
    .map(load_sample, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

test_ds = (
    tf.data.Dataset.from_tensor_slices((x_test_files, y_test_labels))
    .map(load_sample, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)


# ─────────────────────────────────────────────────────
# 🏗️ 2. MODEL DEFINITION
# ─────────────────────────────────────────────────────

In [ ]:
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras import regularizers

# ---------------------------------------------------------
# MODEL DEFINITION  (unchanged architecture)
# ---------------------------------------------------------
def create_submodel(input_shape):
    model = Sequential([
        Input(shape=input_shape),
        Conv2D(16, (4,4), padding='valid', activation='relu'),
        BatchNormalization(),
        Dropout(drop_out_rate),
        Conv2D(32, (4,4), padding='valid', activation='relu'),
        BatchNormalization(),
        MaxPooling2D(2,2),
        Dropout(drop_out_rate),
        Conv2D(64, (2,2), padding='same', activation='relu'),
        BatchNormalization(),
        MaxPooling2D(2,2),
        Dropout(drop_out_rate),
        Conv2D(128, (2,2), padding='same', activation='relu'),
        BatchNormalization(),
        Dropout(drop_out_rate),
        Flatten(),
        Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        Dropout(drop_out_rate),
        Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        Dropout(drop_out_rate),
        Dense(12, activation='relu')
    ])
    return model


mfcc_shape  = MFCC_SHAPE  + (1,)
mslfb_shape = MSLFB_SHAPE + (1,)

model_mfcc  = create_submodel(mfcc_shape)
model_mslfb = create_submodel(mslfb_shape)

mfcc_input    = Input(mfcc_shape)
mslfb_input   = Input(mslfb_shape)
encoded_mfcc  = model_mfcc(mfcc_input)
encoded_mslfb = model_mslfb(mslfb_input)
concatenated  = Concatenate(axis=1)([encoded_mfcc, encoded_mslfb])
prediction    = Dense(num_classes, activation='softmax')(concatenated)
ensemble_model = Model(inputs=[mfcc_input, mslfb_input], outputs=prediction)

# ─────────────────────────────────────────────────────
# 🏋️ 3. TRAINING
# ─────────────────────────────────────────────────────

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint
# ---------------------------------------------------------
# TRAINING
# ---------------------------------------------------------
file_date  = datetime.now().strftime("%Y%m%d_%H%M%S")
lr_sched = step_decay_schedule(initial_lr=0.0005, decay_factor=0.94, step_size=5)
csv_logger = CSVLogger(
    os.path.join(main_directory, f'results/csv_logger_{file_date}.csv'),
    separator=",", append=False
)

checkpoint = ModelCheckpoint(
    filepath='/content/drive/MyDrive/results/checkpoint_epoch_{epoch:03d}_val{val_accuracy:.4f}.keras',
    monitor='val_accuracy',
    save_best_only=True,      # only saves when val_accuracy improves
    mode='max',
    verbose=1
)

early_stop = EarlyStopping(
    monitor='val_accuracy',
    min_delta=0.001,      # was 0.002, smaller delta = less strict
    patience=20,          # was 12, triggered too early
    mode='max',
    restore_best_weights=True
)

ensemble_model.compile(
    loss='categorical_crossentropy',
    optimizer=Adam(learning_rate=0.001),
    metrics=['accuracy', Precision(name='precision'), Recall(name='recall')]
)

history = ensemble_model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=num_epochs,
    callbacks=[csv_logger, lr_sched, early_stop, checkpoint],
    class_weight=class_weight_dict,   # FIX: balanced class weights
    verbose=1
)

model_path = os.path.join(main_directory, f'results/ensemble_mfcc_mslfb_{file_date}.keras')
ensemble_model.save(model_path)
ensemble_model.summary()
print(f"\nModel saved to: {model_path}")

# ─────────────────────────────────────────────────────
# 🏋️ 4. TRAINING FROM CHECKPOINT
# ─────────────────────────────────────────────────────

In [ ]:
def get_lr_at_epoch(epoch, initial_lr=0.0005, decay_factor=0.94, step_size=5):
    lr = initial_lr * (decay_factor ** (epoch // step_size))
    print(f"Epoch {epoch} → LR = {lr:.8f}")
    return lr

get_lr_at_epoch(151)  # replace with whatever epoch you stopped at

In [ ]:
import os
import re
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import Precision, Recall
import shutil

# ---------------------------------------------------------
# AUTO-DETECT BEST CHECKPOINT FROM DRIVE
# ---------------------------------------------------------
checkpoint_dir = '/content/drive/MyDrive/results/'

# Find all checkpoint files
checkpoint_files = [
    f for f in os.listdir(checkpoint_dir)
    if f.startswith('checkpoint_epoch_') and f.endswith('.keras')
]

if not checkpoint_files:
    raise FileNotFoundError(
        "No checkpoint files found in Drive results folder.\n"
        "You need to train from scratch."
    )

# Pick the one with the highest val_accuracy from filename
def parse_val_accuracy(filename):
    match = re.search(r'val(\d+\.\d+)\.keras', filename)
    return float(match.group(1)) if match else 0.0

def parse_epoch(filename):
    match = re.search(r'checkpoint_epoch_(\d+)_', filename)
    return int(match.group(1)) if match else 0

best_checkpoint = max(checkpoint_files, key=parse_val_accuracy)
resume_epoch    = parse_epoch(best_checkpoint)
resume_model_path = os.path.join(checkpoint_dir, best_checkpoint)

print(f"Best checkpoint found : {best_checkpoint}")
print(f"Resuming from epoch   : {resume_epoch}")
print(f"Val accuracy in name  : {parse_val_accuracy(best_checkpoint):.4f}")

# ---------------------------------------------------------
# LOAD MODEL
# ---------------------------------------------------------
ensemble_model = load_model(resume_model_path)
print(f"Model loaded ✓")

# ---------------------------------------------------------
# CALLBACKS
# ---------------------------------------------------------
file_date  = datetime.now().strftime("%Y%m%d_%H%M%S")

lr_sched = step_decay_schedule(initial_lr=0.00007813, decay_factor=0.94, step_size=5)

csv_logger = CSVLogger(
    os.path.join(main_directory, f'results/csv_logger_resume_{file_date}.csv'),
    separator=",", append=False
)

checkpoint = ModelCheckpoint(
    filepath='/content/drive/MyDrive/results/checkpoint_epoch_{epoch:03d}_val{val_accuracy:.4f}.keras',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

early_stop = EarlyStopping(
    monitor='val_accuracy',
    min_delta=0.001,
    patience=20,
    mode='max',
    restore_best_weights=True
)

# ---------------------------------------------------------
# RECOMPILE
# ---------------------------------------------------------
# And compile with the same LR
ensemble_model.compile(
    loss='categorical_crossentropy',
    optimizer=Adam(learning_rate=0.00007813),
    metrics=['accuracy', Precision(name='precision'), Recall(name='recall')]
)

# ---------------------------------------------------------
# RESUME TRAINING
# ---------------------------------------------------------
print(f"Resuming training from epoch {resume_epoch} to {num_epochs}...")
history = ensemble_model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=num_epochs,
    initial_epoch=resume_epoch,
    callbacks=[csv_logger, lr_sched, early_stop, checkpoint],
    class_weight=class_weight_dict,
    verbose=1
)

# ---------------------------------------------------------
# SAVE FINAL MODEL
# ---------------------------------------------------------
model_path = os.path.join(main_directory, f'results/ensemble_mfcc_mslfb_resumed_{file_date}.keras')
ensemble_model.save(model_path)
shutil.copy(model_path, f'/content/drive/MyDrive/results/ensemble_mfcc_mslfb_resumed_{file_date}.keras')
print(f"Final model saved to Drive ✓")


# ---------------------------------------------------------
# PLOTS
# ---------------------------------------------------------
loss_train = history.history['loss']
loss_val = history.history['val_loss']
plt.plot(loss_train, 'g', label='Training loss')
plt.plot(loss_val, 'b', label='Validation loss')
plt.title('Ensemble MSLFB and MFCC loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

train_acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
plt.plot(train_acc, 'g', label='Training Accuracy')
plt.plot(val_acc, 'b', label='Validation Accuracy')
plt.title('Ensemble MSLFB and MFCC Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

plt.plot(history.history['precision'], label='Train Precision', color='green')
plt.plot(history.history['val_precision'], label='Validation Precision', color='blue')
plt.title('Precision over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Precision')
plt.legend()
plt.show()

plt.plot(history.history['recall'], label='Train Recall', color='green')
plt.plot(history.history['val_recall'], label='Validation Recall', color='blue')
plt.title('Recall over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Recall')
plt.legend()
plt.show()


# ─────────────────────────────────────────────────────
# ⚙️ 5. HELPER FUNCTIONS
# ─────────────────────────────────────────────────────

In [ ]:
def compile_and_evaluate(model, test_ds, label=""):
    model.compile(
        loss="categorical_crossentropy",
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        metrics=["accuracy"],   # remove Precision/Recall do Keras (eram micro)
    )
    results = model.evaluate(test_ds, verbose=0)

    all_preds, all_labels = [], []
    for x_batch, y_batch in test_ds:
        preds = model(x_batch, training=False)
        all_preds.append(tf.argmax(preds,   axis=1).numpy())
        all_labels.append(tf.argmax(y_batch, axis=1).numpy())
    all_preds  = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    classes = np.unique(all_labels)
    per_class_precision, per_class_recall, per_class_f1 = [], [], []

    for c in classes:
        tp = np.sum((all_preds == c) & (all_labels == c))
        fp = np.sum((all_preds == c) & (all_labels != c))
        fn = np.sum((all_preds != c) & (all_labels == c))

        p  = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        r  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2*p*r/(p+r) if (p+r) > 0 else 0.0

        per_class_precision.append(p)
        per_class_recall.append(r)
        per_class_f1.append(f1)

    precision    = float(np.mean(per_class_precision))
    recall       = float(np.mean(per_class_recall))
    balanced_acc = recall                       # idêntico por definição
    macro_f1     = float(np.mean(per_class_f1))

    print(f"\n{label}")
    print(f"Loss:              {results[0]:.4f}")
    print(f"Accuracy:          {results[1]:.4f}")
    print(f"Balanced Accuracy: {balanced_acc:.4f}")
    print(f"Precision (macro): {precision:.4f}")
    print(f"Recall (macro):    {recall:.4f}")
    print(f"Macro F1:          {macro_f1:.4f}")

    return results, balanced_acc, macro_f1, precision, recall

In [ ]:
# =========================================================
# COLLECT LAYERS RECURSIVELY
# =========================================================
def collect_all_weight_layers(model):
    layers = []
    def _collect(layer):
        if isinstance(layer, (tf.keras.layers.Conv2D, tf.keras.layers.Dense)):
            layers.append(layer)
        if hasattr(layer, "layers"):
            for sub in layer.layers:
                _collect(sub)
    for l in model.layers:
        _collect(l)
    return layers

# =========================================================
# COMPUTE LAYER STATISTICS
# =========================================================
def compute_layer_stats(layer):
    w = layer.get_weights()[0].flatten()
    return {
        'kurtosis': kurtosis(w),
        'skew':     skew(w),
        'std':      np.std(w),
    }


# =========================================================
# UNIFORM PER-CHANNEL QUANTIZATION
# =========================================================
def uniform_quantize_layer(W, num_bits):
    qmin   = -(2 ** (num_bits - 1))
    qmax   =  (2 ** (num_bits - 1)) - 1
    out_ch = W.shape[-1]
    W_r    = W.reshape(-1, out_ch)
    Wq     = np.zeros_like(W_r)

    for c in range(out_ch):
        w_ch  = W_r[:, c]
        w_max = np.max(np.abs(w_ch)) + 1e-8
        scale = w_max / qmax
        Wq[:, c] = np.clip(np.round(w_ch / scale), qmin, qmax) * scale

    return Wq.reshape(W.shape).astype(np.float32)

In [ ]:
# =========================================================
# CAPTURE LAYER ACTIVATIONS — patches instance not class
# =========================================================
def capture_layer_activations(model, layer, calib_ds):
    """
    Capture input and output activations for a specific layer
    by patching the instance's call method directly.
    Patching the instance (layer.call) instead of the class
    (layer.__class__.call) ensures the hook only fires for
    this specific layer, not all layers of the same type.
    """
    captured = {"inp": [], "out": []}
    original_call = layer.call

    def hooked_call(inputs, *args, **kwargs):
        result = original_call(inputs, *args, **kwargs)
        captured["inp"].append(tf.identity(inputs))
        captured["out"].append(tf.identity(result))
        return result

    layer.call = hooked_call  # patch instance, not class

    for batch in calib_ds:
        x_batch, _ = batch
        model(x_batch, training=False)

    layer.call = original_call  # restore

    return captured["inp"], captured["out"]


# =========================================================
# MANUAL LAYER FORWARD (for Conv2D and Dense)
# =========================================================
def manual_forward(layer, x, w, b):
    if isinstance(layer, tf.keras.layers.Conv2D):
        y = tf.nn.conv2d(x, w, strides=layer.strides, padding=layer.padding.upper())
        if b is not None:
            y = tf.nn.bias_add(y, b)
    elif isinstance(layer, tf.keras.layers.Dense):
        y = tf.matmul(x, w)
        if b is not None:
            y = y + b
    if layer.activation is not None:
        y = layer.activation(y)
    return y

In [ ]:
# =========================================================
# LOAD MODEL HELPER
# =========================================================
def load_fresh_model():
    file_date  = "20260302_070412"
    model_path = os.path.join(main_directory, f"results/ensemble_mfcc_mslfb_resumed_{file_date}.keras")
    return load_model(model_path)


# ─────────────────────────────────────────────────────
# 🔄 6. CONSISTENCY CHECKS
# ─────────────────────────────────────────────────────

In [ ]:
import hashlib
import numpy as np
from tensorflow.keras.models import load_model
from datetime import datetime
import os

# =========================================================
# SESSION CONSISTENCY CHECK
# Run this at the start of every session to verify
# nothing has changed
# =========================================================

print("=" * 60)
print("  SESSION CONSISTENCY CHECK")
print(f"  {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)

# ---------------------------------------------------------
# 1. MODEL FILE
# ---------------------------------------------------------
file_date  = "20260320_064356"
model_path = os.path.join(main_directory, f"results/ensemble_mfcc_mslfb_resumed_{file_date}.keras")
model_size = os.path.getsize(model_path) / 1e6
model_mtime = datetime.fromtimestamp(os.path.getmtime(model_path))
model_md5 = hashlib.md5(open(model_path, 'rb').read()).hexdigest()
print(f"\n[1] Model file")
print(f"    path:     {model_path}")
print(f"    size:     {model_size:.2f} MB")
print(f"    modified: {model_mtime}")
print(f"    md5:      {model_md5}")

# ---------------------------------------------------------
# 2. SILENCE FILES ON DRIVE
# ---------------------------------------------------------
silence_dir = '/content/drive/MyDrive/silence_precomputed'
s0 = np.load(os.path.join(silence_dir, 'silence_0_mfcc.npy'))
s1 = np.load(os.path.join(silence_dir, 'silence_1_mfcc.npy'))
s0_hash = hashlib.md5(s0.tobytes()).hexdigest()
s1_hash = hashlib.md5(s1.tobytes()).hexdigest()
s0_mtime = datetime.fromtimestamp(os.path.getmtime(os.path.join(silence_dir, 'silence_0_mfcc.npy')))
print(f"\n[2] Silence files (Drive)")
print(f"    silence_dir:          {silence_dir}")
print(f"    silence_0 md5:        {s0_hash}")
print(f"    silence_1 md5:        {s1_hash}")
print(f"    silence_0 modified:   {s0_mtime}")
print(f"    silence_0 first 5:    {s0.flatten()[:5]}")
print(f"    total files:          {len(os.listdir(silence_dir))}")

# ---------------------------------------------------------
# 3. TEST SET
# ---------------------------------------------------------
print(f"\n[3] Test set")
print(f"    total samples:    {len(x_test_files)}")
print(f"    silence in test:  {np.sum(['silence_precomputed' in str(f) for f in x_test_files])}")
print(f"    first file:       {x_test_files[0]}")
print(f"    first 5 labels:   {y_test_labels[:5]}")
test_hash = hashlib.md5(np.array([str(f) for f in x_test_files]).tobytes()).hexdigest()
print(f"    test files md5:   {test_hash}")

# ---------------------------------------------------------
# 4. NOISE SAMPLES
# ---------------------------------------------------------
print(f"\n[4] Noise samples")
print(f"    count: {len(noise_samples)}")
for i, ns in enumerate(noise_samples):
    ns_hash = hashlib.md5(ns.tobytes()).hexdigest()
    print(f"    [{i}] length={len(ns)}  md5={ns_hash}")

# ---------------------------------------------------------
# 5. SPOT CHECK — one wav feature file
# ---------------------------------------------------------
sample_wav = [f for f in x_test_files if '.wav' in str(f)][0]
npy_path   = sample_wav.replace('.wav', '_mfcc.npy')
feat       = np.load(npy_path)
feat_hash  = hashlib.md5(feat.tobytes()).hexdigest()
print(f"\n[5] Feature spot check")
print(f"    file:     {sample_wav}")
print(f"    md5:      {feat_hash}")
print(f"    first 5:  {feat.flatten()[:5]}")

# ---------------------------------------------------------
# 6. FP32 BASELINE EVAL
# ---------------------------------------------------------
print(f"\n[6] FP32 Baseline eval")
model   = load_model(model_path)
results = compile_and_evaluate(model, test_ds, label="FP32 Baseline")
print(f"    Accuracy:  {results[1]:.6f}")
print(f"    Precision: {results[2]:.6f}")
print(f"    Recall:    {results[3]:.6f}")

# ---------------------------------------------------------
# REFERENCE VALUES — update these after first clean run
# ---------------------------------------------------------
EXPECTED_MODEL_MD5    = "9783da6825f856dd45a9da8385550169"
EXPECTED_SILENCE_MD5  = "b1869deb444d3e08071711555276d880"
EXPECTED_TEST_MD5     = "0c9313e6904fa9cb0bb264a7f26d3a06"
EXPECTED_ACCURACY     = 0.8723

print(f"\n{'='*60}")
print("  CONSISTENCY VALIDATION")
print(f"{'='*60}")
if EXPECTED_MODEL_MD5 is None:
    print("  ⚠️  No reference values set yet.")
    print("  → Copy the md5 and accuracy values above into")
    print("    the EXPECTED_* variables and re-run next session.")
else:
    checks = {
        "Model md5":    (model_md5,   EXPECTED_MODEL_MD5),
        "Silence md5":  (s0_hash,     EXPECTED_SILENCE_MD5),
        "Test set md5": (test_hash,   EXPECTED_TEST_MD5),
        "Accuracy":     (round(results[1], 4), round(EXPECTED_ACCURACY, 4)),
    }
    all_ok = True
    for name, (actual, expected) in checks.items():
        ok = actual == expected
        if not ok:
            all_ok = False
        print(f"  {'✓' if ok else '✗'} {name}: {actual} {'==' if ok else '!='} {expected}")
    print(f"\n  {'✅ All checks passed' if all_ok else '❌ Something changed — investigate above'}")

# ─────────────────────────────────────────────────────
# 📊 7. WEIGHT DISTRIBUTION ANALYSIS
# ─────────────────────────────────────────────────────

## ─────────────────────────────────────────────────────
## 📊 7.1 WEIGHT DISTRIBUTION FUNCTIONS
## ─────────────────────────────────────────────────────

In [ ]:
from scipy.stats import kurtosis, skew

def plot_sensitivity_by_section_layerwise(model_path, calib_ds, num_bits=4):
    """
    Measures sensitivity by comparing quantized vs FP32 layer OUTPUT
    on calibration data — much faster than full model evaluation.
    Uses MSE between FP32 and quantized layer outputs as sensitivity metric.
    """
    from tensorflow.keras.models import load_model

    base_model = load_model(model_path)
    layers     = collect_all_weight_layers(base_model)

    print(f"Layer output sensitivity analysis — {num_bits}-bit")
    print(f"{'Layer':<20} {'Type':<8} {'Kurt':>7} {'MSE':>14} {'Rel MSE':>10}  Sensitivity")
    print("-" * 75)

    records = []

    for target_layer in layers:

        # Capture FP32 layer inputs and outputs
        fp_inputs, fp_outputs = capture_layer_activations(
            base_model, target_layer, calib_ds
        )

        if not fp_inputs:
            continue

        # Quantize weights
        weights = target_layer.get_weights()
        W  = weights[0]
        b  = weights[1] if len(weights) > 1 else None
        Wq = uniform_quantize_layer(W, num_bits=num_bits)

        # Compute quantized layer output using same FP32 inputs
        mse_list = []
        for x_fp, y_fp in zip(fp_inputs, fp_outputs):
            y_q   = manual_forward(target_layer, x_fp,
                                   tf.constant(Wq),
                                   tf.constant(b) if b is not None else None)
            mse   = tf.reduce_mean((y_fp - y_q) ** 2).numpy()
            mse_list.append(mse)

        mean_mse = np.mean(mse_list)

        # Relative MSE — normalized by FP32 output variance
        fp_var   = np.mean([tf.reduce_mean(y ** 2).numpy() for y in fp_outputs])
        rel_mse  = mean_mse / (fp_var + 1e-10)

        # Sensitivity label based on relative MSE
        if rel_mse > 0.01:
            sensitivity = 'HIGH'
        elif rel_mse > 0.001:
            sensitivity = 'MEDIUM'
        else:
            sensitivity = 'LOW'

        s       = compute_layer_stats(target_layer)
        kurt    = s['kurtosis']
        is_conv = isinstance(target_layer, tf.keras.layers.Conv2D)

        print(f"{target_layer.name:<20} {'Conv2D' if is_conv else 'Dense':<8} "
              f"{kurt:>7.2f} {mean_mse:>14.6e} {rel_mse:>10.6f}  {sensitivity}")

        records.append({
            'name':        target_layer.name,
            'section':     get_section(target_layer, base_model),
            'type':        'Conv2D' if is_conv else 'Dense',
            'kurtosis':    kurt,
            'mse':         mean_mse,
            'rel_mse':     rel_mse,
            'sensitivity': sensitivity,
        })

    df = pd.DataFrame(records)

    # -------------------------------------------------------
    # Plot — same style as Gaussianity plot
    # -------------------------------------------------------
    section_colors = {
        'branch_mfcc':  '#d0e8ff',
        'branch_mslfb': '#d0ffd0',
        'classifier':   '#ffd0d0',
    }
    section_line_colors = {
        'branch_mfcc':  'steelblue',
        'branch_mslfb': 'green',
        'classifier':   'red',
    }
    sensitivity_colors = {
        'HIGH':   'red',
        'MEDIUM': 'orange',
        'LOW':    'green',
    }
    type_markers = {'Conv2D': 'o', 'Dense': 's'}

    thresholds       = [0.01,    0.001]
    threshold_labels = ['High (rel MSE > 1%)', 'Medium (rel MSE > 0.1%)']
    threshold_colors = ['red',   'orange']

    fig, axes = plt.subplots(2, 1, figsize=(22, 10), sharex=True)

    for ax, threshold, tlabel, tcolor in zip(
            axes, thresholds, threshold_labels, threshold_colors):

        # Section backgrounds
        start    = 0
        prev_sec = df['section'].iloc[0]
        for i in range(len(df)):
            sec = df['section'].iloc[i]
            if sec != prev_sec or i == len(df) - 1:
                end = i if sec != prev_sec else i + 1
                ax.axvspan(start, end, alpha=0.15,
                           color=section_colors[prev_sec], zorder=0)
                ax.text((start + end) / 2, 0.95,
                        prev_sec.upper(), fontsize=9, fontweight='bold',
                        ha='center', color=section_line_colors[prev_sec],
                        transform=ax.get_xaxis_transform())
                if sec != prev_sec:
                    ax.axvline(x=i, color='black', linestyle='--',
                               linewidth=1.5, zorder=3)
                start    = i
                prev_sec = sec

        ax.plot(df['rel_mse'].values, color='gray', linewidth=0.8, zorder=2)

        for i, row in df.iterrows():
            color  = sensitivity_colors[row['sensitivity']]
            marker = type_markers[row['type']]
            ax.scatter(i, row['rel_mse'], color=color, s=60,
                       marker=marker, zorder=4)

        ax.axhline(threshold, color=tcolor, linestyle='--',
                   linewidth=1.2, label=tlabel)
        ax.set_ylabel(f'Relative MSE ({num_bits}-bit)', fontsize=9)
        ax.set_ylim(-0.0005, max(df['rel_mse'].max() * 1.2, threshold * 2))

        section_handles = [
            mpatches.Patch(color=section_colors[s], label=s)
            for s in ['branch_mfcc', 'branch_mslfb', 'classifier']
        ]
        point_handles = [
            plt.Line2D([0], [0], marker='o', color='w',
                       markerfacecolor='red',    markersize=8,
                       label=f'rel MSE > 1%  → HIGH'),
            plt.Line2D([0], [0], marker='o', color='w',
                       markerfacecolor='orange', markersize=8,
                       label=f'rel MSE 0.1–1% → MEDIUM'),
            plt.Line2D([0], [0], marker='o', color='w',
                       markerfacecolor='green',  markersize=8,
                       label=f'rel MSE < 0.1% → LOW'),
            plt.Line2D([0], [0], marker='o', color='gray',
                       markersize=8, label='● = Conv2D   ■ = Dense'),
        ]
        ax.legend(handles=section_handles + point_handles,
                  loc='upper right', fontsize=8)

    axes[-1].set_xticks(range(len(df)))
    axes[-1].set_xticklabels(
        [f"{r['name']}\n({r['type']})\nkurt={r['kurtosis']:.1f}"
         for _, r in df.iterrows()],
        rotation=90, fontsize=6
    )
    axes[-1].set_xlabel('Layer', fontsize=10)

    plt.suptitle(
        f'Layer Output Sensitivity — {num_bits}-bit quantization\n'
        'Relative MSE between FP32 and quantized layer output',
        fontsize=13
    )
    plt.tight_layout()
    plt.show()

    # Ranking
    df_sorted = df.sort_values('rel_mse', ascending=False)
    print(f"\n{'='*65}")
    print(f"  SENSITIVITY RANKING ({num_bits}-bit) — by relative MSE")
    print(f"{'='*65}")
    print(f"  {'Layer':<20} {'Type':<8} {'Kurt':>7} {'Rel MSE':>10}  Sensitivity")
    print(f"  {'-'*60}")
    for _, row in df_sorted.iterrows():
        bar = '█' * int(row['rel_mse'] * 2000)
        print(f"  {row['name']:<20} {row['type']:<8} "
              f"{row['kurtosis']:>7.2f} {row['rel_mse']:>10.6f}  "
              f"{row['sensitivity']}  {bar}")

    return df

In [ ]:
def plot_weight_distribution_comparison(fp32_model, quantized_model, label="", bins=60,
                                         skew_threshold=0.5, kurt_threshold=1.0):
    """
    Plots the weight distribution of the FP32 model overlapped with the
    quantized model for each layer side by side.
    """
    def collect_layers(model, prefix=""):
        collected = []
        for layer in model.layers:
            layer_name = f"{prefix}/{layer.name}" if prefix else layer.name
            if isinstance(layer, (tf.keras.Model, tf.keras.Sequential)):
                collected.extend(collect_layers(layer, prefix=layer_name))
            else:
                weights = layer.get_weights()
                if weights:
                    collected.append((layer_name, layer, weights))
        return collected

    fp32_layers  = collect_layers(fp32_model)
    quant_layers = collect_layers(quantized_model)

    # Only plot Conv2D and Dense kernels (skip bias and BatchNorm)
    def is_quantized_layer(layer):
        return isinstance(layer, (tf.keras.layers.Conv2D, tf.keras.layers.Dense))

    fp32_data  = [(name, layer, w[0]) for name, layer, w in fp32_layers  if is_quantized_layer(layer)]
    quant_data = [(name, layer, w[0]) for name, layer, w in quant_layers if is_quantized_layer(layer)]

    n_plots = len(fp32_data)
    n_cols  = 3
    n_rows  = math.ceil(n_plots / n_cols)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 3.5))
    axes      = axes.flatten()

    for i, ((fp_name, fp_layer, fp_w), (_, _, q_w)) in enumerate(zip(fp32_data, quant_data)):
        ax = axes[i]

        fp_flat = fp_w.flatten()
        q_flat  = q_w.flatten()

        # Shared bin range
        w_min = min(fp_flat.min(), q_flat.min())
        w_max = max(fp_flat.max(), q_flat.max())
        bin_edges = np.linspace(w_min, w_max, bins + 1)

        ax.hist(fp_flat, bins=bin_edges, color='steelblue', alpha=0.6,
                edgecolor='none', label='FP32')
        ax.hist(q_flat,  bins=bin_edges, color='tomato',    alpha=0.6,
                edgecolor='none', label='Quantized')

        # Stats
        _, s_fp, k_fp, _ = is_normal_skew_kurt(fp_flat, skew_threshold, kurt_threshold)
        _, s_q,  k_q,  _ = is_normal_skew_kurt(q_flat,  skew_threshold, kurt_threshold)

        # Unique quantization levels
        n_levels = len(np.unique(q_flat))

        annotation = (
            f"FP32:  skew={s_fp:.2f} kurt={k_fp:.2f}\n"
            f"Quant: skew={s_q:.2f}  kurt={k_q:.2f}\n"
            f"Quant levels: {n_levels}"
        )
        ax.annotate(
            annotation,
            xy=(0.97, 0.95), xycoords='axes fraction',
            fontsize=6.5, ha='right', va='top',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='gray', alpha=0.85)
        )

        is_conv = isinstance(fp_layer, tf.keras.layers.Conv2D)
        ax.set_title(f"{fp_name}\n[{'Conv2D' if is_conv else 'Dense'}  n={len(fp_flat)}]",
                     fontsize=7.5, pad=4)
        ax.set_xlabel("Weight value", fontsize=7)
        ax.set_ylabel("Count",        fontsize=7)
        ax.tick_params(labelsize=6)
        ax.legend(fontsize=6)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle(
        f"FP32 vs Quantized Weight Distributions — {label}",
        fontsize=13, y=1.01
    )
    plt.tight_layout()
    plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
import math
import tensorflow as tf


def is_normal_skew_kurt(w_flat, skew_threshold=0.5, kurt_threshold=1.0):
    """
    Assess normality using skewness and excess kurtosis.
    These are scale-independent and not affected by sample size.

    Skewness:        0 = perfectly symmetric
    Excess Kurtosis: 0 = normal tails (positive = heavy, negative = light)

    Returns: (is_normal, skewness, kurtosis, score_str)
    """
    s = stats.skew(w_flat)
    k = stats.kurtosis(w_flat)  # excess kurtosis (normal = 0)

    normal = abs(s) < skew_threshold and abs(k) < kurt_threshold
    score_str = f"skew={s:.2f}  kurt={k:.2f}"
    return normal, s, k, score_str


def plot_weight_distributions(model, bins=60, prefix="", save_path=None,
                              skew_threshold=0.5, kurt_threshold=1.0):
    """
    Traverses all layers of a TF/Keras model recursively,
    plots the weight distribution for Conv2D and Dense layers only.
    """

    def collect_layers(model, prefix=""):
        collected = []
        for layer in model.layers:
            layer_name = f"{prefix}/{layer.name}" if prefix else layer.name

            if isinstance(layer, (tf.keras.Model, tf.keras.Sequential)):
                collected.extend(collect_layers(layer, prefix=layer_name))
            else:
                # Only keep Conv2D and Dense
                if not isinstance(layer, (tf.keras.layers.Conv2D, tf.keras.layers.Dense)):
                    continue

                weights = layer.get_weights()
                if weights:
                    collected.append((layer_name, layer, weights))

        return collected

    # ---- Collect layers ----
    all_layers = collect_layers(model, prefix)

    if not all_layers:
        print("No layers with weights found.")
        return

    plot_data = []

    # ---- Process weights ----
    for layer_name, layer, weights in all_layers:
        has_bias = hasattr(layer, 'use_bias') and layer.use_bias

        # Only take kernel (skip bias)
        w = weights[0]
        tensor_label = "kernel"

        w_flat = w.flatten()

        normal, skewness, kurtosis, score_str = is_normal_skew_kurt(
            w_flat, skew_threshold, kurt_threshold
        )

        title = f"{layer_name}\n[{tensor_label}]  n={len(w_flat)}"
        plot_data.append((title, w_flat, normal, skewness, kurtosis, score_str))

    # ---- Plot setup ----
    n_plots = len(plot_data)
    n_cols  = 3
    n_rows  = math.ceil(n_plots / n_cols)
    fig_h   = n_rows * 3.5

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, fig_h))
    axes = axes.flatten()

    # ---- Plot each distribution ----
    for i, (title, w_flat, normal, skewness, kurtosis, score_str) in enumerate(plot_data):
        ax = axes[i]

        ax.hist(w_flat, bins=bins, color='steelblue', edgecolor='none', alpha=0.85)

        mu, sigma = w_flat.mean(), w_flat.std()
        if sigma > 0:
            x = np.linspace(w_flat.min(), w_flat.max(), 300)
            pdf = stats.norm.pdf(x, mu, sigma)
            scale = len(w_flat) * (w_flat.max() - w_flat.min()) / bins
            ax.plot(x, pdf * scale, color='tomato', linewidth=1.5, label='Normal fit')

        color      = 'green' if normal else 'red'
        label_text = "normal ✓" if normal else "non-normal ✗"
        annotation = f"{score_str}\n{label_text}"

        ax.set_title(title, fontsize=7.5, pad=4)
        ax.annotate(
            annotation,
            xy=(0.97, 0.95), xycoords='axes fraction',
            fontsize=7, ha='right', va='top',
            color=color,
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec=color, alpha=0.8)
        )

        ax.set_xlabel("Weight value", fontsize=7)
        ax.set_ylabel("Count", fontsize=7)
        ax.tick_params(labelsize=6)
        ax.legend(fontsize=6)

    # Hide unused subplots
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle(
        f"Weight Distributions per Layer "
        f"[skew_thresh={skew_threshold}, kurt_thresh={kurt_threshold}]",
        fontsize=14, y=1.01
    )
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Plot saved to: {save_path}")

    plt.show()

    # ---- Print summary ----
    print(f"\n{'Layer':<55} {'Tensor':<10} {'n weights':>10} {'Skew':>8} {'Kurt':>8} {'Normal?':>10}")
    print("-" * 110)

    for title, w_flat, normal, skewness, kurtosis, score_str in plot_data:
        parts = title.split('\n')
        layer_part = parts[0]
        tensor_part = parts[1].split(']')[0].replace('[', '').strip() if len(parts) > 1 else ''
        flag = "✓" if normal else "✗"

        print(f"{layer_part:<55} {tensor_part:<10} {len(w_flat):>10} {skewness:>8.3f} {kurtosis:>8.3f} {flag:>10}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches  # ← add this
import numpy as np
import pandas as pd
from scipy import stats
from tensorflow.keras.models import load_model

# =========================================================
# PLOT ALL LAYER WEIGHTS (Conv2D + Dense)
# =========================================================
def plot_all_weights(model, n=None):
    all_weights      = []
    layer_boundaries = []
    current_idx      = 0
    count            = 0

    for layer in collect_all_weight_layers(model):
        if not isinstance(layer, tf.keras.layers.Conv2D):  # ← skip Dense
            continue
        if n is not None and count >= n:
            break

        w = layer.get_weights()[0].flatten()
        layer_boundaries.append((current_idx, layer.name, True))
        all_weights.append(w)
        current_idx += len(w)
        count += 1

    all_weights = np.concatenate(all_weights)
    layer_ends  = [b[0] for b in layer_boundaries][1:] + [len(all_weights)]

    # --- Plot 1: full waveform
    fig, ax = plt.subplots(figsize=(18, 5))
    ax.plot(all_weights, linewidth=0.3, color='steelblue')
    ax.set_title(f"Conv2D Weights — {count} layers")
    ax.set_xlim(0, len(all_weights))
    ax.set_ylabel("Weight value")
    ax.set_xlabel("Weight index")
    for idx, name, _ in layer_boundaries:
        ax.axvline(x=idx, color='darkred', linestyle='--', linewidth=1.0, alpha=0.7)
        ax.text(idx + 5, 0.9, name, rotation=90, fontsize=7,
                color='darkred', transform=ax.get_xaxis_transform())
    plt.tight_layout()
    plt.show()

    # --- Plot 2: per-layer histograms
    cols = 4
    rows = (count + cols - 1) // cols
    fig2, axes2 = plt.subplots(rows, cols, figsize=(cols * 4, rows * 3))
    axes2 = axes2.flatten()

    for i, (idx_start, name, _) in enumerate(layer_boundaries):
        w_layer = all_weights[idx_start:layer_ends[i]]
        axes2[i].hist(w_layer, bins=60, color='steelblue', alpha=0.8)
        axes2[i].set_title(f"{name}\n(n={len(w_layer):,})", fontsize=7)
        axes2[i].set_xlabel("Weight value", fontsize=6)
        axes2[i].set_ylabel("Count",        fontsize=6)
        axes2[i].tick_params(labelsize=6)

    for j in range(i + 1, len(axes2)):
        axes2[j].set_visible(False)

    fig2.suptitle("Per-layer Conv2D weight histograms", fontsize=13)
    plt.tight_layout()
    plt.show()

    # --- Plot 3: boxplot
    fig3, ax3 = plt.subplots(figsize=(14, 5))
    data_per_layer = [all_weights[b[0]:layer_ends[i]]
                      for i, b in enumerate(layer_boundaries)]
    bp = ax3.boxplot(data_per_layer, patch_artist=True, showfliers=False)
    for patch in bp['boxes']:
        patch.set_facecolor('steelblue')
        patch.set_alpha(0.7)

    ax3.set_xticks(range(1, count + 1))
    ax3.set_xticklabels([b[1] for b in layer_boundaries], rotation=90, fontsize=7)
    ax3.set_ylabel("Weight value")
    ax3.set_title("Conv2D per-layer weight spread (boxplot, outliers hidden)")
    plt.tight_layout()
    plt.show()


# =========================================================
# PLOT P-VALUE BY SECTION (Conv2D + Dense)
# =========================================================
def get_section(layer, model):
    for parent in model.layers:
        if hasattr(parent, "layers") and layer in parent.layers:
            if parent.name == "sequential_2":    # ← was "sequential"
                return "branch_mfcc"
            elif parent.name == "sequential_3":  # ← was "sequential_1"
                return "branch_mslfb"
    return "classifier"


def plot_pvalue_by_section(model):
    # DEBUG — print sections to verify before plotting
    print("Layer → Section mapping:")
    for layer in collect_all_weight_layers(model):
        print(f"  {layer.name:20s} → {get_section(layer, model)}")
    print()

    records = []
    for layer in collect_all_weight_layers(model):
        weights = layer.get_weights()
        if not weights:
            continue

        w = weights[0].flatten()
        if w.std() == 0:
            continue

        w_std    = (w - w.mean()) / w.std()
        _, p_val = stats.kstest(w_std, 'norm')

        records.append({
            'name':    layer.name,
            'section': get_section(layer, model),
            'p_value': p_val,
            'type':    'Conv2D' if isinstance(layer, tf.keras.layers.Conv2D) else 'Dense',
        })

    df = pd.DataFrame(records)

    section_colors = {
        'branch_mfcc':  '#d0e8ff',
        'branch_mslfb': '#d0ffd0',
        'classifier':   '#ffd0d0',
    }
    section_line_colors = {
        'branch_mfcc':  'steelblue',
        'branch_mslfb': 'green',
        'classifier':   'red',
    }
    type_markers = {
        'Conv2D': 'o',
        'Dense':  's',
    }

    thresholds = [0.05, 0.005]
    fig, axes  = plt.subplots(2, 1, figsize=(22, 10), sharex=True)

    for ax, threshold in zip(axes, thresholds):
        start    = 0
        prev_sec = df['section'].iloc[0]

        for i in range(len(df)):
            sec = df['section'].iloc[i]
            if sec != prev_sec or i == len(df) - 1:
                end = i if sec != prev_sec else i + 1
                ax.axvspan(start, end, alpha=0.15,
                           color=section_colors[prev_sec], zorder=0)
                ax.text((start + end) / 2, 0.95,
                        prev_sec.upper(), fontsize=9, fontweight='bold',
                        ha='center', color=section_line_colors[prev_sec],
                        transform=ax.get_xaxis_transform())
                if sec != prev_sec:
                    ax.axvline(x=i, color='black', linestyle='--',
                               linewidth=1.5, zorder=3)
                start    = i
                prev_sec = sec

        ax.plot(df['p_value'].values, color='gray', linewidth=0.8, zorder=2)

        for i, row in df.iterrows():
            color  = 'red' if row['p_value'] < threshold else 'green'
            marker = type_markers[row['type']]
            ax.scatter(i, row['p_value'], color=color, s=60,
                       marker=marker, zorder=4)

        ax.axhline(threshold, color='red', linestyle='--', linewidth=1.2)
        ax.set_ylabel('p-value (KS test)')
        ax.set_ylim(0, 1)
        ax.set_title(f'Gaussianity test — threshold = {threshold} '
                     f'({"5%" if threshold == 0.05 else "0.05%"} risk)')

        section_handles = [
            mpatches.Patch(color=section_colors[s], label=s)
            for s in ['branch_mfcc', 'branch_mslfb', 'classifier']
        ]

        point_handles = [
            plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='red',
                       markersize=8, label=f'p < {threshold} → not Gaussian'),
            plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='green',
                       markersize=8, label=f'p ≥ {threshold} → cannot reject Gaussian'),
            plt.Line2D([0], [0], marker='o', color='gray',
                       markersize=8, label='● = Conv2D   ■ = Dense'),
        ]
        ax.legend(handles=section_handles + point_handles,
                  loc='upper right', fontsize=8)

        n_not = (df['p_value'] < threshold).sum()
        n_is  = (df['p_value'] >= threshold).sum()
        print(f"Threshold {threshold}:")
        print(f"  Not Gaussian: {n_not} layers ({n_not/len(df)*100:.1f}%)")
        print(f"  Gaussian:     {n_is}  layers ({n_is /len(df)*100:.1f}%)")
        print()

    axes[-1].set_xticks(range(len(df)))
    axes[-1].set_xticklabels(
        [f"{r['name']}\n({r['type']})" for _, r in df.iterrows()],
        rotation=90, fontsize=6
    )
    axes[-1].set_xlabel('Layer')
    plt.suptitle('Gaussianity test — MFCC Branch / MSLFB Branch / Classifier',
                 fontsize=13)
    plt.tight_layout()
    plt.show()
    return df

In [ ]:
def plot_weight_statistics(model):
    from scipy.stats import kurtosis, skew

    records = []
    for layer in collect_all_weight_layers(model):
        weights = layer.get_weights()
        if not weights:
            continue
        w = weights[0].flatten()
        if w.std() == 0:
            continue

        records.append({
            'name':     layer.name,
            'section':  get_section(layer, model),
            'type':     'Conv2D' if isinstance(layer, tf.keras.layers.Conv2D) else 'Dense',
            'std':      np.std(w),
            'range':    np.max(w) - np.min(w),
            'kurtosis': kurtosis(w),   # excess kurtosis (0 = Gaussian)
            'skew':     skew(w),
        })

    df = pd.DataFrame(records)

    section_colors = {
        'branch_mfcc':  '#d0e8ff',
        'branch_mslfb': '#d0ffd0',
        'classifier':   '#ffd0d0',
    }
    section_line_colors = {
        'branch_mfcc':  'steelblue',
        'branch_mslfb': 'green',
        'classifier':   'red',
    }
    stat_colors = {
        'std':      'steelblue',
        'range':    'darkorange',
        'kurtosis': 'red',
        'skew':     'purple',
    }

    stats_to_plot = ['std', 'range', 'kurtosis', 'skew']
    fig, axes = plt.subplots(4, 1, figsize=(22, 16), sharex=True)

    for ax, stat in zip(axes, stats_to_plot):
        # Section backgrounds
        start    = 0
        prev_sec = df['section'].iloc[0]

        for i in range(len(df)):
            sec = df['section'].iloc[i]
            if sec != prev_sec or i == len(df) - 1:
                end = i if sec != prev_sec else i + 1
                ax.axvspan(start, end, alpha=0.15,
                           color=section_colors[prev_sec], zorder=0)
                if stat == stats_to_plot[0]:  # only label on top subplot
                    ax.text((start + end) / 2, 0.95,
                            prev_sec.upper(), fontsize=9, fontweight='bold',
                            ha='center', color=section_line_colors[prev_sec],
                            transform=ax.get_xaxis_transform())
                if sec != prev_sec:
                    ax.axvline(x=i, color='black', linestyle='--',
                               linewidth=1.5, zorder=3)
                start    = i
                prev_sec = sec

        # Section label on all subplots
        start    = 0
        prev_sec = df['section'].iloc[0]
        for i in range(len(df)):
            sec = df['section'].iloc[i]
            if sec != prev_sec or i == len(df) - 1:
                end = i if sec != prev_sec else i + 1
                ax.text((start + end) / 2, 0.95,
                        prev_sec.upper(), fontsize=8, fontweight='bold',
                        ha='center', color=section_line_colors[prev_sec],
                        transform=ax.get_xaxis_transform())
                start    = i
                prev_sec = sec

        # Plot stat line + markers, different shape for Conv2D vs Dense
        for i, row in df.iterrows():
            marker = 'o' if row['type'] == 'Conv2D' else 's'
            ax.scatter(i, row[stat], color=stat_colors[stat],
                       s=50, marker=marker, zorder=4)

        ax.plot(df[stat].values, color=stat_colors[stat],
                linewidth=0.8, zorder=2)

        # Zero line for kurtosis and skew (reference = Gaussian)
        if stat in ('kurtosis', 'skew'):
            ax.axhline(0, color='gray', linestyle='--',
                       linewidth=1.0, alpha=0.7)

        ax.set_ylabel(stat, fontsize=10)
        ax.tick_params(labelsize=7)

        # Legend
        type_handles = [
            plt.Line2D([0], [0], marker='o', color='w',
                       markerfacecolor=stat_colors[stat],
                       markersize=8, label='Conv2D'),
            plt.Line2D([0], [0], marker='s', color='w',
                       markerfacecolor=stat_colors[stat],
                       markersize=8, label='Dense'),
        ]
        ax.legend(handles=type_handles, fontsize=7, loc='upper right')

    axes[-1].set_xticks(range(len(df)))
    axes[-1].set_xticklabels(
        [f"{r['name']}\n({r['type']})" for _, r in df.iterrows()],
        rotation=90, fontsize=6
    )
    axes[-1].set_xlabel('Layer', fontsize=10)

    # Print summary
    print("Weight statistics summary:")
    print(df[['name', 'section', 'type', 'std', 'range', 'kurtosis', 'skew']].to_string(index=False))

    plt.suptitle('Weight statistics per layer — std / range / kurtosis / skew\n'
                 'MFCC Branch / MSLFB Branch / Classifier',
                 fontsize=13)
    plt.tight_layout()
    plt.show()
    return df

In [ ]:
from scipy import stats
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scikit_posthocs as sp

def compare_layer_distributions(model):
    layers  = collect_all_weight_layers(model)
    weights = {}

    for layer in layers:
        w = layer.get_weights()[0].flatten()
        if len(w) > 0:
            weights[layer.name] = w

    names  = list(weights.keys())
    groups = list(weights.values())

    # -------------------------------------------------------
    # 1. Kruskal-Wallis
    # -------------------------------------------------------
    stat, p = stats.kruskal(*groups)
    print("=" * 60)
    print("  Kruskal-Wallis Test")
    print("=" * 60)
    print(f"  H statistic: {stat:.4f}")
    print(f"  p-value:     {p:.2e}")
    if p <= 0.001:
        print("  → p ≤ 0.001: Layers are SIGNIFICANTLY different")
        print("    → Individual layer-wise quantization recommended")
    else:
        print("  → p > 0.001: No significant difference")
        print("    → Collective quantization per layer type is acceptable")

    # -------------------------------------------------------
    # 2. Compute mean ranks and confidence intervals
    #    (Multiple comparison style — like Fig. 5 in paper)
    # -------------------------------------------------------
    all_values = np.concatenate(groups)
    all_ranks  = stats.rankdata(all_values)
    N          = len(all_values)

    idx = 0
    mean_ranks = {}
    ci_half    = {}

    for name, w in weights.items():
        n = len(w)
        r = all_ranks[idx:idx + n]

        # Mean rank
        mean_ranks[name] = r.mean()

        # 95% CI half-width using rank-based SE
        # SE = sqrt(N*(N+1)/12 * (1/n))  — Kruskal-Wallis rank variance
        se = np.sqrt(N * (N + 1) / 12.0 * (1.0 / n))
        ci_half[name] = 1.96 * se  # 95% CI

        idx += n

    # -------------------------------------------------------
    # 3. Sort by mean rank (ascending)
    # -------------------------------------------------------
    sorted_names = sorted(mean_ranks.keys(), key=lambda x: mean_ranks[x])
    n_layers     = len(sorted_names)
    y_pos        = {name: i + 1 for i, name in enumerate(sorted_names)}

    # -------------------------------------------------------
    # 4. Plot — Multiple comparison style (Fig. 5)
    # -------------------------------------------------------
    fig, ax = plt.subplots(figsize=(12, max(5, n_layers * 0.55)))

    for name in sorted_names:
        y    = y_pos[name]
        x    = mean_ranks[name]
        half = ci_half[name]

        # Horizontal CI bar
        ax.plot([x - half, x + half], [y, y], '-',
                color='darkgreen', linewidth=2.0, zorder=3)
        # End ticks
        ax.plot([x - half, x - half], [y - 0.2, y + 0.2], '-',
                color='darkgreen', linewidth=1.5, zorder=3)
        ax.plot([x + half, x + half], [y - 0.2, y + 0.2], '-',
                color='darkgreen', linewidth=1.5, zorder=3)
        # Center dot
        ax.plot(x, y, 'o', color='darkgreen', markersize=7,
                zorder=5, markerfacecolor='darkgreen')
        # Label
        ax.text(x + (max(ci_half.values()) * 1.1),
                y, name, va='center', ha='left', fontsize=8)

    # Reference line at grand mean rank
    grand_mean = np.mean(list(mean_ranks.values()))
    ax.axvline(grand_mean, color='gray', linestyle='--',
               linewidth=0.8, alpha=0.5, label='Grand mean')

    ax.set_xlabel('Mean Rank difference', fontsize=10)
    ax.set_ylabel('Weight group of each layer CNN', fontsize=10)
    ax.set_title(
        'Multiple Comparison Test — Layer Weight Distributions\n'
        'Non-overlapping intervals → statistically different distributions',
        fontsize=11
    )
    ax.set_yticks(list(y_pos.values()))
    ax.set_yticklabels(list(y_pos.keys()), fontsize=8)
    ax.invert_yaxis()
    ax.grid(axis='x', linestyle='--', alpha=0.3)
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    # -------------------------------------------------------
    # 5. Print which pairs overlap (not significantly different)
    # -------------------------------------------------------
    print("\n  Pairs with OVERLAPPING intervals (not significantly different):")
    print(f"  {'Layer A':<20} {'Layer B':<20}")
    print(f"  {'-'*42}")
    overlap_count = 0
    no_overlap    = 0
    for i in range(len(sorted_names)):
        for j in range(i + 1, len(sorted_names)):
            a, b   = sorted_names[i], sorted_names[j]
            a_low  = mean_ranks[a] - ci_half[a]
            a_high = mean_ranks[a] + ci_half[a]
            b_low  = mean_ranks[b] - ci_half[b]
            b_high = mean_ranks[b] + ci_half[b]
            # Overlap if intervals intersect
            if a_low <= b_high and b_low <= a_high:
                overlap_count += 1
            else:
                print(f"  {a:<20} {b:<20}  ✗ different")
                no_overlap += 1

    total = overlap_count + no_overlap
    print(f"\n  Overlapping (similar):       {overlap_count} / {total}")
    print(f"  Non-overlapping (different): {no_overlap} / {total}")

    return mean_ranks, ci_half

In [ ]:
def plot_sensitivity_rank_by_section(model_path, calib_ds, num_bits=4):
    """
    Computes layer output sensitivity (relative MSE), ranks layers,
    and plots sensitivity rank per layer grouped by branch section.
    Includes Kruskal-Wallis test between sections.
    """
    from tensorflow.keras.models import load_model
    from scipy import stats

    base_model = load_model(model_path)
    layers     = collect_all_weight_layers(base_model)

    print(f"Computing layer output sensitivity — {num_bits}-bit...")

    records = []
    for target_layer in layers:
        fp_inputs, fp_outputs = capture_layer_activations(
            base_model, target_layer, calib_ds
        )
        if not fp_inputs:
            continue

        weights = target_layer.get_weights()
        W  = weights[0]
        b  = weights[1] if len(weights) > 1 else None
        Wq = uniform_quantize_layer(W, num_bits=num_bits)

        mse_list = []
        for x_fp, y_fp in zip(fp_inputs, fp_outputs):
            y_q  = manual_forward(target_layer, x_fp,
                                  tf.constant(Wq),
                                  tf.constant(b) if b is not None else None)
            mse_list.append(tf.reduce_mean((y_fp - y_q) ** 2).numpy())

        mean_mse = np.mean(mse_list)
        fp_var   = np.mean([tf.reduce_mean(y ** 2).numpy() for y in fp_outputs])
        rel_mse  = mean_mse / (fp_var + 1e-10)

        section = get_section(target_layer, base_model)
        is_conv = isinstance(target_layer, tf.keras.layers.Conv2D)

        records.append({
            'name':    target_layer.name,
            'section': section,
            'type':    'Conv2D' if is_conv else 'Dense',
            'rel_mse': rel_mse,
        })

    df = pd.DataFrame(records)

    # Assign global rank (1 = least sensitive, N = most sensitive)
    df['rank'] = df['rel_mse'].rank(ascending=True).astype(int)
    df = df.reset_index(drop=True)
    df['idx'] = df.index

    # -------------------------------------------------------
    # Kruskal-Wallis between sections
    # -------------------------------------------------------
    section_groups = {s: df[df['section'] == s]['rank'].values
                      for s in df['section'].unique()}

    if len(section_groups) >= 2:
        kw_stat, kw_p = stats.kruskal(*section_groups.values())
    else:
        kw_stat, kw_p = 0, 1

    print(f"Kruskal-Wallis: stat={kw_stat:.2f}, p={kw_p:.4f}")

    # -------------------------------------------------------
    # Section styling
    # -------------------------------------------------------
    section_colors = {
        'branch_mfcc':  '#d0ffd0',
        'branch_mslfb': '#ffd0d0',
        'classifier':   '#ffffd0',
    }
    section_dot_colors = {
        'branch_mfcc':  'green',
        'branch_mslfb': 'red',
        'classifier':   'orange',
    }
    section_mean_colors = {
        'branch_mfcc':  'green',
        'branch_mslfb': 'red',
        'classifier':   'orange',
    }
    section_labels = {
        'branch_mfcc':  'MFCC',
        'branch_mslfb': 'MSLFB',
        'classifier':   'CLASSIFIER',
    }

    # -------------------------------------------------------
    # Plot
    # -------------------------------------------------------
    fig, ax = plt.subplots(figsize=(max(16, len(df) * 0.45), 7))

    # Section backgrounds + labels
    sections_seen = []
    for section in df['section'].unique():
        sec_df  = df[df['section'] == section]
        x_start = sec_df['idx'].min() - 0.5
        x_end   = sec_df['idx'].max() + 0.5
        ax.axvspan(x_start, x_end, alpha=0.15,
                   color=section_dot_colors[section], zorder=0)
        ax.text((x_start + x_end) / 2,
                df['rank'].max() * 0.97,
                section_labels[section],
                ha='center', va='top', fontsize=13,
                fontweight='bold', color=section_dot_colors[section])
        sections_seen.append(section)

    # Section dividers
    for i in range(1, len(df)):
        if df.loc[i, 'section'] != df.loc[i-1, 'section']:
            ax.axvline(x=i - 0.5, color='black',
                       linestyle='--', linewidth=1.5, zorder=3)

    # Line connecting all points
    ax.plot(df['idx'], df['rank'], '-',
            color='gray', linewidth=0.8, zorder=2)

    # Dots colored by section
    for _, row in df.iterrows():
        color  = section_dot_colors[row['section']]
        marker = 'o' if row['type'] == 'Conv2D' else 's'
        ax.scatter(row['idx'], row['rank'],
                   color=color, s=60, marker=marker,
                   zorder=4, edgecolors='black', linewidths=0.4)

    # Mean rank lines per section
    legend_handles = []
    for section in df['section'].unique():
        sec_df    = df[df['section'] == section]
        mean_rank = sec_df['rank'].mean()
        x_start   = sec_df['idx'].min() - 0.5
        x_end     = sec_df['idx'].max() + 0.5
        color     = section_mean_colors[section]
        ax.hlines(mean_rank, x_start, x_end,
                  colors=color, linestyles='--',
                  linewidth=1.5, zorder=3)
        legend_handles.append(
            plt.Line2D([0], [0], color=color, linestyle='--',
                       linewidth=1.5,
                       label=f'{section_labels[section]} mean rank = {mean_rank:.1f}')
        )

    # Marker legend
    legend_handles += [
        plt.Line2D([0], [0], marker='o', color='w',
                   markerfacecolor='gray', markersize=8,
                   markeredgecolor='black', label='● = Conv2D'),
        plt.Line2D([0], [0], marker='s', color='w',
                   markerfacecolor='gray', markersize=8,
                   markeredgecolor='black', label='■ = Dense'),
    ]
    ax.legend(handles=legend_handles, loc='upper right', fontsize=8)

    ax.set_xticks(df['idx'])
    ax.set_xticklabels(
        [f"{r['name']}\n({r['type']})" for _, r in df.iterrows()],
        rotation=90, fontsize=7
    )
    ax.set_ylabel('Sensitivity Rank\n(1 = least sensitive)', fontsize=10)
    ax.set_xlabel('Layer index', fontsize=10)
    ax.set_title(
        f'Sensitivity Rank per Layer — MFCC / MSLFB Branches\n'
        f'Kruskal-Wallis: stat={kw_stat:.2f}, p={kw_p:.4f}',
        fontsize=12
    )
    ax.set_xlim(-0.5, len(df) - 0.5)
    ax.grid(axis='y', linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Summary
    print(f"\n{'='*60}")
    print(f"  SENSITIVITY RANKING — {num_bits}-bit")
    print(f"{'='*60}")
    print(f"  {'Rank':>4} {'Layer':<20} {'Section':<15} {'Rel MSE':>12}")
    print(f"  {'-'*55}")
    for _, row in df.sort_values('rank').iterrows():
        print(f"  {row['rank']:>4} {row['name']:<20} {row['section']:<15} {row['rel_mse']:>12.6f}")

    return df

In [ ]:
def plot_sensitivity_distribution(
    model_path,
    calib_ds,
    num_bits        = 4,
    p_aggressive    = 75,   # percentile below which → aggressive quantization
    p_full_prec     = 90,   # percentile above which → full precision / skip
    label           = None,
):
    """
    Plots the distribution of layer sensitivity (MSE) to quantization,
    with data-driven percentile thresholds.

    Returns a dict with the computed thresholds and tier assignments,
    which can be fed directly into sensitivity_aware_quantize_model.

    Tiers:
        Aggressive  : rel_mse <= p_aggressive  → safe to compress hard
        Moderate    : p_aggressive < rel_mse <= p_full_prec → moderate bits
        Sensitive   : rel_mse > p_full_prec    → keep at high precision
    """
    from tensorflow.keras.models import load_model

    if label is None:
        label = f"Distribution of layer sensitivity to {num_bits}-bit quantization"

    base_model = load_model(model_path)
    layers     = collect_all_weight_layers(base_model)

    print(f"Measuring sensitivity for {len(layers)} layers at {num_bits}-bit...")

    records = []
    for target_layer in layers:
        weights = target_layer.get_weights()
        if not weights:
            continue

        W  = weights[0]
        b  = weights[1] if len(weights) > 1 else None

        fp_inputs, fp_outputs = capture_layer_activations(
            base_model, target_layer, calib_ds
        )
        if not fp_inputs:
            continue

        Wq       = uniform_quantize_layer(W, num_bits=num_bits)
        mse_list = []
        for x_fp, y_fp in zip(fp_inputs, fp_outputs):
            y_q  = manual_forward(
                target_layer, x_fp,
                tf.constant(Wq),
                tf.constant(b) if b is not None else None,
            )
            mse_list.append(tf.reduce_mean((y_fp - y_q) ** 2).numpy())

        records.append({
            'name':    target_layer.name,
            'type':    'Conv2D' if isinstance(target_layer, tf.keras.layers.Conv2D) else 'Dense',
            'mse':     float(np.mean(mse_list)),
        })

    del base_model

    df       = pd.DataFrame(records)
    mse_vals = df['mse'].values

    # ── Compute data-driven thresholds ───────────────────────────────────────
    thresh_aggressive = np.percentile(mse_vals, p_aggressive)
    thresh_full_prec  = np.percentile(mse_vals, p_full_prec)

    df['tier'] = 'Moderate'
    df.loc[df['mse'] <= thresh_aggressive, 'tier'] = 'Aggressive'
    df.loc[df['mse'] >  thresh_full_prec,  'tier'] = 'Sensitive'

    n_agg  = (df['tier'] == 'Aggressive').sum()
    n_mod  = (df['tier'] == 'Moderate').sum()
    n_sens = (df['tier'] == 'Sensitive').sum()
    total  = len(df)

    # ── Plot ─────────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(12, 5))

    ax.hist(mse_vals, bins=40, log=False, color='steelblue', edgecolor='white', linewidth=0.4)

    ax.axvline(thresh_aggressive, color='green',  linestyle='--', linewidth=1.5,
               label=f'p{p_aggressive} = {thresh_aggressive:.4f} → aggressive quantization')
    ax.axvline(thresh_full_prec,  color='orange', linestyle='--', linewidth=1.5,
               label=f'p{p_full_prec} = {thresh_full_prec:.4f} → full precision')

    ax.set_xscale('log')
    ax.set_xlabel('Sensitivity (MSE) — log scale', fontsize=11)
    ax.set_ylabel('Number of layers',              fontsize=11)
    ax.set_title(label,                            fontsize=13)
    ax.legend(fontsize=9)

    plt.tight_layout()

    caption = (
        f"Aggressive (≤ p{p_aggressive}): {n_agg:>3} layers ({100*n_agg/total:.1f}%)\n"
        f"Moderate (p{p_aggressive}–p{p_full_prec}):  {n_mod:>3} layers ({100*n_mod/total:.1f}%)\n"
        f"Sensitive (> p{p_full_prec}):   {n_sens:>3} layers ({100*n_sens/total:.1f}%)"
    )
    fig.text(0.01, -0.06, caption, fontsize=10, family='monospace', va='top')

    plt.show()

    # ── Print ranking ─────────────────────────────────────────────────────────
    print(f"\n{'='*65}")
    print(f"  SENSITIVITY DISTRIBUTION — {num_bits}-bit")
    print(f"  p{p_aggressive} threshold (aggressive) : {thresh_aggressive:.6f}")
    print(f"  p{p_full_prec} threshold (sensitive)  : {thresh_full_prec:.6f}")
    print(f"{'='*65}")
    print(f"  {'Layer':<22} {'Type':<8} {'MSE':>12}  Tier")
    print(f"  {'-'*55}")
    for _, row in df.sort_values('mse').iterrows():
        tier_icon = {'Aggressive': '🟩', 'Moderate': '🟧', 'Sensitive': '🟥'}.get(row['tier'], '  ')
        print(f"  {row['name']:<22} {row['type']:<8} {row['mse']:>12.6f}  {tier_icon} {row['tier']}")

    return {
        'df':                 df,
        'thresh_aggressive':  thresh_aggressive,
        'thresh_full_prec':   thresh_full_prec,
        'p_aggressive':       p_aggressive,
        'p_full_prec':        p_full_prec,
    }

In [ ]:
def layer_sensitivity_analysis_with_clipping(
    model_path,
    calib_ds,
    candidate_bits        = (2, 3, 4, 8),
    clip_percentiles      = (0.95, 0.99, 0.999, 1.0),
    mse_threshold_percentile = 0.3,
):
    """
    For each layer:
      1. Find best clip percentile at max bits (8) by minimizing rel MSE
      2. Measure rel MSE at each candidate bit width using best clip
      3. Assign minimum bits that keeps rel MSE below threshold
    """

    def quantize_clipped(W, num_bits, clip_percentile):
        """Uniform per-channel quantization with weight clipping."""
        clip_val = np.percentile(np.abs(W), clip_percentile * 100)
        W_clipped = np.clip(W, -clip_val, clip_val)
        out_ch    = W_clipped.shape[-1]
        W_r       = W_clipped.reshape(-1, out_ch)
        Wq        = np.zeros_like(W_r)
        qmax      = 2 ** (num_bits - 1) - 1
        qmin      = -qmax - 1
        for c in range(out_ch):
            w_ch  = W_r[:, c]
            w_max = np.max(np.abs(w_ch)) + 1e-8
            scale = w_max / qmax
            Wq[:, c] = np.clip(np.round(w_ch / scale), qmin, qmax) * scale
        return Wq.reshape(W.shape).astype(np.float32)

    base_model  = load_model(model_path)
    sens_layers = collect_all_weight_layers(base_model)
    records     = []

    print(f"{'='*75}")
    print(f"  SENSITIVITY ANALYSIS WITH CLIPPING")
    print(f"  Candidate bits: {candidate_bits} | Clip percentiles: {clip_percentiles}")
    print(f"{'='*75}")

    for target_layer in sens_layers:
        weights = target_layer.get_weights()
        if not weights:
            continue

        W  = weights[0]
        b  = weights[1] if len(weights) > 1 else None
        is_conv = isinstance(target_layer, tf.keras.layers.Conv2D)

        fp_inputs, fp_outputs = capture_layer_activations(
            base_model, target_layer, calib_ds
        )
        if not fp_inputs:
            continue

        fp_var = np.mean([tf.reduce_mean(y ** 2).numpy() for y in fp_outputs]) + 1e-10

        # ── Step 1: find best clip percentile at max bits ─────────────────
        best_clip     = 1.0
        best_clip_mse = float('inf')

        for clip_pct in clip_percentiles:
            Wq = quantize_clipped(W, num_bits=max(candidate_bits), clip_percentile=clip_pct)
            mse_list = []
            for x_fp, y_fp in zip(fp_inputs, fp_outputs):
                y_q = manual_forward(
                    target_layer, x_fp,
                    tf.constant(Wq),
                    tf.constant(b) if b is not None else None,
                )
                mse_list.append(tf.reduce_mean((y_fp - y_q) ** 2).numpy())
            rel_mse = np.mean(mse_list) / fp_var
            print(f"    {target_layer.name} clip={clip_pct:.3f} → rel_mse={rel_mse:.6f}")
            if rel_mse < best_clip_mse:
                best_clip_mse = rel_mse
                best_clip     = clip_pct

        # ── Step 2: measure rel MSE at each bit width using best clip ─────
        bit_rel_mse = {}
        for bits in candidate_bits:
            Wq = quantize_clipped(W, num_bits=bits, clip_percentile=best_clip)
            mse_list = []
            for x_fp, y_fp in zip(fp_inputs, fp_outputs):
                y_q = manual_forward(
                    target_layer, x_fp,
                    tf.constant(Wq),
                    tf.constant(b) if b is not None else None,
                )
                mse_list.append(tf.reduce_mean((y_fp - y_q) ** 2).numpy())
            bit_rel_mse[bits] = np.mean(mse_list) / fp_var

        records.append({
            'name':        target_layer.name,
            'type':        'Conv2D' if is_conv else 'Dense',
            'sensitivity': bit_rel_mse[max(candidate_bits)],
            'best_clip':   best_clip,
            'bit_rel_mse': bit_rel_mse,
        })

        print(f"  {target_layer.name:<22} {'Conv2D' if is_conv else 'Dense':<8} "
              f"clip={best_clip:.3f} | "
              + " | ".join([f"{b}b={bit_rel_mse[b]:.6f}" for b in candidate_bits]))

    del base_model

    df = pd.DataFrame(records)

    # ── Step 3: data-driven MSE threshold ────────────────────────────────────
    all_mses_at_lowest = [r['bit_rel_mse'][min(candidate_bits)] for r in records]
    mse_threshold      = np.percentile(all_mses_at_lowest, mse_threshold_percentile * 100)
    print(f"\n  MSE threshold ({mse_threshold_percentile*100:.0f}th percentile "
          f"of {min(candidate_bits)}-bit rel MSE): {mse_threshold:.6f}")

    # ── Step 4: assign minimum bits that stays below threshold ───────────────
    def assign_bits(bit_mse_dict):
        for bits in sorted(candidate_bits):      # low → high
            if bit_mse_dict[bits] <= mse_threshold:
                return bits
        return max(candidate_bits)

    df['assigned_bits'] = df['bit_rel_mse'].apply(assign_bits)

    print(f"\n  {'='*50}")
    print(f"  BIT ASSIGNMENT SUMMARY")
    print(f"  {'='*50}")
    for b in sorted(candidate_bits):
        n = (df['assigned_bits'] == b).sum()
        print(f"  {b}-bit: {n:>3} layers ({100*n/len(df):.1f}%)")
    print(f"  Avg bits: {df['assigned_bits'].mean():.2f}")
    print(f"\n  {'Layer':<22} {'Type':<8} {'Clip':>6} {'Assigned':>10}  Rel MSE per bit")
    print(f"  {'-'*75}")
    for _, row in df.iterrows():
        mse_str = " | ".join([f"{b}b={row['bit_rel_mse'][b]:.5f}" for b in candidate_bits])
        print(f"  {row['name']:<22} {row['type']:<8} {row['best_clip']:>6.3f} "
              f"{row['assigned_bits']:>8}-bit  {mse_str}")

    return df


def quantize_model_from_df(model, df):
    """Apply quantization to a Keras model using assignments from sensitivity df."""

    def quantize_clipped(W, num_bits, clip_percentile):
        clip_val = np.percentile(np.abs(W), clip_percentile * 100)
        W_clipped = np.clip(W, -clip_val, clip_val)
        out_ch    = W_clipped.shape[-1]
        W_r       = W_clipped.reshape(-1, out_ch)
        Wq        = np.zeros_like(W_r)
        qmax      = 2 ** (num_bits - 1) - 1
        qmin      = -qmax - 1
        for c in range(out_ch):
            w_ch  = W_r[:, c]
            w_max = np.max(np.abs(w_ch)) + 1e-8
            scale = w_max / qmax
            Wq[:, c] = np.clip(np.round(w_ch / scale), qmin, qmax) * scale
        return Wq.reshape(W.shape).astype(np.float32)

    quant_layers = collect_all_weight_layers(model)
    name_to_row  = {row['name']: row for _, row in df.iterrows()}

    print(f"\n{'='*60}")
    print(f"  APPLYING QUANTIZATION FROM SENSITIVITY DF")
    print(f"{'='*60}")

    for i, layer in enumerate(quant_layers):
        weights = layer.get_weights()
        if not weights:
            continue

        W  = weights[0]
        b  = weights[1] if len(weights) > 1 else None

        # positional match fallback
        row = name_to_row.get(layer.name)
        if row is None:
            df_list = list(df.iterrows())
            if i < len(df_list):
                row = df_list[i][1]

        if row is None:
            print(f"  {layer.name:<22} — no match found, skipping")
            continue

        bits = int(row['assigned_bits'])
        clip = float(row['best_clip'])
        Wq   = quantize_clipped(W, num_bits=bits, clip_percentile=clip)

        layer.set_weights([Wq, b] if b is not None else [Wq])
        is_conv = isinstance(layer, tf.keras.layers.Conv2D)
        print(f"  {layer.name:<22} {'Conv2D' if is_conv else 'Dense':<8} "
              f"→ {bits}-bit | clip={clip:.3f}")

    print("\nQuantization complete.")
    return model

In [ ]:
def plot_weight_distributions_per_channel(model, bins=60, prefix="",
                                          skew_threshold=0.5, kurt_threshold=1.0):
    """
    Like plot_weight_distributions but breaks each layer down per output channel
    (Conv2D) or per output neuron (Dense).
    Each subplot = one output channel/neuron's weight distribution with skew/kurt annotation.
    """

    def collect_layers(model, prefix=""):
        collected = []
        for layer in model.layers:
            layer_name = f"{prefix}/{layer.name}" if prefix else layer.name
            if isinstance(layer, (tf.keras.Model, tf.keras.Sequential)):
                collected.extend(collect_layers(layer, prefix=layer_name))
            else:
                if not isinstance(layer, (tf.keras.layers.Conv2D, tf.keras.layers.Dense)):
                    continue
                weights = layer.get_weights()
                if weights:
                    collected.append((layer_name, layer, weights))
        return collected

    all_layers = collect_layers(model, prefix)
    if not all_layers:
        print("No layers with weights found.")
        return

    for layer_name, layer, weights in all_layers:
        W      = weights[0]
        out_ch = W.shape[-1]
        W_r    = W.reshape(-1, out_ch)

        is_conv  = isinstance(layer, tf.keras.layers.Conv2D)
        unit_label = "ch" if is_conv else "n"   # channel vs neuron label

        n_cols = 8
        n_rows = math.ceil(out_ch / n_cols)
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2.2, n_rows * 2.2))
        axes = np.array(axes).flatten()

        for c in range(out_ch):
            ax   = axes[c]
            w_ch = W_r[:, c]

            normal, skewness, kurt, score_str = is_normal_skew_kurt(
                w_ch, skew_threshold, kurt_threshold)

            ax.hist(w_ch, bins=bins, color='steelblue', edgecolor='none', alpha=0.85)

            mu, sigma = w_ch.mean(), w_ch.std()
            if sigma > 0:
                x     = np.linspace(w_ch.min(), w_ch.max(), 200)
                pdf   = stats.norm.pdf(x, mu, sigma)
                scale = len(w_ch) * (w_ch.max() - w_ch.min()) / bins
                ax.plot(x, pdf * scale, color='tomato', linewidth=1.0)

            color = 'green' if normal else 'red'
            ax.annotate(
                f"sk={skewness:.2f}\nku={kurt:.2f}",
                xy=(0.97, 0.95), xycoords='axes fraction',
                fontsize=5.5, ha='right', va='top', color=color,
                bbox=dict(boxstyle='round,pad=0.2', fc='white', ec=color, alpha=0.75)
            )
            ax.set_title(f"{unit_label} {c}", fontsize=6, pad=2)
            ax.tick_params(labelsize=5)
            ax.set_xlabel("")
            ax.set_ylabel("")

        for j in range(out_ch, len(axes)):
            axes[j].set_visible(False)

        layer_type  = 'Conv2D' if is_conv else 'Dense'
        units_label = 'channels' if is_conv else 'neurons'
        fig.suptitle(
            f"{layer_name}  [{layer_type}  "
            f"{out_ch} {units_label}  |  skew_thresh={skew_threshold}  kurt_thresh={kurt_threshold}]",
            fontsize=9, y=1.01
        )
        plt.tight_layout()
        plt.show()

        # ── Per-layer summary table ───────────────────────────────────────────
        n_normal = sum(
            is_normal_skew_kurt(W_r[:, c], skew_threshold, kurt_threshold)[0]
            for c in range(out_ch)
        )
        skews = [stats.skew(W_r[:, c])     for c in range(out_ch)]
        kurts = [stats.kurtosis(W_r[:, c]) for c in range(out_ch)]
        print(f"\n{layer_name}  ({out_ch} {units_label})")
        print(f"  normal {units_label} : {n_normal}/{out_ch}  ({100*n_normal/out_ch:.0f}%)")
        print(f"  skew  — mean={np.mean(skews):+.3f}  std={np.std(skews):.3f}  "
              f"min={np.min(skews):+.3f}  max={np.max(skews):+.3f}")
        print(f"  kurt  — mean={np.mean(kurts):+.3f}  std={np.std(kurts):.3f}  "
              f"min={np.min(kurts):+.3f}  max={np.max(kurts):+.3f}")

## ─────────────────────────────────────────────────────
## 📊 7.2 WEIGHT DISTRIBUTION RUN FUNCTIONS
## ─────────────────────────────────────────────────────

In [ ]:
# =========================================================
# LOAD MODEL AND CALL BOTH
# =========================================================
file_date  = "20260320_064356"
model_path = os.path.join(main_directory, f"results/ensemble_mfcc_mslfb_resumed_{file_date}.keras")

model = load_model(model_path)
print("FP32 model loaded ✓")

for layer in model.layers:
    print(f"  '{layer.name}'  →  {type(layer).__name__}")
    if hasattr(layer, "layers"):
        for sub in layer.layers:
            print(f"      '{sub.name}'")

plot_all_weights(model)
df_pvalues = plot_pvalue_by_section(model)

In [ ]:
from tensorflow.keras.models import load_model

file_date  = "20260320_064356"
model_path = os.path.join(main_directory, f"results/ensemble_mfcc_mslfb_resumed_{file_date}.keras")
model      = load_model(model_path)

print("=== FP32 Weight Distributions (per channel) ===")
plot_weight_distributions_per_channel(model)

In [ ]:
# =========================================================
# CALL
# =========================================================
df_stats = plot_weight_statistics(model)

In [ ]:
# =========================================================
# LOAD AND RUN
# =========================================================
file_date  = "20260320_064356"
model_path = os.path.join(main_directory, f"results/ensemble_mfcc_mslfb_resumed_{file_date}.keras")

model = load_model(model_path)
mean_ranks, ci_half = compare_layer_distributions(model)

In [ ]:
calib_ds = train_ds.unbatch().batch(32).take(1024 // 32)

file_date  = "20260320_064356"
model_path = os.path.join(main_directory, f"results/ensemble_mfcc_mslfb_resumed_{file_date}.keras")

result = plot_sensitivity_distribution(
    model_path,
    calib_ds,
    num_bits     = 4,
    p_aggressive = 75,
    p_full_prec  = 90,
)

# Use the data-driven thresholds in sensitivity-aware quantization
model = load_fresh_model()
sensitivity_aware_quantize_model(
    model,
    model_path         = model_path,
    calib_ds           = calib_ds,
    high_sens_thresh   = result['thresh_full_prec'],
    medium_sens_thresh = result['thresh_aggressive'],
    low_sens_bits      = 3,
    medium_sens_bits   = 5,
    high_sens_bits     = 8,
    label              = "Sensitivity-Aware (data-driven thresholds)",
)
compile_and_evaluate(model, test_ds, label="Sensitivity-Aware (data-driven thresholds)")

In [ ]:
from tensorflow.keras.models import load_model
# ---------------------------------------------------------
# LOAD MODEL
# ---------------------------------------------------------
file_date  = "20260320_064356"
model_path = os.path.join(main_directory, f"results/ensemble_mfcc_mslfb_resumed_{file_date}.keras")
model      = load_model(model_path)

# Before quantization
print("=== FP32 Weight Distributions ===")
plot_weight_distributions(
    model,
    save_path='/content/drive/MyDrive/results/weight_distributions_fp32.png'
)

In [ ]:
# =========================================================
# RUN
# =========================================================
file_date  = "20260320_064356"
model_path = os.path.join(main_directory, f"results/ensemble_mfcc_mslfb_resumed_{file_date}.keras")

calib_ds = train_ds.unbatch().batch(32).take(1024 // 32)

df_sensitivity = plot_sensitivity_by_section_layerwise(
    model_path, calib_ds, num_bits=4
)

In [ ]:

# =========================================================
# RUN
# =========================================================
file_date  = "20260320_064356"
model_path = os.path.join(main_directory, f"results/ensemble_mfcc_mslfb_resumed_{file_date}.keras")
calib_ds   = train_ds.unbatch().batch(32).take(1024 // 32)

df_rank = plot_sensitivity_rank_by_section(model_path, calib_ds, num_bits=4)

In [ ]:
file_date  = "20260320_064356"
model_path = os.path.join(main_directory, f"results/ensemble_mfcc_mslfb_resumed_{file_date}.keras")

# ── RUN ───────────────────────────────────────────────────────────────────────
calib_ds = train_ds.unbatch().batch(32).take(1024 // 32)

# Step 1: analyse sensitivity and get bit assignments
df_sensitivity = layer_sensitivity_analysis_with_clipping(
    model_path               = model_path,
    calib_ds                 = calib_ds,
    candidate_bits           = (2, 3, 4, 8),
    clip_percentiles         = (0.95, 0.99, 0.999, 1.0),
    mse_threshold_percentile = 0.3,
)

# Step 2: apply to model and evaluate
for label, df in [("Sensitivity+Clip", df_sensitivity)]:
    model = load_model(model_path)
    quantize_model_from_df(model, df)
    compile_and_evaluate(model, test_ds, label=label)
    plot_weight_distribution_comparison(fp32_model, model, label=label)

# ─────────────────────────────────────────────────────
# ⚙️ 8. STANDARD QUANTIZATION METHODS
# ─────────────────────────────────────────────────────

## ─────────────────────────────────────────────────────
## ⚙️ 8.1 QUANTIZATION FUNCTIONS
## ─────────────────────────────────────────────────────

In [ ]:
from sklearn.cluster import KMeans

def quantize_dequantize_uniform(model, num_bits: int = 8, symmetric: bool = True, label=None, verbose=True):
    if label is None:
        label = f"Uniform {num_bits}-bit (per-layer)"
    layers = collect_all_weight_layers(model)

    if verbose:
        print(f"\n{'='*60}")
        print(f"  {label}")
        print(f"{'='*60}")

    for layer in layers:
        weights = layer.get_weights()
        if not weights:
            continue
        W  = weights[0]
        b  = weights[1] if len(weights) > 1 else None

        W_t   = tf.convert_to_tensor(W, dtype=tf.float32)
        if symmetric:
            qmax     = 2 ** (num_bits - 1) - 1
            qmin     = -qmax - 1
            w_absmax = tf.reduce_max(tf.abs(W_t))
            if tf.equal(w_absmax, 0):
                Wq = W
            else:
                scale = w_absmax / tf.cast(qmax, tf.float32)
                Wq    = (tf.clip_by_value(tf.round(W_t / scale), qmin, qmax) * scale).numpy()
        else:
            qmin_v, qmax_v = 0, 2 ** num_bits - 1
            w_min = tf.reduce_min(W_t)
            w_max = tf.reduce_max(W_t)
            if tf.equal(w_min, w_max):
                Wq = W
            else:
                scale      = (w_max - w_min) / tf.cast((qmax_v - qmin_v), tf.float32)
                zero_point = tf.clip_by_value(tf.round(tf.cast(qmin_v, tf.float32) - w_min / scale), qmin_v, qmax_v)
                Wq         = ((tf.clip_by_value(tf.round(W_t / scale + zero_point), qmin_v, qmax_v) - zero_point) * scale).numpy()

        layer.set_weights([Wq, b] if b is not None else [Wq])
        if verbose:
            is_conv = isinstance(layer, tf.keras.layers.Conv2D)
            print(f"  {layer.name:<22} {'Conv2D' if is_conv else 'Dense':<8} → {num_bits}-bit")

    if verbose:
        print("\nQuantization complete.")


def quantize_dequantize_logarithmic(model, num_bits: int = 8, label=None, verbose=True):
    if label is None:
        label = f"Logarithmic {num_bits}-bit"
    layers = collect_all_weight_layers(model)

    if verbose:
        print(f"\n{'='*60}")
        print(f"  {label}")
        print(f"{'='*60}")

    for layer in layers:
        weights = layer.get_weights()
        if not weights:
            continue
        W  = weights[0]
        b  = weights[1] if len(weights) > 1 else None

        W_t    = tf.convert_to_tensor(W, dtype=tf.float32)
        eps    = 1e-12
        abs_w  = tf.abs(W_t) + eps
        log2_w = tf.math.log(abs_w) / tf.math.log(tf.constant(2.0, dtype=tf.float32))
        e_min  = -(num_bits * 2)
        e_max  =  (num_bits * 2)
        e_rounded = tf.clip_by_value(tf.round(log2_w), e_min, e_max)
        Wq = tf.where(
            tf.equal(W_t, 0.0),
            tf.zeros_like(W_t),
            tf.sign(W_t) * (2.0 ** e_rounded)
        ).numpy()

        layer.set_weights([Wq, b] if b is not None else [Wq])
        if verbose:
            is_conv = isinstance(layer, tf.keras.layers.Conv2D)
            print(f"  {layer.name:<22} {'Conv2D' if is_conv else 'Dense':<8} → {num_bits}-bit")

    if verbose:
        print("\nQuantization complete.")


def quantize_dequantize_kmeans(model, num_bits: int = 8, label=None, verbose=True):
    if label is None:
        label = f"K-Means {num_bits}-bit"
    layers = collect_all_weight_layers(model)

    if verbose:
        print(f"\n{'='*60}")
        print(f"  {label}")
        print(f"{'='*60}")

    for layer in layers:
        weights = layer.get_weights()
        if not weights:
            continue
        W  = weights[0]
        b  = weights[1] if len(weights) > 1 else None

        w_flat       = W.reshape(-1, 1)
        n_init       = 1 if len(w_flat) > 10000 else 5
        num_clusters = min(2 ** num_bits, max(len(w_flat) // 2, 1))
        kmeans       = KMeans(n_clusters=num_clusters, n_init=n_init, random_state=42)
        kmeans.fit(w_flat)
        centers = kmeans.cluster_centers_.squeeze()
        Wq      = centers[kmeans.labels_].reshape(W.shape).astype(np.float32)

        layer.set_weights([Wq, b] if b is not None else [Wq])
        if verbose:
            is_conv = isinstance(layer, tf.keras.layers.Conv2D)
            print(f"  {layer.name:<22} {'Conv2D' if is_conv else 'Dense':<8} → {num_bits}-bit ({num_clusters} centroids)")

    if verbose:
        print("\nQuantization complete.")


def quantize_dequantize_lloydmax(model, num_bits: int = 8, label=None, verbose=True):
    if label is None:
        label = f"Lloyd-Max {num_bits}-bit"
    layers = collect_all_weight_layers(model)

    if verbose:
        print(f"\n{'='*60}")
        print(f"  {label}")
        print(f"{'='*60}")

    for layer in layers:
        weights = layer.get_weights()
        if not weights:
            continue
        W  = weights[0]
        b  = weights[1] if len(weights) > 1 else None

        w      = W.reshape(-1)
        levels = 2 ** num_bits
        c      = np.linspace(w.min(), w.max(), levels)
        for _ in range(20):
            b_bounds = (c[:-1] + c[1:]) / 2
            idx      = np.clip(np.digitize(w, b_bounds), 0, levels - 1)
            c_new    = c.copy()
            for i in range(levels):
                assigned = w[idx == i]
                if len(assigned) > 0:
                    c_new[i] = assigned.mean()
            if np.max(np.abs(c_new - c)) < 1e-6:
                c = c_new
                break
            c = c_new
        b_bounds = (c[:-1] + c[1:]) / 2
        idx      = np.clip(np.digitize(w, b_bounds), 0, levels - 1)
        Wq       = c[idx].astype(np.float32).reshape(W.shape)

        layer.set_weights([Wq, b] if b is not None else [Wq])
        if verbose:
            is_conv = isinstance(layer, tf.keras.layers.Conv2D)
            print(f"  {layer.name:<22} {'Conv2D' if is_conv else 'Dense':<8} → {num_bits}-bit ({levels} levels)")

    if verbose:
        print("\nQuantization complete.")

In [ ]:
def uniform_quantize_model(model, num_bits, label=None):
    if label is None:
        label = f"Uniform {num_bits}-bit (per-channel)"

    layers = collect_all_weight_layers(model)

    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"  All layers → {num_bits}-bit per-channel uniform")
    print(f"{'='*60}")

    for layer in layers:
        weights = layer.get_weights()
        if not weights:
            continue
        W  = weights[0]
        b  = weights[1] if len(weights) > 1 else None
        Wq = uniform_quantize_layer(W, num_bits=num_bits)
        layer.set_weights([Wq, b] if b is not None else [Wq])
        is_conv = isinstance(layer, tf.keras.layers.Conv2D)
        print(f"  {layer.name:<22} {'Conv2D' if is_conv else 'Dense':<8} → {num_bits}-bit")

    print(f"\nQuantization complete.")

In [ ]:
def quantize_dequantize_groupwise(model, num_bits=8, group_size=64, label=None, verbose=True):
    if label is None:
        label = f"Group-wise {num_bits}-bit (group={group_size})"
    layers = collect_all_weight_layers(model)

    if verbose:
        print(f"\n{'='*60}")
        print(f"  {label}")
        print(f"{'='*60}")

    for layer in layers:
        weights = layer.get_weights()
        if not weights:
            continue
        W  = weights[0]
        b  = weights[1] if len(weights) > 1 else None

        out_ch        = W.shape[-1]
        ch_elements   = W[..., 0].size
        w_per_channel = W.reshape(ch_elements, out_ch)
        Wq            = np.zeros_like(w_per_channel)

        effective_group_size = min(group_size, ch_elements)

        if verbose and ch_elements < group_size:
            is_conv = isinstance(layer, tf.keras.layers.Conv2D)
            print(f"  INFO: {layer.name} ({'Conv2D' if is_conv else 'Dense'}) has "
                  f"{ch_elements} weights/channel < group_size={group_size} "
                  f"→ using per-channel (group_size={ch_elements})")

        for c in range(out_ch):
            for start in range(0, ch_elements, effective_group_size):
                end   = min(start + effective_group_size, ch_elements)
                group = w_per_channel[start:end, c]
                # uniform per-group quantization
                w_max = np.max(np.abs(group)) + 1e-8
                qmax  = 2 ** (num_bits - 1) - 1
                qmin  = -qmax - 1
                scale = w_max / qmax
                Wq[start:end, c] = np.clip(np.round(group / scale), qmin, qmax) * scale

        Wq = Wq.reshape(W.shape).astype(np.float32)
        layer.set_weights([Wq, b] if b is not None else [Wq])

        if verbose:
            is_conv          = isinstance(layer, tf.keras.layers.Conv2D)
            num_groups_per_ch = (ch_elements + effective_group_size - 1) // effective_group_size
            total_groups      = out_ch * num_groups_per_ch
            print(f"  {layer.name:<22} {'Conv2D' if is_conv else 'Dense':<8} → {num_bits}-bit | "
                  f"groups/ch={num_groups_per_ch} | total_groups={total_groups}")

    if verbose:
        print("\nQuantization complete.")

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from scipy.stats import kurtosis, skew

# =========================================================
# ADAPTIVE QUANTIZATION
#
# Parameters (all bitwidths can be tuned):
#
#   conv_bits         : bits for all Conv2D layers
#                       (Gaussian-like → can afford fewer bits)
#
#   dense_high_kurt_bits   : bits for Dense with kurtosis > kurt_high_thresh
#                            (heavy tails → needs more bits for outliers)
#
#   dense_mid_kurt_bits    : bits for Dense with kurt_low_thresh < kurtosis < kurt_high_thresh
#                            (moderate tails)
#
#   dense_low_kurt_bits    : bits for Dense with kurtosis < kurt_low_thresh
#                            (near-Gaussian → safe to compress)
#
#   kurt_high_thresh  : kurtosis threshold for "high" (default 4.0)
#   kurt_low_thresh   : kurtosis threshold for "low"  (default 1.0)
#
# CONSERVATIVE (default):   conv=4, dense high=8, mid=6, low=4
# AGGRESSIVE:               conv=3, dense high=6, mid=4, low=3
# ULTRA-AGRESSIVE:          conv=2, dense high=4, mid=3, low=2
# =========================================================

def adaptive_quantize_model(
    model,
    conv_bits            = 4,
    dense_high_kurt_bits = 8,
    dense_mid_kurt_bits  = 6,
    dense_low_kurt_bits  = 4,
    kurt_high_thresh     = 4.0,
    kurt_low_thresh      = 1.0,
    label                = "Adaptive"
):
    layers = collect_all_weight_layers(model)

    print(f"\n{'='*75}")
    print(f"  {label}")
    print(f"  Conv={conv_bits}bit | Dense: high_kurt={dense_high_kurt_bits}bit "
          f"mid_kurt={dense_mid_kurt_bits}bit low_kurt={dense_low_kurt_bits}bit")
    print(f"  Thresholds: kurt>{kurt_high_thresh} = high | kurt<{kurt_low_thresh} = low")
    print(f"{'='*75}")
    print(f"{'Layer':<20} {'Type':<8} {'Kurt':>7} {'Skew':>7} {'Std':>7}  Strategy")
    print("-" * 75)

    for layer in layers:
        weights = layer.get_weights()
        if not weights:
            continue

        W    = weights[0]
        b    = weights[1] if len(weights) > 1 else None
        s    = compute_layer_stats(layer)
        kurt = s['kurtosis']
        sk   = s['skew']
        std  = s['std']

        is_conv = isinstance(layer, tf.keras.layers.Conv2D)

        if is_conv:
            Wq       = uniform_quantize_layer(W, num_bits=conv_bits)
            strategy = f"Uniform {conv_bits}-bit per-channel"
        else:
            if kurt > kurt_high_thresh:
                Wq       = uniform_quantize_layer(W, num_bits=dense_high_kurt_bits)
                strategy = f"Uniform {dense_high_kurt_bits}-bit (high kurtosis)"
            elif kurt > kurt_low_thresh:
                Wq       = uniform_quantize_layer(W, num_bits=dense_mid_kurt_bits)
                strategy = f"Uniform {dense_mid_kurt_bits}-bit (moderate kurtosis)"
            else:
                Wq       = uniform_quantize_layer(W, num_bits=dense_low_kurt_bits)
                strategy = f"Uniform {dense_low_kurt_bits}-bit (near-Gaussian)"

        print(f"{layer.name:<20} {'Conv2D' if is_conv else 'Dense':<8} "
              f"{kurt:>7.2f} {sk:>7.2f} {std:>7.4f}  {strategy}")

        if b is not None:
            layer.set_weights([Wq, b])
        else:
            layer.set_weights([Wq])

    print("\nQuantization complete.")


In [ ]:
def sensitivity_aware_quantize_model(
    model,
    model_path,
    calib_ds,
    high_sens_thresh   = 0.01,
    medium_sens_thresh = 0.001,
    low_sens_bits      = 3,
    medium_sens_bits   = 5,
    high_sens_bits     = 8,
    fallback_bits      = 4,
    label              = "Sensitivity-Aware",
    verbose            = True,
):
    # ── Step 1: measure sensitivity ──────────────────────────────────────────
    base_model  = load_model(model_path)
    sens_layers = collect_all_weight_layers(base_model)

    sensitivity_map  = {}   # layer_name → {'rel_mse', 'tier', 'bits'}
    sens_layers_list = []   # ordered list of names for positional matching

    if verbose:
        print(f"\n{'='*75}")
        print(f"  SENSITIVITY MEASUREMENT PASS")
        print(f"  Thresholds: HIGH > {high_sens_thresh} | "
              f"MEDIUM > {medium_sens_thresh} | LOW ≤ {medium_sens_thresh}")
        print(f"  Bits:       HIGH={high_sens_bits} | "
              f"MEDIUM={medium_sens_bits} | LOW={low_sens_bits}")
        print(f"{'='*75}")
        print(f"{'Layer':<20} {'Type':<8} {'Kurt':>7} {'Rel MSE':>12}  Tier  → Bits")
        print("-" * 75)

    for target_layer in sens_layers:
        weights = target_layer.get_weights()
        if not weights:
            continue

        W = weights[0]
        b = weights[1] if len(weights) > 1 else None

        fp_inputs, fp_outputs = capture_layer_activations(
            base_model, target_layer, calib_ds
        )

        if not fp_inputs:
            sensitivity_map[target_layer.name] = {
                'rel_mse': None,
                'tier':    'FALLBACK',
                'bits':    fallback_bits,
            }
            sens_layers_list.append(target_layer.name)
            continue

        Wq             = uniform_quantize_layer(W, num_bits=fallback_bits)
        mse_list       = []
        var_list       = []

        for x_fp, y_fp in zip(fp_inputs, fp_outputs):
            y_q = manual_forward(
                target_layer,
                x_fp,
                tf.constant(Wq),
                tf.constant(b) if b is not None else None,
            )
            mse_list.append(tf.reduce_mean((y_fp - y_q) ** 2).numpy())
            var_list.append(tf.reduce_mean(y_fp ** 2).numpy())

        mean_mse = np.mean(mse_list)
        fp_var   = np.mean(var_list)
        rel_mse  = mean_mse / (fp_var + 1e-10)

        if rel_mse > high_sens_thresh:
            tier = 'HIGH'
            bits = high_sens_bits
        elif rel_mse > medium_sens_thresh:
            tier = 'MEDIUM'
            bits = medium_sens_bits
        else:
            tier = 'LOW'
            bits = low_sens_bits

        sensitivity_map[target_layer.name] = {
            'rel_mse': rel_mse,
            'tier':    tier,
            'bits':    bits,
        }
        sens_layers_list.append(target_layer.name)

        if verbose:
            is_conv    = isinstance(target_layer, tf.keras.layers.Conv2D)
            s          = compute_layer_stats(target_layer)
            tier_color = {'HIGH': '⬛', 'MEDIUM': '🟧', 'LOW': '🟩'}.get(tier, '  ')
            print(f"{target_layer.name:<20} {'Conv2D' if is_conv else 'Dense':<8} "
                  f"{s['kurtosis']:>7.2f} {rel_mse:>12.6f}  "
                  f"{tier_color} {tier:<6} → {bits}-bit")

    del base_model

    # ── Step 2: apply quantization ───────────────────────────────────────────
    quant_layers = collect_all_weight_layers(model)

    if verbose:
        print(f"\n{'='*75}")
        print(f"  QUANTIZATION PASS  —  {label}")
        print(f"{'='*75}")

    bits_used = []
    for i, layer in enumerate(quant_layers):
        weights = layer.get_weights()
        if not weights:
            continue

        W = weights[0]
        b = weights[1] if len(weights) > 1 else None

        # Try name match first, fall back to positional match
        info = sensitivity_map.get(layer.name)
        if info is None and i < len(sens_layers_list):
            info = sensitivity_map.get(sens_layers_list[i])

        bits = info['bits'] if info else fallback_bits
        tier = info['tier'] if info else 'FALLBACK'

        Wq = uniform_quantize_layer(W, num_bits=bits)

        if verbose:
            is_conv = isinstance(layer, tf.keras.layers.Conv2D)
            print(f"  {layer.name:<22} {'Conv2D' if is_conv else 'Dense':<8} "
                  f"{tier:<8} → quantized to {bits}-bit")

        layer.set_weights([Wq, b] if b is not None else [Wq])
        bits_used.append(bits)

    avg_bits = np.mean(bits_used) if bits_used else fallback_bits
    print(f"\n  ✓ Done. Layers quantized: {len(bits_used)} | "
          f"Avg bits: {avg_bits:.2f} | "
          f"Bit counts used: {sorted(set(bits_used))}")

    return sensitivity_map

In [ ]:
def sensitivity_quartile_greedy_sweep(
    model_path,
    calib_ds,
    test_ds,
    valid_bits       = [8, 6, 5, 4, 3, 2],  # ordered HIGH → LOW
    accuracy_floor   = 0.85,
    measurement_bits = 4,
    verbose          = True,
):
    """
    Greedy top-down quartile bit-width search.

    Layers are split into 4 sensitivity quartiles (Q1=least, Q4=most sensitive).
    Constraint: bits(Q1) <= bits(Q2) <= bits(Q3) <= bits(Q4).
    Each quartile is swept downward from highest to lowest bits.
    Stops when accuracy drops below accuracy_floor and locks at last good value.
    """

    def quantize_and_evaluate(bits_map, quartile_map, sens_layer_names):
        model        = load_model(model_path)
        quant_layers = collect_all_weight_layers(model)

        for i, layer in enumerate(quant_layers):
            weights = layer.get_weights()
            if not weights:
                continue
            W          = weights[0]
            b          = weights[1] if len(weights) > 1 else None
            layer_name = sens_layer_names[i] if i < len(sens_layer_names) else None
            q          = quartile_map.get(layer_name, 4)
            bits       = bits_map[q]
            Wq         = uniform_quantize_layer(W, num_bits=bits)
            layer.set_weights([Wq, b] if b is not None else [Wq])

        model.compile(
            loss      = 'categorical_crossentropy',
            optimizer = 'adam',
            metrics   = ['accuracy',
                         tf.keras.metrics.Precision(name='precision'),
                         tf.keras.metrics.Recall(name='recall')]
        )
        res             = model.evaluate(test_ds, verbose=0)
        loss, acc, prec, rec = res[:4]
        avg_bits        = np.mean([bits_map[q] for q in quartile_map.values()])
        del model
        return loss, acc, prec, rec, avg_bits

    # ── Step 1: measure sensitivity ──────────────────────────────────────────
    print("=" * 70)
    print("  SENSITIVITY MEASUREMENT")
    print("=" * 70)

    base_model    = load_model(model_path)
    sens_layers   = collect_all_weight_layers(base_model)
    layer_records = []

    for target_layer in sens_layers:
        weights = target_layer.get_weights()
        if not weights:
            continue
        W = weights[0]
        b = weights[1] if len(weights) > 1 else None

        fp_inputs, fp_outputs = capture_layer_activations(
            base_model, target_layer, calib_ds
        )

        if not fp_inputs:
            layer_records.append({
                'name':    target_layer.name,
                'rel_mse': 0.0,
                'type':    'Conv2D' if isinstance(target_layer,
                           tf.keras.layers.Conv2D) else 'Dense',
            })
            continue

        Wq             = uniform_quantize_layer(W, num_bits=measurement_bits)
        mse_list, var_list = [], []

        for x_fp, y_fp in zip(fp_inputs, fp_outputs):
            y_q = manual_forward(
                target_layer, x_fp,
                tf.constant(Wq),
                tf.constant(b) if b is not None else None,
            )
            mse_list.append(tf.reduce_mean((y_fp - y_q) ** 2).numpy())
            var_list.append(tf.reduce_mean(y_fp ** 2).numpy())

        rel_mse = np.mean(mse_list) / (np.mean(var_list) + 1e-10)
        layer_records.append({
            'name':    target_layer.name,
            'rel_mse': rel_mse,
            'type':    'Conv2D' if isinstance(target_layer,
                       tf.keras.layers.Conv2D) else 'Dense',
        })

    del base_model

    df_layers = pd.DataFrame(layer_records).sort_values('rel_mse').reset_index(drop=True)
    n         = len(df_layers)

    # ── Step 2: assign quartiles ─────────────────────────────────────────────
    q_boundaries = [0, n//4, n//2, 3*n//4, n]
    quartile_map = {}

    print(f"\n{'Layer':<22} {'Type':<8} {'Rel MSE':>12}  Quartile")
    print("-" * 55)

    for i, row in df_layers.iterrows():
        if   i < q_boundaries[1]: q = 1
        elif i < q_boundaries[2]: q = 2
        elif i < q_boundaries[3]: q = 3
        else:                      q = 4
        quartile_map[row['name']] = q
        print(f"  {row['name']:<20} {row['type']:<8} {row['rel_mse']:>12.6f}  Q{q}")

    sens_layer_names = list(df_layers['name'])

    # ── Step 3: greedy top-down sweep ────────────────────────────────────────
    start_bits  = valid_bits[0]
    locked_bits = {1: start_bits, 2: start_bits, 3: start_bits, 4: start_bits}
    all_results = []

    print(f"\n{'='*70}")
    print(f"  GREEDY SWEEP | accuracy floor = {accuracy_floor} | "
          f"bits order = {valid_bits}")
    print(f"{'='*70}")

    for stage, quartile in enumerate([1, 2, 3, 4], 1):
        print(f"\n  STAGE {stage} — sweeping Q{quartile} downward "
              f"(Q{quartile} currently {locked_bits[quartile]}-bit)")
        print(f"  {'Bits':>6}  {'Avg':>6}  {'Acc':>7}  {'Prec':>7}  "
              f"{'Rec':>7}  Decision")
        print(f"  {'-'*60}")

        best_bits_this_stage = locked_bits[quartile]

        for bits in valid_bits:
            # monotonicity: can't be LOWER than less sensitive quartile
            if quartile > 1 and bits < locked_bits[quartile - 1]:
                if verbose:
                    print(f"  Q{quartile}={bits:>2}bit  —  skipped "
                          f"(below Q{quartile-1}={locked_bits[quartile-1]}-bit)")
                continue
            # monotonicity: can't be HIGHER than more sensitive quartile
            if quartile < 4 and bits > locked_bits[quartile + 1]:
                if verbose:
                    print(f"  Q{quartile}={bits:>2}bit  —  skipped "
                          f"(above Q{quartile+1}={locked_bits[quartile+1]}-bit)")
                continue

            trial_bits = dict(locked_bits)
            trial_bits[quartile] = bits

            loss, acc, prec, rec, avg_bits = quantize_and_evaluate(
                trial_bits, quartile_map, sens_layer_names
            )

            all_results.append({
                'stage':    stage,
                'quartile': quartile,
                'Q1_bits':  trial_bits[1],
                'Q2_bits':  trial_bits[2],
                'Q3_bits':  trial_bits[3],
                'Q4_bits':  trial_bits[4],
                'avg_bits': round(avg_bits, 2),
                'loss':     round(loss,  4),
                'accuracy': round(acc,   4),
                'precision':round(prec,  4),
                'recall':   round(rec,   4),
            })

            decision = "✓ above floor"
            if acc >= accuracy_floor:
                best_bits_this_stage = bits
            else:
                decision = "✗ below floor — stop"

            print(f"  Q{quartile}={bits:>2}bit  {avg_bits:>6.2f}  "
                  f"{acc:>7.4f}  {prec:>7.4f}  {rec:>7.4f}  {decision}")

            if acc < accuracy_floor:
                break

        locked_bits[quartile] = best_bits_this_stage
        print(f"\n  → Q{quartile} locked at {locked_bits[quartile]}-bit")

    # ── Step 4: final evaluation ──────────────────────────────────────────────
    _, final_acc, final_prec, final_rec, final_avg = quantize_and_evaluate(
        locked_bits, quartile_map, sens_layer_names
    )

    print(f"\n{'='*70}")
    print(f"  FINAL CONFIGURATION")
    print(f"{'='*70}")
    print(f"  Q1 (least sensitive): {locked_bits[1]}-bit")
    print(f"  Q2:                   {locked_bits[2]}-bit")
    print(f"  Q3:                   {locked_bits[3]}-bit")
    print(f"  Q4 (most sensitive):  {locked_bits[4]}-bit")
    print(f"  Avg bits:             {final_avg:.2f}")
    print(f"  Accuracy:             {final_acc:.4f}")
    print(f"  Precision:            {final_prec:.4f}")
    print(f"  Recall:               {final_rec:.4f}")

    df_all = pd.DataFrame(all_results)
    return df_all, locked_bits, quartile_map, sens_layer_names

In [ ]:
def quantize_activations_only_per_layer(model_path, calib_ds, num_bits, clip_percentile=0.999):

    def _quant_sym(act_np, scale, num_bits):
        qmax = 2 ** (num_bits - 1) - 1
        qmin = -qmax - 1
        act_q = np.clip(np.round(act_np / scale), qmin, qmax) * scale
        return act_q.astype(np.float32)

    def _iter_all_layers(model):
        for layer in model.layers:
            yield layer
            if hasattr(layer, 'layers'):
                yield from _iter_all_layers(layer)

    # ── Step 1: calibrate scales ────────────────────────────────────────────
    base_model  = load_model(model_path)
    sens_layers = collect_all_weight_layers(base_model)

    layer_scales = {}
    layer_mses   = {}

    print(f"\n{'='*65}")
    print(f"  ACTIVATION-ONLY QUANTIZATION  —  {num_bits}-bit symmetric PER-LAYER")
    print(f"{'='*65}")
    print(f"  {'Layer':<26} {'Scale':>12}  {'RelMSE':>10}")
    print(f"  {'-'*55}")

    for layer in sens_layers:
        _, fp_outputs = capture_layer_activations(base_model, layer, calib_ds)
        if not fp_outputs:
            continue

        all_acts = np.concatenate(
            [a.numpy() if hasattr(a, 'numpy') else a for a in fp_outputs],
            axis=0
        )

        qmax = 2 ** (num_bits - 1) - 1

        # 🔥 SINGLE SCALE FOR ENTIRE LAYER
        scale = np.percentile(np.abs(all_acts), clip_percentile * 100) / qmax + 1e-8

        act_q   = _quant_sym(all_acts, scale, num_bits)
        rel_mse = np.mean((all_acts - act_q) ** 2) / (np.mean(all_acts ** 2) + 1e-10)

        layer_scales[layer.name] = float(scale)
        layer_mses[layer.name]   = rel_mse

        print(f"  {layer.name:<26} {scale:>12.6f}  {rel_mse:>10.6f}")

    del base_model

    # ── Step 2: patch model ────────────────────────────────────────────────
    orig_model = load_model(model_path)

    def make_quant_fn(scale, nb):
        scale_const = tf.constant(scale, dtype=tf.float32)
        qmax_f = float(2 ** (nb - 1) - 1)
        qmin_f = -qmax_f - 1.0

        @tf.function
        def quant_fn(x):
            x_q = tf.clip_by_value(tf.round(x / scale_const), qmin_f, qmax_f)
            return x_q * scale_const

        return quant_fn

    def make_patched_call(orig_call, qfn):
        def patched_call(inputs, **kwargs):
            out = orig_call(inputs, **kwargs)
            return qfn(out)
        return patched_call

    for layer in _iter_all_layers(orig_model):
        if layer.name not in layer_scales:
            continue

        quant_fn   = make_quant_fn(layer_scales[layer.name], num_bits)
        layer.call = make_patched_call(layer.call, quant_fn)

    print(f"\n  Activation quantizers patched ✓  ({len(layer_scales)} layers)")
    return orig_model, layer_scales, layer_mses

In [ ]:
import copy
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import load_model
from scipy.stats import skew as scipy_skew, kurtosis as scipy_kurtosis


# =========================================================
# LAYER HELPERS
# =========================================================

def collect_all_weight_layers(model):
    layers = []

    def _collect(layer):
        if isinstance(layer, (tf.keras.layers.Conv2D, tf.keras.layers.Dense)):
            layers.append(layer)
        if hasattr(layer, "layers"):
            for sub in layer.layers:
                _collect(sub)

    for layer in model.layers:
        _collect(layer)

    return layers


def capture_layer_activations(model, layer, calib_ds):
    captured = {"inp": [], "out": []}
    original_call = layer.call

    def hooked_call(inputs, *args, **kwargs):
        result = original_call(inputs, *args, **kwargs)
        captured["inp"].append(tf.identity(inputs))
        captured["out"].append(tf.identity(result))
        return result

    layer.call = hooked_call

    for x_batch, _ in calib_ds:
        model(x_batch, training=False)

    layer.call = original_call

    return captured["inp"], captured["out"]


def manual_forward(layer, x, w, b):
    if isinstance(layer, tf.keras.layers.Conv2D):
        y = tf.nn.conv2d(
            x,
            w,
            strides=layer.strides,
            padding=layer.padding.upper(),
        )
        if b is not None:
            y = tf.nn.bias_add(y, b)

    elif isinstance(layer, tf.keras.layers.Dense):
        y = tf.matmul(x, w)
        if b is not None:
            y = y + b

    else:
        raise TypeError(f"Unsupported layer type: {type(layer)}")

    if layer.activation is not None:
        y = layer.activation(y)

    return y


# =========================================================
# WEIGHT QUANTIZATION HELPERS
# =========================================================

def _quantize_channel_symmetric(w_ch, num_bits, clip_percentile):
    clip_val = np.percentile(np.abs(w_ch), clip_percentile * 100)
    w_clipped = np.clip(w_ch, -clip_val, clip_val)

    qmax = 2 ** (num_bits - 1) - 1
    qmin = -qmax - 1

    scale = np.max(np.abs(w_clipped)) / qmax + 1e-8
    q = np.clip(np.round(w_clipped / scale), qmin, qmax)

    return q * scale


def _quantize_channel_asymmetric(w_ch, num_bits, clip_percentile):
    lo_clip = np.percentile(w_ch, (1 - clip_percentile) * 100)
    hi_clip = np.percentile(w_ch, clip_percentile * 100)

    w_clipped = np.clip(w_ch, lo_clip, hi_clip)

    n_levels = 2 ** num_bits
    w_min = w_clipped.min()
    w_max = w_clipped.max()

    scale = (w_max - w_min) / (n_levels - 1) + 1e-8
    zero_pt = np.round(-w_min / scale)

    q = np.clip(
        np.round(w_clipped / scale) + zero_pt,
        0,
        n_levels - 1,
    )

    return (q - zero_pt) * scale


def _is_genuinely_skewed(w_ch, z_skew=2.0, z_kurt=3.0):
    n = len(w_ch)

    if n < 4:
        return False

    sk = scipy_skew(w_ch)
    ku = scipy_kurtosis(w_ch)

    if not np.isfinite(sk) or not np.isfinite(ku):
        return False

    se_skew = np.sqrt(6.0 / n)
    se_kurt = np.sqrt(24.0 / n)

    return (
        abs(sk) > z_skew * se_skew
        and abs(ku) < z_kurt * se_kurt
    )


def _pick_qfn(w_ch, z_skew=2.0, z_kurt=3.0):
    if _is_genuinely_skewed(w_ch, z_skew, z_kurt):
        return _quantize_channel_asymmetric, "asym"
    return _quantize_channel_symmetric, "sym"


def _make_trial_weight_matrix(
    W_r,
    target_ch,
    target_bits,
    target_clip,
    quant_mode="symmetric",
    z_skew=2.0,
    z_kurt=3.0,
):
    W_trial = W_r.copy()
    w_ch = W_r[:, target_ch]

    if quant_mode == "symmetric":
        qfn = _quantize_channel_symmetric
    elif quant_mode == "adaptive":
        qfn, _ = _pick_qfn(w_ch, z_skew, z_kurt)
    else:
        raise ValueError("quant_mode must be 'symmetric' or 'adaptive'")

    W_trial[:, target_ch] = qfn(w_ch, target_bits, target_clip)
    return W_trial


def _channel_rel_mse(
    W_trial_r,
    W_shape,
    ch_idx,
    fp_inputs,
    fp_outputs,
    target_layer,
    b,
):
    W_const = tf.constant(W_trial_r.reshape(W_shape).astype(np.float32))
    b_const = tf.constant(b) if b is not None else None

    mse_list = []

    for x_fp, y_fp in zip(fp_inputs, fp_outputs):
        y_q = manual_forward(target_layer, x_fp, W_const, b_const)

        if len(y_fp.shape) == 4:
            err = tf.reduce_mean(
                (y_fp[..., ch_idx] - y_q[..., ch_idx]) ** 2
            ).numpy()
            var = tf.reduce_mean(y_fp[..., ch_idx] ** 2).numpy() + 1e-10
        else:
            err = tf.reduce_mean(
                (y_fp[:, ch_idx] - y_q[:, ch_idx]) ** 2
            ).numpy()
            var = tf.reduce_mean(y_fp[:, ch_idx] ** 2).numpy() + 1e-10

        mse_list.append(err / var)

    return float(np.mean(mse_list))


# =========================================================
# ANALYSIS: CLIP + 2-BIT SENSITIVITY
# =========================================================

def channel_sensitivity_analysis_budgeted(
    model_path,
    calib_ds,
    sensitivity_bits=2,
    clip_percentiles=(0.95, 0.99, 0.999, 1.0),
    analysis_quant_mode="symmetric",
    z_skew=2.0,
    z_kurt=3.0,
):
    base_model = load_model(model_path)
    sens_layers = collect_all_weight_layers(base_model)

    layer_records = []
    df_rows = []

    print("=" * 90)
    print("  CHANNEL SENSITIVITY ANALYSIS — WEIGHTS ONLY")
    print("=" * 90)
    print(f"  Sensitivity score: {sensitivity_bits}-bit RelMSE")
    print(f"  Clip candidates:   {clip_percentiles}")
    print(f"  Analysis mode:     {analysis_quant_mode}")
    print(f"  QDrop:             disabled")
    print(f"  Activations:       FP32")
    print("=" * 90)

    for target_layer in sens_layers:
        weights = target_layer.get_weights()

        if not weights:
            continue

        W = weights[0]
        b = weights[1] if len(weights) > 1 else None

        is_conv = isinstance(target_layer, tf.keras.layers.Conv2D)
        out_ch = W.shape[-1]
        W_r = W.reshape(-1, out_ch)

        fp_inputs, fp_outputs = capture_layer_activations(
            base_model,
            target_layer,
            calib_ds,
        )

        if not fp_inputs:
            continue

        print(
            f"\n  {target_layer.name} "
            f"({'Conv2D' if is_conv else 'Dense'}) — {out_ch} channels"
        )

        channel_best_clip = []
        channel_scores = []
        channel_skews = []
        channel_kurtoses = []
        channel_quant_schemes = []

        for c in range(out_ch):
            w_ch = W_r[:, c]

            sk = scipy_skew(w_ch)
            ku = scipy_kurtosis(w_ch)

            sk = float(sk) if np.isfinite(sk) else 0.0
            ku = float(ku) if np.isfinite(ku) else 0.0

            _, scheme = _pick_qfn(w_ch, z_skew, z_kurt)

            best_clip = 1.0
            best_score = float("inf")

            for clip_pct in clip_percentiles:
                W_trial_r = _make_trial_weight_matrix(
                    W_r=W_r,
                    target_ch=c,
                    target_bits=sensitivity_bits,
                    target_clip=clip_pct,
                    quant_mode=analysis_quant_mode,
                    z_skew=z_skew,
                    z_kurt=z_kurt,
                )

                rel_mse = _channel_rel_mse(
                    W_trial_r=W_trial_r,
                    W_shape=W.shape,
                    ch_idx=c,
                    fp_inputs=fp_inputs,
                    fp_outputs=fp_outputs,
                    target_layer=target_layer,
                    b=b,
                )

                if rel_mse < best_score:
                    best_score = rel_mse
                    best_clip = clip_pct

            channel_best_clip.append(best_clip)
            channel_scores.append(best_score)
            channel_skews.append(sk)
            channel_kurtoses.append(ku)
            channel_quant_schemes.append(scheme)

            df_rows.append({
                "layer": target_layer.name,
                "type": "Conv2D" if is_conv else "Dense",
                "channel": c,
                "best_clip": best_clip,
                "sensitivity_score": best_score,
                "sensitivity_source": f"{sensitivity_bits}b_mse",
                f"{sensitivity_bits}b_mse": best_score,
                "skewness": sk,
                "kurtosis": ku,
                "n_weights": len(w_ch),
                "quant_scheme": scheme,
            })

            if c % 16 == 0 or c == out_ch - 1:
                print(
                    f"    ch {c:>4}/{out_ch} | "
                    f"{scheme:<4} | clip={best_clip:.3f} | "
                    f"score={best_score:.6f}"
                )

        layer_records.append({
            "layer_name": target_layer.name,
            "type": "Conv2D" if is_conv else "Dense",
            "out_ch": out_ch,
            "W": W,
            "b": b,
            "channel_best_clip": channel_best_clip,
            "sensitivity_scores": channel_scores,
            "channel_skews": channel_skews,
            "channel_kurtoses": channel_kurtoses,
            "channel_quant_schemes": channel_quant_schemes,
            "assigned_bits": None,
        })

    del base_model

    df_channels = pd.DataFrame(df_rows)

    print("\n" + "=" * 90)
    print("  ANALYSIS COMPLETE")
    print("=" * 90)
    print(f"  Channels analyzed: {len(df_channels)}")
    print("  Final weight bits are assigned by budget next.")

    return df_channels, layer_records


# =========================================================
# APPLY BIT BUDGET
# =========================================================

def apply_bit_budget_to_records(
    df_channels,
    layer_records,
    budget,
    score_col="sensitivity_score",
):
    if not np.isclose(sum(budget.values()), 1.0):
        raise ValueError("Budget fractions must sum to 1.0")

    df = df_channels.copy()
    bits_sorted = sorted(budget.keys())
    n = len(df)

    raw_counts = {b: budget[b] * n for b in bits_sorted}
    counts = {b: int(np.floor(raw_counts[b])) for b in bits_sorted}

    remainder = n - sum(counts.values())

    fractional_order = sorted(
        bits_sorted,
        key=lambda b: raw_counts[b] - counts[b],
        reverse=True,
    )

    for b in fractional_order[:remainder]:
        counts[b] += 1

    df_ranked = df.sort_values(score_col, ascending=True).reset_index(drop=True)

    assigned = []
    for b in bits_sorted:
        assigned.extend([b] * counts[b])

    df_ranked["assigned_bits"] = assigned

    assignment_map = {
        (row["layer"], int(row["channel"])): int(row["assigned_bits"])
        for _, row in df_ranked.iterrows()
    }

    new_records = copy.deepcopy(layer_records)

    for rec in new_records:
        rec["assigned_bits"] = [
            assignment_map[(rec["layer_name"], c)]
            for c in range(rec["out_ch"])
        ]

    df_out = df_ranked.sort_values(["layer", "channel"]).reset_index(drop=True)

    avg_channel_bits = float(df_out["assigned_bits"].mean())
    avg_weight_bits = float(
        np.average(df_out["assigned_bits"], weights=df_out["n_weights"])
    )

    print("\n" + "=" * 80)
    print("  BIT BUDGET APPLIED")
    print("=" * 80)
    print(f"  Budget: {budget}")
    print(f"  Avg bits/channel: {avg_channel_bits:.3f}")
    print(f"  Avg bits/weight:  {avg_weight_bits:.3f}")

    for b in bits_sorted:
        n_b = int((df_out["assigned_bits"] == b).sum())
        print(f"  {b}-bit: {n_b:>4} channels ({100*n_b/n:.1f}%)")

    return df_out, new_records, {
        "budget": budget,
        "counts": counts,
        "avg_channel_bits": avg_channel_bits,
        "avg_weight_bits": avg_weight_bits,
    }


# =========================================================
# QUANTIZE MODEL — WEIGHTS ONLY
# =========================================================

def quantize_model_from_budget_records(
    model,
    layer_records,
    weight_mode="uniform_symmetric",
    z_skew=2.0,
    z_kurt=3.0,
    scale_bits=0,
    break_point="norm",
    pw_opt=2,
    approximate=True,
):
    """
    weight_mode:
      - uniform_symmetric
      - uniform_adaptive
      - pwlq
    """

    if weight_mode not in ["uniform_symmetric", "uniform_adaptive", "pwlq"]:
        raise ValueError("Invalid weight_mode")

    name_to_record = {rec["layer_name"]: rec for rec in layer_records}

    print("\n" + "=" * 80)
    print(f"  WEIGHT QUANTIZATION: {weight_mode}")
    print("=" * 80)

    for layer in collect_all_weight_layers(model):
        weights = layer.get_weights()

        if not weights:
            continue

        rec = name_to_record.get(layer.name)

        if rec is None:
            continue

        if rec["assigned_bits"] is None:
            raise ValueError("assigned_bits missing. Apply budget first.")

        W = weights[0]
        b = weights[1] if len(weights) > 1 else None

        out_ch = W.shape[-1]
        W_r = W.reshape(-1, out_ch)

        assigned_bits = rec["assigned_bits"]
        channel_clips = rec["channel_best_clip"]

        bits_counter = {}

        if weight_mode in ["uniform_symmetric", "uniform_adaptive"]:
            Wq_r = np.zeros_like(W_r)

            for c in range(out_ch):
                bits_c = int(assigned_bits[c])
                clip_c = float(channel_clips[c])
                w_ch = W_r[:, c]

                if weight_mode == "uniform_symmetric":
                    qfn = _quantize_channel_symmetric
                else:
                    qfn, _ = _pick_qfn(w_ch, z_skew, z_kurt)

                Wq_r[:, c] = qfn(w_ch, bits_c, clip_c)
                bits_counter[bits_c] = bits_counter.get(bits_c, 0) + 1

            Wq = Wq_r.reshape(W.shape).astype(np.float32)

        else:
            import torch

            if "piecewise_linear_quant" not in globals():
                raise ImportError("piecewise_linear_quant must be defined for PWLQ.")

            W_by_ch = W_r.T
            qw_list = []

            for c in range(out_ch):
                bits_c = int(assigned_bits[c])
                w_ch_t = torch.from_numpy(W_by_ch[c].astype(np.float32))

                qw, _, _ = piecewise_linear_quant(
                    w_ch_t,
                    bits=bits_c,
                    scale_bits=scale_bits,
                    break_point_approach=break_point,
                    pw_opt=pw_opt,
                    approximate=approximate,
                )

                if hasattr(qw, "detach"):
                    qw_np = qw.detach().cpu().numpy()
                else:
                    qw_np = np.asarray(qw)

                qw_list.append(qw_np)
                bits_counter[bits_c] = bits_counter.get(bits_c, 0) + 1

            Wq = np.stack(qw_list, axis=0).T.reshape(W.shape).astype(np.float32)

        layer.set_weights([Wq, b] if b is not None else [Wq])

        print(
            f"  {layer.name:<26} "
            f"avg={np.mean(assigned_bits):.2f}b | "
            + " | ".join([f"{k}b:{v}" for k, v in sorted(bits_counter.items())])
        )

    return model


# =========================================================
# EVALUATION
# =========================================================

def compile_and_evaluate(model, test_ds, label=""):
    model.compile(
        loss="categorical_crossentropy",
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        metrics=["accuracy"],
    )

    results = model.evaluate(test_ds, verbose=0)

    all_preds = []
    all_labels = []

    for x_batch, y_batch in test_ds:
        preds = model(x_batch, training=False)
        all_preds.append(tf.argmax(preds, axis=1).numpy())
        all_labels.append(tf.argmax(y_batch, axis=1).numpy())

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    classes = np.unique(all_labels)

    precisions = []
    recalls = []
    f1s = []

    for c in classes:
        tp = np.sum((all_preds == c) & (all_labels == c))
        fp = np.sum((all_preds == c) & (all_labels != c))
        fn = np.sum((all_preds != c) & (all_labels == c))

        p = tp / (tp + fp) if tp + fp > 0 else 0.0
        r = tp / (tp + fn) if tp + fn > 0 else 0.0
        f1 = 2 * p * r / (p + r) if p + r > 0 else 0.0

        precisions.append(p)
        recalls.append(r)
        f1s.append(f1)

    metrics = {
        "loss": float(results[0]),
        "accuracy": float(results[1]),
        "balanced_accuracy": float(np.mean(recalls)),
        "precision_macro": float(np.mean(precisions)),
        "recall_macro": float(np.mean(recalls)),
        "macro_f1": float(np.mean(f1s)),
    }

    print(f"\n{label}")
    for k, v in metrics.items():
        print(f"{k:<20}: {v:.4f}")

    return metrics


# =========================================================
# RUNNER — WEIGHT SEARCH ONLY
# =========================================================

def run_weight_budget_search(
    model_path,
    calib_ds,
    test_ds,
    budgets=None,
    weight_modes=("uniform_symmetric", "uniform_adaptive", "pwlq"),
    sensitivity_bits=2,
    clip_percentiles=(0.95, 0.99, 0.999, 1.0),
    analysis_quant_mode="symmetric",
    max_ba_drop_pct=5.0,
    z_skew=2.0,
    z_kurt=3.0,
    scale_bits=0,
    break_point="norm",
    pw_opt=2,
    approximate=True,
):
    if budgets is None:
        budgets = {
            "conservative_30_45_20_5": {2: 0.30, 3: 0.45, 4: 0.20, 8: 0.05},
            "balanced_40_40_15_5":     {2: 0.40, 3: 0.40, 4: 0.15, 8: 0.05},
            "aggressive_50_35_10_5":   {2: 0.50, 3: 0.35, 4: 0.10, 8: 0.05},
            "very_aggr_60_30_5_5":     {2: 0.60, 3: 0.30, 4: 0.05, 8: 0.05},
            "extreme_70_20_5_5":       {2: 0.70, 3: 0.20, 4: 0.05, 8: 0.05},
        }

    df_channels_base, layer_records_base = channel_sensitivity_analysis_budgeted(
        model_path=model_path,
        calib_ds=calib_ds,
        sensitivity_bits=sensitivity_bits,
        clip_percentiles=clip_percentiles,
        analysis_quant_mode=analysis_quant_mode,
        z_skew=z_skew,
        z_kurt=z_kurt,
    )

    fp32_model = load_model(model_path)
    fp32_metrics = compile_and_evaluate(fp32_model, test_ds, "FP32 baseline")
    fp32_ba = fp32_metrics["balanced_accuracy"]

    rows = []
    saved_configs = {}

    for budget_name, budget in budgets.items():
        for weight_mode in weight_modes:
            print("\n" + "=" * 100)
            print(f"  TESTING {budget_name} | {weight_mode}")
            print("=" * 100)

            df_budgeted, layer_records_budgeted, budget_info = apply_bit_budget_to_records(
                df_channels_base,
                layer_records_base,
                budget,
            )

            model_q = load_model(model_path)

            model_q = quantize_model_from_budget_records(
                model=model_q,
                layer_records=layer_records_budgeted,
                weight_mode=weight_mode,
                z_skew=z_skew,
                z_kurt=z_kurt,
                scale_bits=scale_bits,
                break_point=break_point,
                pw_opt=pw_opt,
                approximate=approximate,
            )

            metrics = compile_and_evaluate(
                model_q,
                test_ds,
                label=f"{budget_name} | {weight_mode}",
            )

            ba = metrics["balanced_accuracy"]
            ba_drop_abs = fp32_ba - ba
            ba_drop_pct = 100.0 * ba_drop_abs / max(fp32_ba, 1e-10)

            accepted = ba_drop_pct <= max_ba_drop_pct

            rows.append({
                "budget": budget_name,
                "weight_mode": weight_mode,
                "avg_channel_bits": budget_info["avg_channel_bits"],
                "avg_weight_bits": budget_info["avg_weight_bits"],
                "balanced_accuracy": ba,
                "ba_drop_abs": ba_drop_abs,
                "ba_drop_pct": ba_drop_pct,
                "accepted": accepted,
                "accuracy": metrics["accuracy"],
                "macro_f1": metrics["macro_f1"],
                "precision_macro": metrics["precision_macro"],
                "recall_macro": metrics["recall_macro"],
            })

            saved_configs[(budget_name, weight_mode)] = {
                "df_channels": df_budgeted,
                "layer_records": layer_records_budgeted,
                "budget_info": budget_info,
            }

    df_results = pd.DataFrame(rows)

    print("\n" + "=" * 100)
    print("FINAL SUMMARY — WEIGHTS ONLY")
    print("=" * 100)
    print(df_results.to_string(index=False))

    valid = df_results[df_results["accepted"]].copy()

    if len(valid) > 0:
        best = valid.sort_values(
            ["avg_weight_bits", "balanced_accuracy"],
            ascending=[True, False],
        ).iloc[0]

        print("\nSELECTED STRATEGY")
        print(best.to_string())
    else:
        best = None
        print("\nNo strategy stayed within the BA degradation limit.")

    return {
        "df_results": df_results,
        "best": best,
        "df_channels_base": df_channels_base,
        "layer_records_base": layer_records_base,
        "saved_configs": saved_configs,
        "fp32_metrics": fp32_metrics,
    }

## ─────────────────────────────────────────────────────
## ⚙️ 8.2 RUN QUANTIZATION FUNCTIONS
## ─────────────────────────────────────────────────────

In [ ]:
file_date  = "20260320_064356"
model_path = os.path.join(main_directory, f"results/ensemble_mfcc_mslfb_resumed_{file_date}.keras")

# ==========================================================
# FP32 BASELINE — evaluate once
# ==========================================================
fp32_model = load_model(model_path)
compile_and_evaluate(fp32_model, test_ds, label="FP32 Baseline")

In [ ]:
# =========================================================
# QUANTIZATION RUNNER SETUP
# =========================================================

# If you want to use the model saved at the end of training:
file_date  = "20260320_064356"
model_path = os.path.join(main_directory, f"results/ensemble_mfcc_mslfb_resumed_{file_date}.keras")
print("Using model:", model_path)


# =========================================================
# CALIBRATION DATASET
# =========================================================
# Deterministic, non-shuffled subset of training data.
# Used only for sensitivity analysis, not final evaluation.

num_calib_batches = 5

calib_ds = (
    tf.data.Dataset.from_tensor_slices((x_train_files, y_train_labels))
    .map(load_sample, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .take(num_calib_batches)
    .prefetch(tf.data.AUTOTUNE)
)

print(f"Calibration dataset: {num_calib_batches} batches × {batch_size} samples")

In [ ]:
budgets = {
    # Very safe: mostly 4-bit / 3-bit
    "very_safe_0_50_45_5":   {2: 0.00, 3: 0.50, 4: 0.45, 8: 0.05},
    "very_safe_5_50_40_5":   {2: 0.05, 3: 0.50, 4: 0.40, 8: 0.05},
    "very_safe_5_55_35_5":   {2: 0.05, 3: 0.55, 4: 0.35, 8: 0.05},

    # Safe
    "safe_10_50_35_5":       {2: 0.10, 3: 0.50, 4: 0.35, 8: 0.05},
    "safe_10_55_30_5":       {2: 0.10, 3: 0.55, 4: 0.30, 8: 0.05},
    "safe_10_60_25_5":       {2: 0.10, 3: 0.60, 4: 0.25, 8: 0.05},

    # Medium
    "medium_15_50_30_5":     {2: 0.15, 3: 0.50, 4: 0.30, 8: 0.05},
    "medium_15_55_25_5":     {2: 0.15, 3: 0.55, 4: 0.25, 8: 0.05},
    "medium_20_45_30_5":     {2: 0.20, 3: 0.45, 4: 0.30, 8: 0.05},
    "medium_20_50_25_5":     {2: 0.20, 3: 0.50, 4: 0.25, 8: 0.05},

    # More aggressive, but still less extreme than before
    "aggr_25_45_25_5":       {2: 0.25, 3: 0.45, 4: 0.25, 8: 0.05},
    "aggr_25_50_20_5":       {2: 0.25, 3: 0.50, 4: 0.20, 8: 0.05},
    "aggr_30_40_25_5":       {2: 0.30, 3: 0.40, 4: 0.25, 8: 0.05},
    "aggr_30_45_20_5":       {2: 0.30, 3: 0.45, 4: 0.20, 8: 0.05},
}

# =========================================================
# RUN WEIGHT-ONLY BUDGET SEARCH
# =========================================================

results = run_weight_budget_search(
    model_path=model_path,
    calib_ds=calib_ds,
    test_ds=test_ds,
    budgets=budgets,
    weight_modes=("uniform_symmetric", "uniform_adaptive", "pwlq"),
    sensitivity_bits=3,
    clip_percentiles=(0.99, 0.999, 1.0),
    analysis_quant_mode="symmetric",
    max_ba_drop_pct=5.0,
    scale_bits=0,
    break_point="norm",
    pw_opt=2,
    approximate=True,
)

In [ ]:
pwlq_aggressive_budgets = {
    # around the previous accepted boundary
    "pwlq_15_55_25_5": {2: 0.15, 3: 0.55, 4: 0.25, 8: 0.05},
    "pwlq_20_50_25_5": {2: 0.20, 3: 0.50, 4: 0.25, 8: 0.05},
    "pwlq_20_55_20_5": {2: 0.20, 3: 0.55, 4: 0.20, 8: 0.05},

    # more aggressive
    "pwlq_25_50_20_5": {2: 0.25, 3: 0.50, 4: 0.20, 8: 0.05},
    "pwlq_25_55_15_5": {2: 0.25, 3: 0.55, 4: 0.15, 8: 0.05},
    "pwlq_30_45_20_5": {2: 0.30, 3: 0.45, 4: 0.20, 8: 0.05},
    "pwlq_30_50_15_5": {2: 0.30, 3: 0.50, 4: 0.15, 8: 0.05},

    # aggressive frontier
    "pwlq_35_45_15_5": {2: 0.35, 3: 0.45, 4: 0.15, 8: 0.05},
    "pwlq_40_40_15_5": {2: 0.40, 3: 0.40, 4: 0.15, 8: 0.05},
    "pwlq_45_35_15_5": {2: 0.45, 3: 0.35, 4: 0.15, 8: 0.05},

    # extreme, only to find failure point
    "pwlq_50_35_10_5": {2: 0.50, 3: 0.35, 4: 0.10, 8: 0.05},
    "pwlq_60_30_5_5":  {2: 0.60, 3: 0.30, 4: 0.05, 8: 0.05},
}

results_pwlq_aggressive = run_weight_budget_search(
    model_path=model_path,
    calib_ds=calib_ds,
    test_ds=test_ds,

    budgets=pwlq_aggressive_budgets,

    weight_modes=("pwlq",),

    sensitivity_bits=3,
    clip_percentiles=(0.99, 0.999, 1.0),
    analysis_quant_mode="symmetric",

    max_ba_drop_pct=5.0,

    scale_bits=0,
    break_point="norm",
    pw_opt=2,
    approximate=True,
)

In [ ]:
# =========================================================
# MORE AGGRESSIVE PWLQ BUDGET SWEEP
# =========================================================

pwlq_more_aggressive_budgets = {
    # Start just beyond the previous best
    "pwlq_65_25_5_5":  {2: 0.65, 3: 0.25, 4: 0.05, 8: 0.05},
    "pwlq_70_20_5_5":  {2: 0.70, 3: 0.20, 4: 0.05, 8: 0.05},
    "pwlq_75_15_5_5":  {2: 0.75, 3: 0.15, 4: 0.05, 8: 0.05},
    "pwlq_80_10_5_5":  {2: 0.80, 3: 0.10, 4: 0.05, 8: 0.05},

    # Keep 8-bit protection, remove most 4-bit
    "pwlq_65_30_0_5":  {2: 0.65, 3: 0.30, 4: 0.00, 8: 0.05},
    "pwlq_70_25_0_5":  {2: 0.70, 3: 0.25, 4: 0.00, 8: 0.05},
    "pwlq_75_20_0_5":  {2: 0.75, 3: 0.20, 4: 0.00, 8: 0.05},
    "pwlq_80_15_0_5":  {2: 0.80, 3: 0.15, 4: 0.00, 8: 0.05},

    # Reduce 8-bit protection slightly
    "pwlq_70_22_5_3":  {2: 0.70, 3: 0.22, 4: 0.05, 8: 0.03},
    "pwlq_75_17_5_3":  {2: 0.75, 3: 0.17, 4: 0.05, 8: 0.03},
    "pwlq_80_12_5_3":  {2: 0.80, 3: 0.12, 4: 0.05, 8: 0.03},

    # Very extreme failure-point tests
    "pwlq_85_10_2_3":  {2: 0.85, 3: 0.10, 4: 0.02, 8: 0.03},
    "pwlq_90_7_0_3":   {2: 0.90, 3: 0.07, 4: 0.00, 8: 0.03},
    "pwlq_95_3_0_2":   {2: 0.95, 3: 0.03, 4: 0.00, 8: 0.02},
}

results_pwlq_more_aggressive = run_weight_budget_search(
    model_path=model_path,
    calib_ds=calib_ds,
    test_ds=test_ds,

    budgets=pwlq_more_aggressive_budgets,
    weight_modes=("pwlq",),

    sensitivity_bits=3,
    clip_percentiles=(0.99, 0.999, 1.0),
    analysis_quant_mode="symmetric",

    max_ba_drop_pct=5.0,

    scale_bits=0,
    break_point="norm",
    pw_opt=2,
    approximate=True,
)

# Inspect best accepted configurations
df_pwlq_more = results_pwlq_more_aggressive["df_results"]

df_pwlq_more.sort_values(
    ["accepted", "avg_weight_bits", "balanced_accuracy"],
    ascending=[False, True, False]
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def collect_quant_results(*result_objects):
    """
    Combine multiple run_weight_budget_search outputs into one dataframe.

    Usage:
        df_all = collect_quant_results(
            results,
            results_pwlq_aggressive,
            results_pwlq_more_aggressive
        )
    """
    dfs = []

    for i, res in enumerate(result_objects):
        if res is None:
            continue

        if isinstance(res, dict) and "df_results" in res:
            df = res["df_results"].copy()
            df["run_id"] = i
            dfs.append(df)

        elif isinstance(res, pd.DataFrame):
            df = res.copy()
            df["run_id"] = i
            dfs.append(df)

        else:
            print(f"Skipping object {i}: not a result dict or dataframe")

    if len(dfs) == 0:
        raise ValueError("No valid result dataframes found.")

    df_all = pd.concat(dfs, ignore_index=True)

    # Remove duplicated rows if same budget/mode was rerun
    df_all = df_all.drop_duplicates(
        subset=["budget", "weight_mode", "avg_weight_bits", "balanced_accuracy"],
        keep="last"
    )

    return df_all


def plot_quant_tradeoff(
    df_all,
    fp32_ba=None,
    max_drop_pct=5.0,
    title="Quantization trade-off",
    annotate_best=True,
):
    """
    Plot avg weight bits vs balanced accuracy.
    Lower x is more compressed.
    Higher y is better accuracy.
    """

    df = df_all.copy()

    if fp32_ba is None:
        if "ba_drop_abs" in df.columns and "balanced_accuracy" in df.columns:
            fp32_ba = float((df["balanced_accuracy"] + df["ba_drop_abs"]).median())
        else:
            fp32_ba = None

    plt.figure(figsize=(10, 6))

    for mode in sorted(df["weight_mode"].unique()):
        sub = df[df["weight_mode"] == mode]

        plt.scatter(
            sub["avg_weight_bits"],
            sub["balanced_accuracy"],
            label=mode,
            s=70,
            alpha=0.85
        )

        # Connect points sorted by compression
        sub_sorted = sub.sort_values("avg_weight_bits")
        plt.plot(
            sub_sorted["avg_weight_bits"],
            sub_sorted["balanced_accuracy"],
            alpha=0.45
        )

    if fp32_ba is not None:
        plt.axhline(
            fp32_ba,
            linestyle="--",
            linewidth=1,
            label=f"FP32 BA = {fp32_ba:.4f}"
        )

        allowed_ba = fp32_ba * (1 - max_drop_pct / 100.0)

        plt.axhline(
            allowed_ba,
            linestyle=":",
            linewidth=2,
            label=f"{max_drop_pct:.1f}% drop limit = {allowed_ba:.4f}"
        )

    accepted = df[df["accepted"] == True].copy()

    if annotate_best and len(accepted) > 0:
        best = accepted.sort_values(
            ["avg_weight_bits", "balanced_accuracy"],
            ascending=[True, False]
        ).iloc[0]

        plt.scatter(
            [best["avg_weight_bits"]],
            [best["balanced_accuracy"]],
            s=180,
            marker="*",
            label="Selected best"
        )

        plt.annotate(
            best["budget"],
            xy=(best["avg_weight_bits"], best["balanced_accuracy"]),
            xytext=(8, 8),
            textcoords="offset points",
            fontsize=9
        )

    plt.xlabel("Average weight bits")
    plt.ylabel("Balanced accuracy")
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.gca().invert_xaxis()  # left = less compressed, right = more compressed
    plt.show()


def plot_ba_drop_vs_bits(
    df_all,
    max_drop_pct=5.0,
    title="Balanced-accuracy drop vs average weight bits",
):
    df = df_all.copy()

    plt.figure(figsize=(10, 6))

    for mode in sorted(df["weight_mode"].unique()):
        sub = df[df["weight_mode"] == mode].sort_values("avg_weight_bits")

        plt.scatter(
            sub["avg_weight_bits"],
            sub["ba_drop_pct"],
            label=mode,
            s=70,
            alpha=0.85
        )

        plt.plot(
            sub["avg_weight_bits"],
            sub["ba_drop_pct"],
            alpha=0.45
        )

    plt.axhline(
        max_drop_pct,
        linestyle=":",
        linewidth=2,
        label=f"{max_drop_pct:.1f}% drop limit"
    )

    plt.xlabel("Average weight bits")
    plt.ylabel("Balanced accuracy drop (%)")
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.gca().invert_xaxis()
    plt.show()

In [ ]:
df_all = collect_quant_results(
    results,
    results_pwlq_aggressive,
    results_pwlq_more_aggressive
)

plot_quant_tradeoff(
    df_all,
    max_drop_pct=5.0,
    title="Weight quantization trade-off"
)

plot_ba_drop_vs_bits(
    df_all,
    max_drop_pct=5.0,
    title="BA degradation vs compression"
)

In [ ]:
# =========================================================
# ACTIVATION QUANTIZATION HELPERS
# =========================================================

def _iter_all_layers(model):
    for layer in model.layers:
        yield layer
        if hasattr(layer, "layers"):
            yield from _iter_all_layers(layer)


def _collect_activation_stats_for_layers(
    model,
    calib_ds,
    target_layer_names,
    act_clip_percentile=0.999,
):
    """
    Collect activation clip/scale stats per selected layer.
    One activation quantizer per layer.
    """

    act_stats = {}
    weight_layers = collect_all_weight_layers(model)

    for layer in weight_layers:
        if layer.name not in target_layer_names:
            continue

        _, outputs = capture_layer_activations(model, layer, calib_ds)

        if not outputs:
            continue

        all_acts = np.concatenate([
            (a.numpy() if hasattr(a, "numpy") else a).reshape(-1)
            for a in outputs
        ])

        signed = bool(all_acts.min() < 0)
        clip_hi = float(np.percentile(np.abs(all_acts), act_clip_percentile * 100))

        act_stats[layer.name] = {
            "signed": signed,
            "clip_hi": clip_hi,
        }

    return act_stats


def _patch_fixed_activation_bits(
    model,
    act_stats,
    act_bits,
):
    """
    Patch activation quantizers into the model.
    Uses the same bitwidth for all selected activation layers.
    """

    patched = {}

    def make_qfn(bits, signed, clip_hi):
        clip_hi_tf = tf.constant(clip_hi, dtype=tf.float32)

        if signed:
            qmax = 2 ** (bits - 1) - 1
            qmin = -qmax - 1
            scale = clip_hi / qmax + 1e-8
        else:
            qmax = 2 ** bits - 1
            qmin = 0
            scale = clip_hi / qmax + 1e-8

        scale_tf = tf.constant(scale, dtype=tf.float32)

        def qfn(x):
            if signed:
                x_c = tf.clip_by_value(x, -clip_hi_tf, clip_hi_tf)
            else:
                x_c = tf.clip_by_value(x, 0.0, clip_hi_tf)

            q = tf.round(x_c / scale_tf)
            q = tf.clip_by_value(q, float(qmin), float(qmax))
            return q * scale_tf

        return qfn, scale, qmin, qmax

    def make_patched_call(original_call, qfn):
        def patched_call(inputs, **kwargs):
            out = original_call(inputs, **kwargs)
            return qfn(out)
        return patched_call

    for layer in _iter_all_layers(model):
        if layer.name not in act_stats:
            continue

        stats = act_stats[layer.name]
        qfn, scale, qmin, qmax = make_qfn(
            bits=act_bits,
            signed=stats["signed"],
            clip_hi=stats["clip_hi"],
        )

        layer.call = make_patched_call(layer.call, qfn)

        patched[layer.name] = {
            "bits": act_bits,
            "signed": stats["signed"],
            "clip_hi": stats["clip_hi"],
            "scale": scale,
            "qmin": qmin,
            "qmax": qmax,
        }

    print(f"Activation quantizers patched: {len(patched)} layers at {act_bits}-bit")

    return patched


# =========================================================
# WEIGHT + ACTIVATION SWEEP
# =========================================================

def run_activation_sweep_on_weight_config(
    model_path,
    calib_ds,
    test_ds,
    weight_result_obj,
    budget_name,
    weight_mode="pwlq",
    activation_bits=(8, 6, 4, 3, 2),
    act_clip_percentile=0.999,
    max_final_ba_drop_pct=8.0,
    scale_bits=0,
    break_point="norm",
    pw_opt=2,
    approximate=True,
):
    """
    Freeze one selected weight configuration and sweep uniform activation bits.
    """

    key = (budget_name, weight_mode)

    if key not in weight_result_obj["saved_configs"]:
        raise KeyError(
            f"{key} not found in saved_configs. Available keys:\n"
            f"{list(weight_result_obj['saved_configs'].keys())}"
        )

    layer_records = weight_result_obj["saved_configs"][key]["layer_records"]

    fp32_ba = weight_result_obj["fp32_metrics"]["balanced_accuracy"]

    rows = {}
    rows_list = []

    # Collect activation stats once on the weight-quantized model with FP32 activations
    base_w_model = load_model(model_path)

    base_w_model = quantize_model_from_budget_records(
        model=base_w_model,
        layer_records=layer_records,
        weight_mode=weight_mode,
        scale_bits=scale_bits,
        break_point=break_point,
        pw_opt=pw_opt,
        approximate=approximate,
    )

    target_layer_names = [rec["layer_name"] for rec in layer_records]

    print("\n" + "=" * 90)
    print("COLLECTING ACTIVATION STATS ON FIXED WEIGHT-QUANTIZED MODEL")
    print("=" * 90)

    act_stats = _collect_activation_stats_for_layers(
        model=base_w_model,
        calib_ds=calib_ds,
        target_layer_names=target_layer_names,
        act_clip_percentile=act_clip_percentile,
    )

    print(f"Collected activation stats for {len(act_stats)} layers")

    # Evaluate weight-only reference again
    weight_only_metrics = compile_and_evaluate(
        base_w_model,
        test_ds,
        label=f"{budget_name} | {weight_mode} | weight-only reference",
    )

    weight_only_ba = weight_only_metrics["balanced_accuracy"]
    weight_only_drop_pct = 100.0 * (fp32_ba - weight_only_ba) / max(fp32_ba, 1e-10)

    rows_list.append({
        "budget": budget_name,
        "weight_mode": weight_mode,
        "activation_bits": "FP32",
        "balanced_accuracy": weight_only_ba,
        "ba_drop_pct": weight_only_drop_pct,
        "accepted_final": weight_only_drop_pct <= max_final_ba_drop_pct,
        "accuracy": weight_only_metrics["accuracy"],
        "macro_f1": weight_only_metrics["macro_f1"],
        "precision_macro": weight_only_metrics["precision_macro"],
        "recall_macro": weight_only_metrics["recall_macro"],
    })

    # Sweep activation bits
    for act_bits in activation_bits:
        print("\n" + "=" * 90)
        print(f"TESTING ACTIVATION QUANTIZATION: {act_bits}-bit")
        print("=" * 90)

        model_q = load_model(model_path)

        model_q = quantize_model_from_budget_records(
            model=model_q,
            layer_records=layer_records,
            weight_mode=weight_mode,
            scale_bits=scale_bits,
            break_point=break_point,
            pw_opt=pw_opt,
            approximate=approximate,
        )

        patched_stats = _patch_fixed_activation_bits(
            model=model_q,
            act_stats=act_stats,
            act_bits=int(act_bits),
        )

        metrics = compile_and_evaluate(
            model_q,
            test_ds,
            label=f"{budget_name} | {weight_mode} | act {act_bits}-bit",
        )

        ba = metrics["balanced_accuracy"]
        ba_drop_pct = 100.0 * (fp32_ba - ba) / max(fp32_ba, 1e-10)

        rows_list.append({
            "budget": budget_name,
            "weight_mode": weight_mode,
            "activation_bits": int(act_bits),
            "balanced_accuracy": ba,
            "ba_drop_pct": ba_drop_pct,
            "accepted_final": ba_drop_pct <= max_final_ba_drop_pct,
            "accuracy": metrics["accuracy"],
            "macro_f1": metrics["macro_f1"],
            "precision_macro": metrics["precision_macro"],
            "recall_macro": metrics["recall_macro"],
        })

    df_act_sweep = pd.DataFrame(rows_list)

    print("\n" + "=" * 90)
    print("FINAL ACTIVATION SWEEP SUMMARY")
    print("=" * 90)
    print(df_act_sweep.to_string(index=False))

    valid = df_act_sweep[df_act_sweep["accepted_final"]].copy()

    if len(valid) > 0:
        # prefer lowest activation bits; FP32 is treated as worst compression
        valid["_act_sort"] = valid["activation_bits"].apply(
            lambda x: 999 if x == "FP32" else int(x)
        )

        best = valid.sort_values(
            ["_act_sort", "balanced_accuracy"],
            ascending=[True, False],
        ).iloc[0].drop(labels=["_act_sort"])

        print("\nSELECTED ACTIVATION CONFIG")
        print(best.to_string())
    else:
        best = None
        print("\nNo activation setting stayed within final BA degradation limit.")

    return {
        "df_act_sweep": df_act_sweep,
        "best": best,
        "act_stats": act_stats,
        "layer_records": layer_records,
        "fp32_ba": fp32_ba,
    }

In [ ]:
activation_results = run_activation_sweep_on_weight_config(
    model_path=model_path,
    calib_ds=calib_ds,
    test_ds=test_ds,

    weight_result_obj=results_pwlq_more_aggressive,
    budget_name="pwlq_85_10_2_3",
    weight_mode="pwlq",

    activation_bits=(8, 4, 3, 2),
    act_clip_percentile=0.999,

    max_final_ba_drop_pct=8.0,

    scale_bits=0,
    break_point="norm",
    pw_opt=2,
    approximate=True,
)

In [ ]:
def summarize_activation_results(
    activation_results,
    max_final_ba_drop_pct=8.0,
    min_improvement_threshold=0.001,
):
    """
    Show activation sweep sorted by compression and choose best config.

    Strategy:
        1. Keep only accepted configs
        2. Find best BA among accepted
        3. Select lowest activation bits whose BA is within
           min_improvement_threshold of best BA

    This avoids choosing 8-bit when 4-bit performs almost identically.
    """

    df = activation_results["df_act_sweep"].copy()

    print("\n" + "=" * 100)
    print("ACTIVATION SWEEP RESULTS")
    print("=" * 100)

    print(df.to_string(index=False))

    valid = df[df["accepted_final"]].copy()

    if len(valid) == 0:
        print("\nNo valid activation configuration found.")
        return None

    valid["_bits"] = valid["activation_bits"].apply(
        lambda x: 999 if x == "FP32" else int(x)
    )

    best_ba = valid["balanced_accuracy"].max()

    near_best = valid[
        valid["balanced_accuracy"]
        >= (best_ba - min_improvement_threshold)
    ].copy()

    selected = near_best.sort_values(
        ["_bits", "balanced_accuracy"],
        ascending=[True, False]
    ).iloc[0]

    print("\n" + "=" * 100)
    print("BEST ACCEPTED BA")
    print("=" * 100)

    best_row = valid.loc[
        valid["balanced_accuracy"].idxmax()
    ]

    print(best_row.to_string())

    print("\n" + "=" * 100)
    print("SELECTED ACTIVATION CONFIG")
    print("=" * 100)

    print(selected.to_string())

    return selected

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

def plot_activation_sweep(
    activation_results,
    max_final_ba_drop_pct=8.0,
    fp32_ba=None,
    title="Activation quantization sweep",
):
    """
    Plot activation bits vs balanced accuracy / BA drop.

    Usage:
        plot_activation_sweep(activation_results)
    """

    df = activation_results["df_act_sweep"].copy()

    if fp32_ba is None:
        fp32_ba = activation_results.get("fp32_ba", None)

    if fp32_ba is None:
        fp32_ba = float(
            (df["balanced_accuracy"] + (df["ba_drop_pct"] / 100.0) * df["balanced_accuracy"]).max()
        )

    df["_act_bits_numeric"] = df["activation_bits"].apply(
        lambda x: 32 if x == "FP32" else int(x)
    )

    df = df.sort_values("_act_bits_numeric", ascending=False)

    allowed_ba = fp32_ba * (1 - max_final_ba_drop_pct / 100.0)

    fig, ax1 = plt.subplots(figsize=(10, 6))

    ax1.plot(
        df["_act_bits_numeric"],
        df["balanced_accuracy"],
        marker="o",
        linewidth=2,
        label="Balanced accuracy",
    )

    ax1.axhline(
        fp32_ba,
        linestyle="--",
        linewidth=1,
        label=f"FP32 BA = {fp32_ba:.4f}",
    )

    ax1.axhline(
        allowed_ba,
        linestyle=":",
        linewidth=2,
        label=f"{max_final_ba_drop_pct:.1f}% final drop limit = {allowed_ba:.4f}",
    )

    accepted = df[df["accepted_final"] == True]
    failed = df[df["accepted_final"] == False]

    ax1.scatter(
        accepted["_act_bits_numeric"],
        accepted["balanced_accuracy"],
        s=100,
        marker="o",
        label="Accepted",
    )

    if len(failed) > 0:
        ax1.scatter(
            failed["_act_bits_numeric"],
            failed["balanced_accuracy"],
            s=120,
            marker="x",
            label="Rejected",
        )

    best_valid = accepted.copy()

    if len(best_valid) > 0:
        best_valid["_sort_bits"] = best_valid["_act_bits_numeric"]
        best = best_valid.sort_values(
            ["_sort_bits", "balanced_accuracy"],
            ascending=[True, False],
        ).iloc[0]

        ax1.scatter(
            [best["_act_bits_numeric"]],
            [best["balanced_accuracy"]],
            s=220,
            marker="*",
            label="Selected lowest accepted bits",
        )

        ax1.annotate(
            f"{best['activation_bits']}-bit",
            xy=(best["_act_bits_numeric"], best["balanced_accuracy"]),
            xytext=(8, 8),
            textcoords="offset points",
            fontsize=10,
        )

    ax1.set_xlabel("Activation bits")
    ax1.set_ylabel("Balanced accuracy")
    ax1.set_title(title)
    ax1.grid(True, alpha=0.3)

    # Make x ticks readable
    tick_values = df["_act_bits_numeric"].tolist()
    tick_labels = [
        "FP32" if x == 32 else str(int(x))
        for x in tick_values
    ]

    ax1.set_xticks(tick_values)
    ax1.set_xticklabels(tick_labels)

    ax1.invert_xaxis()
    ax1.legend()
    plt.show()


def plot_activation_drop_sweep(
    activation_results,
    max_final_ba_drop_pct=8.0,
    title="Activation quantization BA drop",
):
    """
    Plot activation bits vs total BA drop percentage.
    """

    df = activation_results["df_act_sweep"].copy()

    df["_act_bits_numeric"] = df["activation_bits"].apply(
        lambda x: 32 if x == "FP32" else int(x)
    )

    df = df.sort_values("_act_bits_numeric", ascending=False)

    plt.figure(figsize=(10, 6))

    plt.plot(
        df["_act_bits_numeric"],
        df["ba_drop_pct"],
        marker="o",
        linewidth=2,
        label="Total BA drop",
    )

    plt.axhline(
        max_final_ba_drop_pct,
        linestyle=":",
        linewidth=2,
        label=f"{max_final_ba_drop_pct:.1f}% final drop limit",
    )

    accepted = df[df["accepted_final"] == True]
    failed = df[df["accepted_final"] == False]

    plt.scatter(
        accepted["_act_bits_numeric"],
        accepted["ba_drop_pct"],
        s=100,
        marker="o",
        label="Accepted",
    )

    if len(failed) > 0:
        plt.scatter(
            failed["_act_bits_numeric"],
            failed["ba_drop_pct"],
            s=120,
            marker="x",
            label="Rejected",
        )

    tick_values = df["_act_bits_numeric"].tolist()
    tick_labels = [
        "FP32" if x == 32 else str(int(x))
        for x in tick_values
    ]

    plt.xticks(tick_values, tick_labels)
    plt.gca().invert_xaxis()

    plt.xlabel("Activation bits")
    plt.ylabel("Total BA drop (%)")
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

In [ ]:
plot_activation_sweep(
    activation_results,
    max_final_ba_drop_pct=8.0,
    title="PWLQ weights + activation quantization"
)

plot_activation_drop_sweep(
    activation_results,
    max_final_ba_drop_pct=8.0,
    title="Final BA drop after activation quantization"
)

In [ ]:
best_act = summarize_activation_results(
    activation_results,
    max_final_ba_drop_pct=8.0,
    min_improvement_threshold=0.001
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import load_model
from scipy.io import wavfile


def preprocess_single_wav_for_model(wav_path):
    sr, sample = wavfile.read(wav_path)
    signal = sample.astype(np.float32)

    if len(signal) < audio_length:
        signal = np.pad(signal, (0, audio_length - len(signal)))
    else:
        signal = signal[:audio_length]

    mfcc_feat  = normalize(MFCC(signal)).astype(np.float32)
    mslfb_feat = normalize(MSLFB(signal)).astype(np.float32)

    mfcc_feat  = np.expand_dims(mfcc_feat, axis=(0, -1))
    mslfb_feat = np.expand_dims(mslfb_feat, axis=(0, -1))

    return [mfcc_feat, mslfb_feat]


def _capture_outputs_for_layers(model, layer_names, x):
    captured = {}
    original_calls = {}

    for layer in _iter_all_layers(model):
        if layer.name in layer_names:
            original_calls[layer.name] = layer.call

            def make_hook(layer_name, original_call):
                def hooked_call(inputs, *args, **kwargs):
                    out = original_call(inputs, *args, **kwargs)
                    captured[layer_name] = out
                    return out
                return hooked_call

            layer.call = make_hook(layer.name, layer.call)

    _ = model(x, training=False)

    for layer in _iter_all_layers(model):
        if layer.name in original_calls:
            layer.call = original_calls[layer.name]

    acts = {}
    for name, act in captured.items():
        arr = act.numpy() if hasattr(act, "numpy") else act
        acts[name] = arr.reshape(-1)

    return acts


def build_weight_only_and_weight_act_models(
    model_path,
    activation_results,
    act_bits=4,
    weight_mode="pwlq",
    scale_bits=0,
    break_point="norm",
    pw_opt=2,
    approximate=True,
):
    layer_records = activation_results["layer_records"]
    act_stats = activation_results["act_stats"]

    model_weight_only = load_model(model_path)

    model_weight_only = quantize_model_from_budget_records(
        model=model_weight_only,
        layer_records=layer_records,
        weight_mode=weight_mode,
        scale_bits=scale_bits,
        break_point=break_point,
        pw_opt=pw_opt,
        approximate=approximate,
    )

    model_weight_act = load_model(model_path)

    model_weight_act = quantize_model_from_budget_records(
        model=model_weight_act,
        layer_records=layer_records,
        weight_mode=weight_mode,
        scale_bits=scale_bits,
        break_point=break_point,
        pw_opt=pw_opt,
        approximate=approximate,
    )

    _patch_fixed_activation_bits(
        model=model_weight_act,
        act_stats=act_stats,
        act_bits=act_bits,
    )

    return model_weight_only, model_weight_act


def compare_sound_prediction_and_global_activations(
    wav_path,
    model_path,
    activation_results,
    act_bits=4,
    class_names=None,
    top_k=5,
    weight_mode="pwlq",
    bins=200,
    scatter_samples=20000,
):
    """
    Compares one sound using:
      1. PWLQ weight-only model with FP32 activations
      2. PWLQ weight + activation-quantized model

    It plots:
      - prediction probabilities
      - one global activation distribution plot with all layers combined
      - one global FP32-vs-quantized activation scatter plot
    """

    if class_names is None:
        class_names = list(le.classes_)

    x = preprocess_single_wav_for_model(wav_path)

    model_weight_only, model_weight_act = build_weight_only_and_weight_act_models(
        model_path=model_path,
        activation_results=activation_results,
        act_bits=act_bits,
        weight_mode=weight_mode,
    )

    # =====================================================
    # Prediction comparison
    # =====================================================
    p_fp32act = model_weight_only(x, training=False).numpy()[0]
    p_qact    = model_weight_act(x, training=False).numpy()[0]

    pred_fp32act = int(np.argmax(p_fp32act))
    pred_qact    = int(np.argmax(p_qact))

    print("\n" + "=" * 80)
    print("SINGLE SOUND PREDICTION COMPARISON")
    print("=" * 80)
    print(f"File: {wav_path}")
    print(f"Weight-only / FP32 activations: {class_names[pred_fp32act]} ({p_fp32act[pred_fp32act]:.4f})")
    print(f"Weight + {act_bits}b activations:     {class_names[pred_qact]} ({p_qact[pred_qact]:.4f})")
    print(f"Same prediction? {pred_fp32act == pred_qact}")

    prob_l1 = float(np.sum(np.abs(p_fp32act - p_qact)))
    print(f"Probability L1 difference: {prob_l1:.6f}")

    top_idx = np.argsort(np.maximum(p_fp32act, p_qact))[-top_k:][::-1]
    labels = [class_names[i] for i in top_idx]

    x_pos = np.arange(len(labels))
    width = 0.35

    plt.figure(figsize=(10, 5))
    plt.bar(x_pos - width / 2, p_fp32act[top_idx], width, label="Weights only / FP32 acts")
    plt.bar(x_pos + width / 2, p_qact[top_idx], width, label=f"Weights + {act_bits}b acts")
    plt.xticks(x_pos, labels, rotation=30)
    plt.ylabel("Predicted probability")
    plt.title("Prediction comparison")
    plt.grid(True, axis="y", alpha=0.3)
    plt.legend()
    plt.show()

    # =====================================================
    # Activation capture
    # =====================================================
    layer_names = list(activation_results["act_stats"].keys())

    acts_fp32 = _capture_outputs_for_layers(
        model_weight_only,
        layer_names,
        x,
    )

    acts_quant = _capture_outputs_for_layers(
        model_weight_act,
        layer_names,
        x,
    )

    all_fp32 = np.concatenate([
        acts_fp32[name].reshape(-1)
        for name in layer_names
        if name in acts_fp32 and name in acts_quant
    ])

    all_quant = np.concatenate([
        acts_quant[name].reshape(-1)
        for name in layer_names
        if name in acts_fp32 and name in acts_quant
    ])

    min_len = min(len(all_fp32), len(all_quant))
    all_fp32 = all_fp32[:min_len]
    all_quant = all_quant[:min_len]

    global_mse = float(np.mean((all_fp32 - all_quant) ** 2))
    global_rel_mse = float(global_mse / (np.mean(all_fp32 ** 2) + 1e-10))
    global_l1 = float(np.mean(np.abs(all_fp32 - all_quant)))

    print("\n" + "=" * 80)
    print("GLOBAL ACTIVATION COMPARISON")
    print("=" * 80)
    print(f"Layers included: {len(layer_names)}")
    print(f"Total activation values: {len(all_fp32)}")
    print(f"Global activation MSE:    {global_mse:.8f}")
    print(f"Global activation RelMSE: {global_rel_mse:.8f}")
    print(f"Global mean abs diff:     {global_l1:.8f}")

    # =====================================================
    # One global activation distribution plot
    # =====================================================
    plt.figure(figsize=(11, 5))

    plt.hist(
        all_fp32,
        bins=bins,
        density=True,
        alpha=0.55,
        label="Weights only / FP32 activations",
    )

    plt.hist(
        all_quant,
        bins=bins,
        density=True,
        alpha=0.55,
        label=f"Weights + {act_bits}b activations",
    )

    plt.title(f"Global activation distribution | RelMSE={global_rel_mse:.6f}")
    plt.xlabel("Activation value")
    plt.ylabel("Density")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

    # =====================================================
    # One global scatter plot
    # =====================================================
    if len(all_fp32) > scatter_samples:
        idx = np.random.choice(len(all_fp32), scatter_samples, replace=False)
        x_scatter = all_fp32[idx]
        y_scatter = all_quant[idx]
    else:
        x_scatter = all_fp32
        y_scatter = all_quant

    plt.figure(figsize=(7, 7))

    plt.scatter(
        x_scatter,
        y_scatter,
        s=3,
        alpha=0.2,
    )

    lo = min(float(np.min(x_scatter)), float(np.min(y_scatter)))
    hi = max(float(np.max(x_scatter)), float(np.max(y_scatter)))

    plt.plot(
        [lo, hi],
        [lo, hi],
        linestyle="--",
        linewidth=1,
        label="ideal: y = x",
    )

    plt.title("Global FP32 activations vs quantized activations")
    plt.xlabel("Weights only / FP32 activations")
    plt.ylabel(f"Weights + {act_bits}b activations")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

    return {
        "wav_path": wav_path,
        "act_bits": act_bits,
        "same_prediction": bool(pred_fp32act == pred_qact),
        "fp32act_pred": class_names[pred_fp32act],
        "qact_pred": class_names[pred_qact],
        "fp32act_probs": p_fp32act,
        "qact_probs": p_qact,
        "prob_l1_difference": prob_l1,
        "global_activation_mse": global_mse,
        "global_activation_rel_mse": global_rel_mse,
        "global_activation_mean_abs_diff": global_l1,
        "all_fp32_activations": all_fp32,
        "all_quant_activations": all_quant,
        "acts_fp32_by_layer": acts_fp32,
        "acts_quant_by_layer": acts_quant,
    }

In [ ]:
comparison = compare_sound_prediction_and_global_activations(
    wav_path="/content/dataset/go/00b01445_nohash_0.wav",
    model_path=model_path,
    activation_results=activation_results,
    act_bits=4,
    class_names=list(le.classes_),
    top_k=5,
    weight_mode="pwlq",
    bins=200,
    scatter_samples=20000,
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import load_model

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)


def _collect_preds_labels_probs(model, test_ds):
    all_probs = []
    all_preds = []
    all_labels = []

    for x_batch, y_batch in test_ds:
        probs = model(x_batch, training=False).numpy()
        preds = np.argmax(probs, axis=1)
        labels = np.argmax(y_batch.numpy(), axis=1)

        all_probs.append(probs)
        all_preds.append(preds)
        all_labels.append(labels)

    all_probs = np.concatenate(all_probs)
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    return all_labels, all_preds, all_probs


def _compute_full_classification_metrics(y_true, y_pred, class_names=None):
    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    precision_macro = precision_score(y_true, y_pred, average="macro", zero_division=0)
    recall_macro = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)

    cm = confusion_matrix(y_true, y_pred)

    report = classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        zero_division=0,
        output_dict=True,
    )

    return {
        "accuracy": float(acc),
        "balanced_accuracy": float(bal_acc),
        "precision_macro": float(precision_macro),
        "recall_macro": float(recall_macro),
        "macro_f1": float(f1_macro),
        "confusion_matrix": cm,
        "classification_report": report,
    }


def _plot_confusion_matrix(
    cm,
    class_names,
    title="Confusion matrix",
    normalize=False,
    figsize=(9, 8),
    cmap="Blues",          # <-- Much higher contrast
):
    if normalize:
        cm_plot = cm.astype(np.float32)
        row_sums = cm_plot.sum(axis=1, keepdims=True) + 1e-10
        cm_plot = cm_plot / row_sums
        fmt = ".2f"
    else:
        cm_plot = cm.astype(np.int64)
        fmt = "d"

    plt.figure(figsize=figsize)

    im = plt.imshow(
        cm_plot,
        interpolation="nearest",
        cmap=cmap,
    )

    plt.title(title, fontsize=16, fontweight="bold")
    plt.colorbar(im, fraction=0.046, pad=0.04)

    tick_marks = np.arange(len(class_names))
    plt.xticks(
        tick_marks,
        class_names,
        rotation=45,
        ha="right",
        fontsize=11,
    )
    plt.yticks(
        tick_marks,
        class_names,
        fontsize=11,
    )

    thresh = cm_plot.max() * 0.5

    for i in range(cm_plot.shape[0]):
        for j in range(cm_plot.shape[1]):
            value = cm_plot[i, j]

            if normalize:
                text = f"{value:.2f}"
            else:
                text = f"{int(value)}"

            plt.text(
                j,
                i,
                text,
                ha="center",
                va="center",
                fontsize=9,
                fontweight="bold",
                color="white" if value > thresh else "black",
            )

    plt.ylabel("True label", fontsize=13)
    plt.xlabel("Predicted label", fontsize=13)

    plt.tight_layout()
    plt.show()


def build_best_quantized_model_from_activation_results(
    model_path,
    activation_results,
    act_bits=None,
    weight_mode="pwlq",
    scale_bits=0,
    break_point="norm",
    pw_opt=2,
    approximate=True,
):
    """
    Builds final model:
      selected mixed-precision weight quantization
      +
      activation quantization.

    If act_bits is None, it uses activation_results["best"].
    """

    if act_bits is None:
        best = activation_results.get("best", None)

        if best is None:
            raise ValueError("act_bits was not provided and activation_results['best'] is None.")

        act_bits = best["activation_bits"]

        if act_bits == "FP32":
            act_bits = None

    layer_records = activation_results["layer_records"]
    act_stats = activation_results["act_stats"]

    model_q = load_model(model_path)

    model_q = quantize_model_from_budget_records(
        model=model_q,
        layer_records=layer_records,
        weight_mode=weight_mode,
        scale_bits=scale_bits,
        break_point=break_point,
        pw_opt=pw_opt,
        approximate=approximate,
    )

    if act_bits is not None:
        _patch_fixed_activation_bits(
            model=model_q,
            act_stats=act_stats,
            act_bits=int(act_bits),
        )

    return model_q


def evaluate_fp32_vs_best_quantized_model(
    model_path,
    test_ds,
    activation_results,
    act_bits=4,
    class_names=None,
    weight_mode="pwlq",
    normalize_cm=True,
    scale_bits=0,
    break_point="norm",
    pw_opt=2,
    approximate=True,
):
    """
    Evaluates and compares:
      1. FP32 model
      2. Weight-only quantized model
      3. Weight + activation quantized model

    Returns all metrics and plots confusion matrices.
    """

    if class_names is None:
        class_names = list(le.classes_)

    # =====================================================
    # Build models
    # =====================================================
    print("\nBuilding FP32 model...")
    model_fp32 = load_model(model_path)

    print("\nBuilding weight-only quantized model...")
    model_weight_only = load_model(model_path)

    model_weight_only = quantize_model_from_budget_records(
        model=model_weight_only,
        layer_records=activation_results["layer_records"],
        weight_mode=weight_mode,
        scale_bits=scale_bits,
        break_point=break_point,
        pw_opt=pw_opt,
        approximate=approximate,
    )

    print("\nBuilding final weight + activation quantized model...")
    model_final = build_best_quantized_model_from_activation_results(
        model_path=model_path,
        activation_results=activation_results,
        act_bits=act_bits,
        weight_mode=weight_mode,
        scale_bits=scale_bits,
        break_point=break_point,
        pw_opt=pw_opt,
        approximate=approximate,
    )

    models = {
        "FP32": model_fp32,
        "Weight-only": model_weight_only,
        f"Weight + {act_bits}b activations": model_final,
    }

    results = {}
    summary_rows = []

    # =====================================================
    # Evaluate all models
    # =====================================================
    for name, model in models.items():
        print("\n" + "=" * 90)
        print(f"EVALUATING: {name}")
        print("=" * 90)

        y_true, y_pred, y_prob = _collect_preds_labels_probs(model, test_ds)

        metrics = _compute_full_classification_metrics(
            y_true,
            y_pred,
            class_names=class_names,
        )

        results[name] = {
            "y_true": y_true,
            "y_pred": y_pred,
            "y_prob": y_prob,
            **metrics,
        }

        summary_rows.append({
            "model": name,
            "accuracy": metrics["accuracy"],
            "balanced_accuracy": metrics["balanced_accuracy"],
            "precision_macro": metrics["precision_macro"],
            "recall_macro": metrics["recall_macro"],
            "macro_f1": metrics["macro_f1"],
        })

        print(f"Accuracy:          {metrics['accuracy']:.4f}")
        print(f"Balanced Accuracy: {metrics['balanced_accuracy']:.4f}")
        print(f"Precision macro:   {metrics['precision_macro']:.4f}")
        print(f"Recall macro:      {metrics['recall_macro']:.4f}")
        print(f"Macro F1:          {metrics['macro_f1']:.4f}")

        _plot_confusion_matrix(
            metrics["confusion_matrix"],
            class_names=class_names,
            title=f"{name} confusion matrix",
            normalize=False,
        )

        if normalize_cm:
            _plot_confusion_matrix(
                metrics["confusion_matrix"],
                class_names=class_names,
                title=f"{name} normalized confusion matrix",
                normalize=True,
            )

    df_summary = pd.DataFrame(summary_rows)

    # =====================================================
    # Drop relative to FP32
    # =====================================================
    fp32_ba = df_summary.loc[df_summary["model"] == "FP32", "balanced_accuracy"].iloc[0]

    df_summary["ba_drop_abs_vs_fp32"] = fp32_ba - df_summary["balanced_accuracy"]
    df_summary["ba_drop_pct_vs_fp32"] = (
        100.0 * df_summary["ba_drop_abs_vs_fp32"] / max(fp32_ba, 1e-10)
    )

    print("\n" + "=" * 90)
    print("SUMMARY")
    print("=" * 90)
    print(df_summary.to_string(index=False))

    return {
        "df_summary": df_summary,
        "results": results,
        "models": models,
    }

In [ ]:
final_eval = evaluate_fp32_vs_best_quantized_model(
    model_path=model_path,
    test_ds=test_ds,
    activation_results=activation_results,
    act_bits=4,
    class_names=list(le.classes_),
    weight_mode="pwlq",
    normalize_cm=True,
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model


def plot_fp32_vs_final_quantized_weights(
    model_path,
    activation_results,
    weight_mode="pwlq",
    scale_bits=0,
    break_point="norm",
    pw_opt=2,
    approximate=True,
    bins=250,
    density=True,
    use_log_y=True,
    sample_per_model=None,
):
    """
    Plot overlapping weight distributions:
      - FP32 model weights
      - Final quantized model weights

    This does not evaluate the model. It only loads FP32, builds the final
    weight-quantized version, extracts weights, and plots distributions.
    """

    # -----------------------------
    # Build FP32 model
    # -----------------------------
    model_fp32 = load_model(model_path)

    # -----------------------------
    # Build final quantized model
    # -----------------------------
    model_q = load_model(model_path)

    model_q = quantize_model_from_budget_records(
        model=model_q,
        layer_records=activation_results["layer_records"],
        weight_mode=weight_mode,
        scale_bits=scale_bits,
        break_point=break_point,
        pw_opt=pw_opt,
        approximate=approximate,
    )

    # -----------------------------
    # Collect all weights
    # -----------------------------
    fp32_weights = []
    q_weights = []

    for layer_fp32, layer_q in zip(
        collect_all_weight_layers(model_fp32),
        collect_all_weight_layers(model_q),
    ):
        w_fp32 = layer_fp32.get_weights()
        w_q = layer_q.get_weights()

        if not w_fp32 or not w_q:
            continue

        fp32_weights.append(w_fp32[0].reshape(-1))
        q_weights.append(w_q[0].reshape(-1))

    fp32_weights = np.concatenate(fp32_weights)
    q_weights = np.concatenate(q_weights)

    # Optional sampling for faster plotting
    if sample_per_model is not None:
        if len(fp32_weights) > sample_per_model:
            idx = np.random.choice(len(fp32_weights), sample_per_model, replace=False)
            fp32_weights = fp32_weights[idx]

        if len(q_weights) > sample_per_model:
            idx = np.random.choice(len(q_weights), sample_per_model, replace=False)
            q_weights = q_weights[idx]

    # -----------------------------
    # Print quick stats
    # -----------------------------
    print("\n" + "=" * 80)
    print("WEIGHT DISTRIBUTION COMPARISON")
    print("=" * 80)
    print(f"FP32 weights:      {len(fp32_weights):,}")
    print(f"Quantized weights: {len(q_weights):,}")
    print(f"FP32 mean/std:     {np.mean(fp32_weights):.6f} / {np.std(fp32_weights):.6f}")
    print(f"Quant mean/std:    {np.mean(q_weights):.6f} / {np.std(q_weights):.6f}")
    print(f"FP32 min/max:      {np.min(fp32_weights):.6f} / {np.max(fp32_weights):.6f}")
    print(f"Quant min/max:     {np.min(q_weights):.6f} / {np.max(q_weights):.6f}")
    print(f"Unique quantized values: {len(np.unique(q_weights)):,}")

    # -----------------------------
    # Plot overlapping distributions
    # -----------------------------
    plt.figure(figsize=(11, 5))

    plt.hist(
        fp32_weights,
        bins=bins,
        density=density,
        alpha=0.55,
        label="FP32 weights",
    )

    plt.hist(
        q_weights,
        bins=bins,
        density=density,
        alpha=0.55,
        label=f"Final quantized weights ({weight_mode})",
    )

    plt.title("FP32 vs final quantized weight distribution")
    plt.xlabel("Weight value")
    plt.ylabel("Density" if density else "Count")
    plt.grid(True, alpha=0.3)
    plt.legend()

    if use_log_y:
        plt.yscale("log")

    plt.show()

    return {
        "fp32_weights": fp32_weights,
        "quantized_weights": q_weights,
    }

In [ ]:
weight_dist = plot_fp32_vs_final_quantized_weights(
    model_path=model_path,
    activation_results=activation_results,
    weight_mode="pwlq",
    bins=250,
    use_log_y=True,
    sample_per_model=None,
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model


def _get_all_weight_arrays(model):
    weights = []
    layer_info = []

    for layer in collect_all_weight_layers(model):
        w = layer.get_weights()
        if not w:
            continue

        W = w[0].astype(np.float32)
        weights.append(W.reshape(-1))

        layer_info.append({
            "layer": layer.name,
            "weights": W,
            "flat": W.reshape(-1),
        })

    return np.concatenate(weights), layer_info


def _clip_weights_array(W, clip_percentile):
    clip_hi = np.percentile(np.abs(W), clip_percentile * 100)
    W_clip = np.clip(W, -clip_hi, clip_hi)

    clipped_mask = np.abs(W) > clip_hi
    n_clipped = int(np.sum(clipped_mask))

    mse = float(np.mean((W - W_clip) ** 2))
    rel_mse = float(mse / (np.mean(W ** 2) + 1e-10))

    return W_clip, clip_hi, n_clipped, mse, rel_mse


def analyze_global_weight_clipping(
    model_path,
    clip_percentiles=(1.0, 0.999, 0.99, 0.95),
    bins=250,
    density=True,
    log_y=True,
):
    model = load_model(model_path)
    fp32_weights, layer_info = _get_all_weight_arrays(model)

    rows = []
    clipped_cache = {}

    print("\n" + "=" * 90)
    print("GLOBAL WEIGHT CLIPPING ANALYSIS")
    print("=" * 90)
    print(f"Total weights: {len(fp32_weights):,}")

    for p in clip_percentiles:
        clipped_weights, clip_hi, n_clipped, mse, rel_mse = _clip_weights_array(
            fp32_weights,
            p,
        )

        clipped_pct = 100.0 * n_clipped / len(fp32_weights)

        rows.append({
            "clip_percentile": p,
            "clip_hi": clip_hi,
            "n_clipped": n_clipped,
            "pct_clipped": clipped_pct,
            "mse": mse,
            "rel_mse": rel_mse,
        })

        clipped_cache[p] = clipped_weights

    df_clip = pd.DataFrame(rows)

    best_idx = df_clip["rel_mse"].idxmin()
    best_row = df_clip.loc[best_idx]
    best_p = float(best_row["clip_percentile"])

    print("\n" + "=" * 90)
    print("CLIPPING SUMMARY")
    print("=" * 90)
    print(df_clip.to_string(index=False))

    print("\n" + "=" * 90)
    print("SELECTED GLOBAL CLIPPING STRATEGY")
    print("=" * 90)
    print(f"Chosen clip percentile: {best_p}")
    print(f"Clip threshold:         {best_row['clip_hi']:.8f}")
    print(f"Weights clipped:        {int(best_row['n_clipped']):,} / {len(fp32_weights):,} "
          f"({best_row['pct_clipped']:.4f}%)")
    print(f"MSE:                    {best_row['mse']:.8e}")
    print(f"RelMSE:                 {best_row['rel_mse']:.8e}")
    print("Reason: lowest global weight RelMSE among tested clipping candidates.")

    for _, row in df_clip.iterrows():
        p = float(row["clip_percentile"])
        clipped_weights = clipped_cache[p]

        selected = p == best_p
        title_prefix = "SELECTED | " if selected else ""

        plt.figure(figsize=(11, 5))

        plt.hist(
            fp32_weights,
            bins=bins,
            density=density,
            alpha=0.55,
            label="FP32 original weights",
        )

        plt.hist(
            clipped_weights,
            bins=bins,
            density=density,
            alpha=0.55,
            label=f"Clipped weights p={p}",
        )

        plt.title(
            f"{title_prefix}FP32 vs clipped weights | p={p} | "
            f"clipped={int(row['n_clipped']):,} ({row['pct_clipped']:.4f}%) | "
            f"RelMSE={row['rel_mse']:.2e}"
        )

        plt.xlabel("Weight value")
        plt.ylabel("Density" if density else "Count")
        plt.grid(True, alpha=0.3)
        plt.legend()

        if log_y:
            plt.yscale("log")

        plt.show()

    return {
        "df_clip": df_clip,
        "best_clip_percentile": best_p,
        "best_clip_row": best_row,
        "fp32_weights": fp32_weights,
        "best_clipped_weights": clipped_cache[best_p],
    }

In [ ]:
clip_analysis = analyze_global_weight_clipping(
    model_path=model_path,
    clip_percentiles = (
    0.9999,
    0.999,
    0.9975,
    0.995,
    0.9925,
    0.99,
    0.985,
    0.98,
    0.975,
    0.97,
    0.965,
    0.96,
    0.95,
),
    bins=250,
    log_y=True,
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# =========================================================
# 1. EXTRACT BIT ALLOCATION TABLE
# =========================================================

def make_bit_allocation_dataframe(layer_records):
    """
    Creates a dataframe with bit allocation per layer.

    Expected layer_records format:
        rec["layer_name"]
        rec["assigned_bits"]
    """

    rows = []

    for rec in layer_records:
        layer_name = rec["layer_name"]
        bits = np.array(rec["assigned_bits"]).astype(int)

        counts = {b: int(np.sum(bits == b)) for b in sorted(np.unique(bits))}

        row = {
            "layer": layer_name,
            "n_channels": len(bits),
            "avg_bits": float(np.mean(bits)),
        }

        for b in [2, 3, 4, 8]:
            row[f"{b}b"] = counts.get(b, 0)

        rows.append(row)

    df = pd.DataFrame(rows)

    return df


# =========================================================
# 2. PLOT BIT ALLOCATION PER LAYER
# =========================================================

def plot_bit_allocation_per_layer(
    layer_records,
    figsize=(12, 6),
):
    """
    Stacked bar plot:
      x-axis = layer
      y-axis = number of channels
      stacked by assigned bit-width
    """

    df = make_bit_allocation_dataframe(layer_records)

    x = np.arange(len(df))
    bottom = np.zeros(len(df))

    plt.figure(figsize=figsize)

    for b in [2, 3, 4, 8]:
        values = df[f"{b}b"].values
        plt.bar(
            x,
            values,
            bottom=bottom,
            label=f"{b}-bit",
        )
        bottom += values

    plt.xticks(x, df["layer"], rotation=45, ha="right")
    plt.ylabel("Number of output channels")
    plt.title("Bit allocation per layer")
    plt.grid(True, axis="y", alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    print("\nBIT ALLOCATION PER LAYER")
    print(df.to_string(index=False))

    return df


# =========================================================
# 3. PLOT GLOBAL BIT ALLOCATION
# =========================================================

def plot_global_bit_allocation(
    layer_records,
    figsize=(7, 5),
):
    """
    Bar plot of total number of channels assigned to each bit-width.
    """

    df = make_bit_allocation_dataframe(layer_records)

    totals = {
        "2-bit": int(df["2b"].sum()),
        "3-bit": int(df["3b"].sum()),
        "4-bit": int(df["4b"].sum()),
        "8-bit": int(df["8b"].sum()),
    }

    plt.figure(figsize=figsize)
    plt.bar(list(totals.keys()), list(totals.values()))
    plt.ylabel("Number of output channels")
    plt.title("Global bit allocation")
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

    print("\nGLOBAL BIT ALLOCATION")
    for k, v in totals.items():
        print(f"{k}: {v}")

    return totals


# =========================================================
# 4. COMPRESSION / MEMORY SAVING
# =========================================================

def compute_weight_compression_summary(
    model,
    layer_records,
    fp32_bits=32,
):
    """
    Computes memory saving for the mixed-precision weight configuration.

    FP32 memory:
        total number of weights * 32 bits

    Quantized memory:
        sum(number of weights in channel * assigned channel bits)

    Biases are ignored by default because they are usually kept in FP32
    and are much smaller than the kernels.
    """

    name_to_rec = {
        rec["layer_name"]: rec
        for rec in layer_records
    }

    rows = []

    total_fp32_bits = 0
    total_quant_bits = 0
    total_weights = 0

    for layer in collect_all_weight_layers(model):
        if layer.name not in name_to_rec:
            continue

        weights = layer.get_weights()
        if not weights:
            continue

        W = weights[0]
        rec = name_to_rec[layer.name]
        assigned_bits = np.array(rec["assigned_bits"]).astype(int)

        out_ch = W.shape[-1]

        if len(assigned_bits) != out_ch:
            print(
                f"Warning: {layer.name} has {out_ch} output channels "
                f"but {len(assigned_bits)} assigned bits"
            )
            continue

        weights_per_channel = int(np.prod(W.shape[:-1]))

        layer_weights = int(np.prod(W.shape))
        layer_fp32_bits = layer_weights * fp32_bits

        layer_quant_bits = int(
            np.sum(assigned_bits * weights_per_channel)
        )

        avg_bits = float(np.mean(assigned_bits))

        compression_ratio = layer_fp32_bits / max(layer_quant_bits, 1)
        memory_reduction_pct = 100.0 * (1.0 - layer_quant_bits / layer_fp32_bits)

        rows.append({
            "layer": layer.name,
            "n_weights": layer_weights,
            "n_channels": out_ch,
            "weights_per_channel": weights_per_channel,
            "avg_bits": avg_bits,
            "fp32_bits": layer_fp32_bits,
            "quant_bits": layer_quant_bits,
            "compression_ratio": compression_ratio,
            "memory_reduction_pct": memory_reduction_pct,
        })

        total_weights += layer_weights
        total_fp32_bits += layer_fp32_bits
        total_quant_bits += layer_quant_bits

    df = pd.DataFrame(rows)

    global_compression_ratio = total_fp32_bits / max(total_quant_bits, 1)
    global_memory_reduction_pct = 100.0 * (1.0 - total_quant_bits / total_fp32_bits)
    global_avg_bits = total_quant_bits / max(total_weights, 1)

    summary = {
        "total_weights": total_weights,
        "fp32_bits": total_fp32_bits,
        "quant_bits": total_quant_bits,
        "fp32_MB": total_fp32_bits / 8 / 1024**2,
        "quant_MB": total_quant_bits / 8 / 1024**2,
        "global_avg_bits": global_avg_bits,
        "compression_ratio": global_compression_ratio,
        "memory_reduction_pct": global_memory_reduction_pct,
    }

    print("\n" + "=" * 80)
    print("WEIGHT COMPRESSION SUMMARY")
    print("=" * 80)
    print(f"Total weights:          {summary['total_weights']:,}")
    print(f"FP32 size:              {summary['fp32_MB']:.4f} MB")
    print(f"Quantized size:         {summary['quant_MB']:.4f} MB")
    print(f"Global average bits:    {summary['global_avg_bits']:.4f}")
    print(f"Compression ratio:      {summary['compression_ratio']:.2f}x")
    print(f"Memory reduction:       {summary['memory_reduction_pct']:.2f}%")

    return df, summary


def plot_compression_per_layer(
    compression_df,
    figsize=(12, 5),
):
    """
    Plots per-layer compression ratio.
    """

    df = compression_df.copy()

    plt.figure(figsize=figsize)
    plt.bar(df["layer"], df["compression_ratio"])
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Compression ratio vs FP32")
    plt.title("Per-layer weight compression ratio")
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
# Use the final/best layer_records
layer_records = activation_results["layer_records"]

# 1. Bit allocation plots
df_bits = plot_bit_allocation_per_layer(layer_records)
global_bits = plot_global_bit_allocation(layer_records)

# 2. Compression summary
fp32_model = load_model(model_path)

compression_df, compression_summary = compute_weight_compression_summary(
    model=fp32_model,
    layer_records=layer_records,
)

plot_compression_per_layer(compression_df)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def merge_weight_search_results(*result_objs, drop_duplicates=True):
    """
    Merge multiple weight-search result objects into one dataframe.

    Example:
        df_all = merge_weight_search_results(
            results,
            results_pwlq_aggressive,
            results_pwlq_more_aggressive,
        )
    """

    dfs = []

    for obj in result_objs:
        if obj is None:
            continue

        if isinstance(obj, pd.DataFrame):
            df = obj.copy()
        elif isinstance(obj, dict) and "df_results" in obj:
            df = obj["df_results"].copy()
        else:
            raise ValueError(
                "Each object must be either a dataframe or a dict with key 'df_results'."
            )

        dfs.append(df)

    if len(dfs) == 0:
        raise ValueError("No valid result objects were provided.")

    df_all = pd.concat(dfs, ignore_index=True)

    if drop_duplicates:
        df_all = (
            df_all
            .drop_duplicates(subset=["budget", "weight_mode"])
            .reset_index(drop=True)
        )

    return df_all


def plot_all_weight_results_tradeoff(
    *result_objs,
    accepted_only=False,
    annotate_best=True,
    annotate_all=False,
    figsize=(18, 10),
):
    """
    Merge and plot all weight budget results.

    Use:
        plot_all_weight_results_tradeoff(
            results,
            results_pwlq_aggressive,
            results_pwlq_more_aggressive,
        )
    """

    df = merge_weight_search_results(*result_objs)

    required_cols = [
        "budget",
        "weight_mode",
        "avg_weight_bits",
        "balanced_accuracy",
    ]

    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}\nAvailable: {list(df.columns)}")

    if accepted_only and "accepted" in df.columns:
        df_plot = df[df["accepted"] == True].copy()
    else:
        df_plot = df.copy()

    markers = {
        "uniform_symmetric": "o",
        "uniform_adaptive": "s",
        "pwlq": "^",
    }

    plt.figure(figsize=figsize)

    for mode in sorted(df_plot["weight_mode"].unique()):
        sub = df_plot[df_plot["weight_mode"] == mode].copy()
        sub = sub.sort_values("avg_weight_bits")

        plt.plot(
            sub["avg_weight_bits"],
            sub["balanced_accuracy"],
            linewidth=2.5,
            alpha=0.65,
        )

        plt.scatter(
            sub["avg_weight_bits"],
            sub["balanced_accuracy"],
            s=140,
            marker=markers.get(mode, "o"),
            label=mode,
            edgecolors="black",
            linewidths=0.8,
            zorder=5,
        )

        if annotate_best:
            best = sub.loc[sub["balanced_accuracy"].idxmax()]

            plt.scatter(
                best["avg_weight_bits"],
                best["balanced_accuracy"],
                s=350,
                marker="*",
                edgecolors="black",
                linewidths=1.2,
                zorder=10,
            )

            plt.annotate(
                str(best["budget"]),
                xy=(best["avg_weight_bits"], best["balanced_accuracy"]),
                xytext=(8, 8),
                textcoords="offset points",
                fontsize=11,
                fontweight="bold",
            )

        if annotate_all:
            for _, row in sub.iterrows():
                plt.annotate(
                    str(row["budget"]),
                    xy=(row["avg_weight_bits"], row["balanced_accuracy"]),
                    xytext=(5, 5),
                    textcoords="offset points",
                    fontsize=8,
                )

    plt.title("Weight quantization: compression-performance trade-off", fontsize=20)
    plt.xlabel("Average weight bit-width", fontsize=15)
    plt.ylabel("Balanced Accuracy", fontsize=15)
    plt.grid(True, linestyle="--", alpha=0.35)
    plt.legend(fontsize=12)
    plt.tight_layout()
    plt.show()

    print("\nMerged budgets plotted:")
    print(
        df_plot[
            ["budget", "weight_mode", "avg_weight_bits", "balanced_accuracy", "accepted"]
        ]
        .sort_values(["weight_mode", "avg_weight_bits"])
        .to_string(index=False)
    )

    return df_plot

In [ ]:
df_all_weight_results = plot_all_weight_results_tradeoff(
    results,
    results_pwlq_aggressive,
    results_pwlq_more_aggressive,
    accepted_only=False,
    annotate_best=True,
    annotate_all=False,
)

In [ ]:
calib_ds = train_ds.unbatch().batch(32).take(1024 // 32)

# ── CONSERVATIVE ────────────────────────────────────────
model = load_fresh_model()
sens_map_conservative = sensitivity_aware_quantize_model(
    model,
    model_path         = model_path,
    calib_ds           = calib_ds,
    high_sens_thresh   = 0.01,
    medium_sens_thresh = 0.001,
    low_sens_bits      = 4,
    medium_sens_bits   = 6,
    high_sens_bits     = 8,
    label              = "Sensitivity-Aware Conservative (4/6/8-bit)",
)
compile_and_evaluate(model, test_ds, label="Sensitivity-Aware Conservative (4/6/8-bit)")
plot_weight_distribution_comparison(fp32_model, model, label="Sensitivity-Aware Conservative (4/6/8-bit)")


# ── AGGRESSIVE ──────────────────────────────────────────
model = load_fresh_model()
sens_map_aggressive = sensitivity_aware_quantize_model(
    model,
    model_path         = model_path,
    calib_ds           = calib_ds,
    high_sens_thresh   = 0.01,
    medium_sens_thresh = 0.001,
    low_sens_bits      = 3,
    medium_sens_bits   = 4,
    high_sens_bits     = 6,
    label              = "Sensitivity-Aware Aggressive (3/4/6-bit)",
)
compile_and_evaluate(model, test_ds, label="Sensitivity-Aware Aggressive (3/4/6-bit)")
plot_weight_distribution_comparison(fp32_model, model, label="Sensitivity-Aware Aggressive (3/4/6-bit)")


# ── ULTRA-AGGRESSIVE ────────────────────────────────────
model = load_fresh_model()
sens_map_ultra = sensitivity_aware_quantize_model(
    model,
    model_path         = model_path,
    calib_ds           = calib_ds,
    high_sens_thresh   = 0.01,
    medium_sens_thresh = 0.001,
    low_sens_bits      = 2,
    medium_sens_bits   = 3,
    high_sens_bits     = 4,
    label              = "Sensitivity-Aware Ultra-Aggressive (2/3/4-bit)",
)
compile_and_evaluate(model, test_ds, label="Sensitivity-Aware Ultra-Aggressive (2/3/4-bit)")
plot_weight_distribution_comparison(fp32_model, model, label="Sensitivity-Aware Ultra-Aggressive (2/3/4-bit)")

In [ ]:
df_results, locked_bits, quartile_map, sens_layer_names = \
    sensitivity_quartile_greedy_sweep(
        model_path       = model_path,
        calib_ds         = calib_ds,
        test_ds          = test_ds,
        valid_bits       = [8, 6, 5, 4, 3, 2],
        accuracy_floor   = 0.85,
        measurement_bits = 4,
    )

In [ ]:
# =========================================================
# RUN — CONSERVATIVE
# =========================================================
model = load_fresh_model()
adaptive_quantize_model(
    model,
    conv_bits            = 4,
    dense_high_kurt_bits = 8,
    dense_mid_kurt_bits  = 6,
    dense_low_kurt_bits  = 4,
    kurt_high_thresh     = 4.0,
    kurt_low_thresh      = 1.0,
    label                = "Adaptive Conservative (4/6/8-bit)"
)
compile_and_evaluate(model, test_ds, label="Adaptive Conservative (4/6/8-bit)")
plot_weight_distribution_comparison(fp32_model, model, label="Adaptive Conservative (4/6/8-bit)")


# =========================================================
# RUN — AGGRESSIVE
# =========================================================
model = load_fresh_model()
adaptive_quantize_model(
    model,
    conv_bits            = 3,
    dense_high_kurt_bits = 6,
    dense_mid_kurt_bits  = 4,
    dense_low_kurt_bits  = 3,
    kurt_high_thresh     = 4.0,
    kurt_low_thresh      = 1.0,
    label                = "Adaptive Aggressive (3/4/6-bit)"
)
compile_and_evaluate(model, test_ds, label="Adaptive Aggressive (3/4/6-bit)")
plot_weight_distribution_comparison(fp32_model, model, label="Adaptive Aggressive (3/4/6-bit)")


# =========================================================
# RUN — ULTRA-AGGRESSIVE
# =========================================================
model = load_fresh_model()
adaptive_quantize_model(
    model,
    conv_bits            = 2,
    dense_high_kurt_bits = 4,
    dense_mid_kurt_bits  = 3,
    dense_low_kurt_bits  = 2,
    label                = "Adaptive Ultra-Aggressive (2/3/4-bit)"
)
compile_and_evaluate(model, test_ds, label="Adaptive Ultra-Aggressive (2/3/4-bit)")
plot_weight_distribution_comparison(fp32_model, model, label="Adaptive Ultra-Aggressive (2/3/4-bit)")

In [ ]:
# ==========================================================
# UNIFORM PER-LAYER
# ==========================================================
for bits in [8, 4, 2]:
    model = load_model(model_path)
    lbl   = f"Uniform {bits}-bit (per-layer)"
    quantize_dequantize_uniform(model, num_bits=bits, label=lbl)
    compile_and_evaluate(model, test_ds, label=lbl)
    #plot_weight_distribution_comparison(fp32_model, model, label=lbl)

# ==========================================================
# LOGARITHMIC
# ==========================================================
for bits in [8, 4, 2]:
    model = load_model(model_path)
    lbl   = f"Logarithmic {bits}-bit"
    quantize_dequantize_logarithmic(model, num_bits=bits, label=lbl)
    compile_and_evaluate(model, test_ds, label=lbl)
    #plot_weight_distribution_comparison(fp32_model, model, label=lbl)

# ==========================================================
# K-MEANS
# ==========================================================
for bits in [8, 4, 2]:
    model = load_model(model_path)
    lbl   = f"K-Means {bits}-bit"
    quantize_dequantize_kmeans(model, num_bits=bits, label=lbl)
    compile_and_evaluate(model, test_ds, label=lbl)
    #plot_weight_distribution_comparison(fp32_model, model, label=lbl)

# ==========================================================
# LLOYD-MAX
# ==========================================================
for bits in [8, 4, 2]:
    model = load_model(model_path)
    lbl   = f"Lloyd-Max {bits}-bit"
    quantize_dequantize_lloydmax(model, num_bits=bits, label=lbl)
    compile_and_evaluate(model, test_ds, label=lbl)
    #plot_weight_distribution_comparison(fp32_model, model, label=lbl)

# ==========================================================
# GROUP-WISE
# ==========================================================
for bits in [8, 4, 2]:
    for gs in [64, 32, 16]:
        model = load_model(model_path)
        lbl   = f"Group-wise {bits}-bit (group={gs})"
        quantize_dequantize_groupwise(model, num_bits=bits, group_size=gs, label=lbl)
        compile_and_evaluate(model, test_ds, label=lbl)
        #plot_weight_distribution_comparison(fp32_model, model, label=lbl)

# ==========================================================
# UNIFORM PER-CHANNEL
# ==========================================================
for bits in [8, 4, 2]:
    model = load_model(model_path)
    lbl   = f"Uniform {bits}-bit (per-channel)"
    uniform_quantize_model(model, num_bits=bits, label=lbl)
    compile_and_evaluate(model, test_ds, label=lbl)
    #plot_weight_distribution_comparison(fp32_model, model, label=lbl)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  Runner
# ─────────────────────────────────────────────────────────────────────────────

calib_ds = train_ds.unbatch().batch(32).take(1024 // 32)
results  = {}

for num_bits in [8, 4, 2]:
    model, act_scales, rel_mses = quantize_activations_only_per_layer(
        model_path      = model_path,
        calib_ds        = calib_ds,
        num_bits        = num_bits,
        clip_percentile = 0.999,
    )
    compile_and_evaluate(model, test_ds, label=f"Act-only quant {num_bits}-bit")
    results[num_bits] = rel_mses


# ── Summary table ─────────────────────────────────────────────────────────────
print(f"\n{'='*75}")
print(f"  ACTIVATION QUANTIZATION SUMMARY — Relative MSE per layer")
print(f"{'='*75}")
print(f"  {'Layer':<26}  {'8-bit':>10}  {'4-bit':>10}  {'2-bit':>10}")
print(f"  {'-'*60}")

for layer_name in results[8].keys():
    print(f"  {layer_name:<26}  "
          f"{results[8].get(layer_name, float('nan')):>10.6f}  "
          f"{results[4].get(layer_name, float('nan')):>10.6f}  "
          f"{results[2].get(layer_name, float('nan')):>10.6f}")

print(f"\n  {'AVERAGE':<26}  "
      f"{np.mean(list(results[8].values())):>10.6f}  "
      f"{np.mean(list(results[4].values())):>10.6f}  "
      f"{np.mean(list(results[2].values())):>10.6f}")


# ─────────────────────────────────────────────────────
# 🎯 9. COMPRESSION ANALYSIS METHODS
# ─────────────────────────────────────────────────────

In [ ]:
def compression_analysis_pwlq(model, bits_list=[2, 3, 4, 8], mode="channel", zero_point=False):
    assert mode in ("channel", "layer"), "mode must be 'channel' or 'layer'"

    total_weight_params = 0
    total_bias_params   = 0
    total_scale_factors = 0

    def process_layer(layer):
        nonlocal total_weight_params, total_bias_params, total_scale_factors

        # Recurse into nested models
        if isinstance(layer, tf.keras.Model):
            for sub_layer in layer.layers:
                process_layer(sub_layer)
            return

        if isinstance(layer, (tf.keras.layers.Conv2D, tf.keras.layers.Dense)):
            weights = layer.get_weights()
            if not weights:
                return
            kernel = weights[0]
            total_weight_params += kernel.size

            # Conv2D: (kH, kW, in_ch, out_ch) — Dense: (in_features, out_features)
            out_channels = kernel.shape[-1]

            if mode == "channel":
                total_scale_factors += out_channels
            else:
                total_scale_factors += 1

            if layer.use_bias and len(weights) > 1:
                total_bias_params += weights[1].size

    for layer in model.layers:
        process_layer(layer)

    original_bits      = 32
    weight_original_mb = (total_weight_params * original_bits) / (8 * 1024**2)
    bias_mb            = (total_bias_params   * original_bits) / (8 * 1024**2)

    # PWLQ specific overhead:
    # - 2 scale factors per channel/layer (center + tail region)
    # - 1 breakpoint per channel/layer (float32)
    # - optionally 2 zero points per channel/layer if asymmetric
    scale_mb      = (total_scale_factors * 2 * original_bits) / (8 * 1024**2)  # x2 vs uniform
    breakpoint_mb = (total_scale_factors * 1 * original_bits) / (8 * 1024**2)  # breakpoint p
    zp_mb         = (total_scale_factors * 2 * 32) / (8 * 1024**2) if zero_point else 0.0  # x2 vs uniform

    total_original_mb = weight_original_mb + bias_mb

    print(f"Mode:              per-{'channel' if mode == 'channel' else 'layer'}")
    print(f"Quantization:      PWLQ {'asymmetric (2 scales + 2 zp + breakpoint)' if zero_point else 'symmetric (2 scales + breakpoint)'}")
    print(f"Weight params:     {total_weight_params:,}")
    print(f"Bias params:       {total_bias_params:,}")
    print(f"Scale factors:     {total_scale_factors * 2:,} ({scale_mb:.4f} MB) — 2 per {'channel' if mode == 'channel' else 'layer'}")
    print(f"Breakpoints:       {total_scale_factors:,} ({breakpoint_mb:.4f} MB) — 1 per {'channel' if mode == 'channel' else 'layer'}")
    if zero_point:
        print(f"Zero points:       {total_scale_factors * 2:,} ({zp_mb:.4f} MB) — 2 per {'channel' if mode == 'channel' else 'layer'}")
    print(f"Original size:     {total_original_mb:.4f} MB (float32)")
    print(f"{'─'*95}")
    print(f"{'Bits':<8} {'Weight MB':<14} {'Bias MB':<10} {'Scale MB':<12} {'BKP MB':<10} {'ZP MB':<10} {'Total MB':<12} {'Compression'}")
    print(f"{'─'*95}")
    for bits in sorted(bits_list):
        weight_quantized_mb = (total_weight_params * bits) / (8 * 1024**2)
        total_quantized_mb  = weight_quantized_mb + bias_mb + scale_mb + breakpoint_mb + zp_mb
        ratio               = total_original_mb / total_quantized_mb
        print(f"{bits:<8} {weight_quantized_mb:<14.4f} {bias_mb:<10.4f} {scale_mb:<12.4f} {breakpoint_mb:<10.4f} {zp_mb:<10.4f} {total_quantized_mb:<12.4f} x{ratio:.2f}")
    print(f"{'─'*95}")

# symmetric PWLQ
compression_analysis_pwlq(model, bits_list=[2, 3, 4, 8], mode="channel", zero_point=False)
# asymmetric PWLQ
compression_analysis_pwlq(model, bits_list=[2, 3, 4, 8], mode="channel", zero_point=True)

# layer-wise symmetric PWLQ
compression_analysis_pwlq(model, bits_list=[2, 3, 4, 8], mode="layer", zero_point=False)
# layer-wise asymmetric PWLQ
compression_analysis_pwlq(model, bits_list=[2, 3, 4, 8], mode="layer", zero_point=True)

In [ ]:
def analyze_model_weights(model, prefix=""):
    """
    Recursively analyzes every Conv2D and Dense layer in a Keras model,
    printing and returning the number of weights per channel and per layer.

    Returns:
        dict: {
            layer_name: {
                "shape":               tuple,
                "out_channels":        int,
                "weights_per_channel": int,
                "total_weights":       int
            }
        }
    """
    info = {}

    for layer in model.layers:
        full_name = f"{prefix}.{layer.name}" if prefix else layer.name

        # Recurse into nested models (e.g. MFCC/MSLFB sub-models in ensemble)
        if isinstance(layer, tf.keras.Model):
            sub_info = analyze_model_weights(layer, full_name)
            info.update(sub_info)
            continue

        if not isinstance(layer, (tf.keras.layers.Conv2D, tf.keras.layers.Dense)):
            continue

        weights = layer.get_weights()
        if not weights:
            continue

        kernel = weights[0]
        out_ch         = kernel.shape[-1]
        weights_per_ch = kernel[..., 0].size   # all dims except out_channel
        total_weights  = kernel.size

        info[full_name] = {
            "shape":               kernel.shape,
            "out_channels":        out_ch,
            "weights_per_channel": weights_per_ch,
            "total_weights":       total_weights
        }

        print(f"Layer: {full_name}")
        print(f"  shape:               {kernel.shape}")
        print(f"  out_channels:        {out_ch}")
        print(f"  weights_per_channel: {weights_per_ch}")
        print(f"  total_weights:       {total_weights}")
        print()

    return info

In [ ]:
def compression_analysis(model, bits_list=[2, 3, 4, 8], mode="channel", zero_point=False):
    assert mode in ("channel", "layer"), "mode must be 'channel' or 'layer'"

    total_weight_params = 0
    total_bias_params   = 0
    total_scale_factors = 0

    def process_layer(layer):
        nonlocal total_weight_params, total_bias_params, total_scale_factors

        # Recurse into nested models
        if isinstance(layer, tf.keras.Model):
            for sub_layer in layer.layers:
                process_layer(sub_layer)
            return

        if isinstance(layer, (tf.keras.layers.Conv2D, tf.keras.layers.Dense)):
            weights = layer.get_weights()
            if not weights:
                return
            kernel = weights[0]
            total_weight_params += kernel.size

            if isinstance(layer, tf.keras.layers.Conv2D):
                # Conv2D kernel shape: (kH, kW, in_ch, out_ch)
                out_channels = kernel.shape[-1]
            else:
                # Dense kernel shape: (in_features, out_features)
                out_channels = kernel.shape[-1]

            if mode == "channel":
                total_scale_factors += out_channels  # one scale per out_channel/neuron
            else:
                total_scale_factors += 1             # one scale per layer

            if layer.use_bias and len(weights) > 1:
                total_bias_params += weights[1].size

    for layer in model.layers:
        process_layer(layer)

    original_bits      = 32
    weight_original_mb = (total_weight_params * original_bits) / (8 * 1024**2)
    bias_mb            = (total_bias_params   * original_bits) / (8 * 1024**2)
    scale_mb           = (total_scale_factors * original_bits) / (8 * 1024**2)
    zp_mb              = (total_scale_factors * 32)            / (8 * 1024**2) if zero_point else 0.0
    total_original_mb  = weight_original_mb + bias_mb

    print(f"Mode:              per-{'channel' if mode == 'channel' else 'layer'}")
    print(f"Quantization:      {'asymmetric (scale + zero_point)' if zero_point else 'symmetric (scale only)'}")
    print(f"Weight params:     {total_weight_params:,}")
    print(f"Bias params:       {total_bias_params:,}")
    print(f"Scale factors:     {total_scale_factors:,} ({scale_mb:.4f} MB at float32)")
    if zero_point:
        print(f"Zero points:       {total_scale_factors:,} ({zp_mb:.4f} MB at int32)")
    print(f"Original size:     {total_original_mb:.4f} MB (float32)")
    print(f"{'─'*85}")
    print(f"{'Bits':<8} {'Weight MB':<14} {'Bias MB':<10} {'Scale MB':<12} {'ZP MB':<10} {'Total MB':<12} {'Compression'}")
    print(f"{'─'*85}")
    for bits in sorted(bits_list):
        weight_quantized_mb = (total_weight_params * bits) / (8 * 1024**2)
        total_quantized_mb  = weight_quantized_mb + bias_mb + scale_mb + zp_mb
        ratio               = total_original_mb / total_quantized_mb
        print(f"{bits:<8} {weight_quantized_mb:<14.4f} {bias_mb:<10.4f} {scale_mb:<12.4f} {zp_mb:<10.4f} {total_quantized_mb:<12.4f} x{ratio:.2f}")
    print(f"{'─'*85}")

# symmetric (no zero point)
compression_analysis(model, bits_list=[2, 3, 4, 8], mode="channel", zero_point=False)
# asymmetric (with zero point)
compression_analysis(model, bits_list=[2, 3, 4, 8], mode="channel", zero_point=True)

# layer-wise symmetric (no zero point)
compression_analysis(model, bits_list=[2, 3, 4, 8], mode="layer", zero_point=False)
# layer-wise asymmetric (with zero point)
compression_analysis(model, bits_list=[2, 3, 4, 8], mode="layer", zero_point=True)

In [ ]:
def print_unique_weights(model, max_print=16, prefix=""):
    for layer in model.layers:
        layer_name = f"{prefix}/{layer.name}" if prefix else layer.name

        if isinstance(layer, (tf.keras.Model, tf.keras.Sequential)):
            print_unique_weights(layer, max_print, prefix=layer_name)
            continue

        if not layer.get_weights():
            continue

        kernel = layer.get_weights()[0]

        if kernel.ndim != 4:
            continue

        w = tf.reshape(kernel, [-1])
        unique_vals = tf.sort(tf.unique(w).y)

        print(f"\n{layer_name}")
        print(f"  kernel shape: {kernel.shape}")
        print(f"  number of unique weights: {unique_vals.shape[0]}")
        print(f"  first {min(max_print, unique_vals.shape[0])} values:")
        print(unique_vals[:max_print].numpy())


# ─────────────────────────────────────────────────────
# 🔬 10. STATE OF THE ART METHODS
# ─────────────────────────────────────────────────────

## ─────────────────────────────────────────────────────
## 🔬 10.1 PWLQ
## ─────────────────────────────────────────────────────

In [ ]:
import os
import numpy as np
import torch
import tensorflow as tf
from tensorflow.keras.models import load_model
import sys
sys.path.append("/content")
from pwlq.pwlq import piecewise_linear_quant


def quantize_model_pwlq(model, bits, scale_bits=0, break_point="norm",
                         pw_opt=2, approximate=True, granularity="per_layer",
                         label=None, verbose=True):
    assert granularity in ("per_layer", "per_channel"), \
        "granularity must be 'per_layer' or 'per_channel'"

    if label is None:
        label = f"PWLQ {bits}-bit ({granularity})"
    layers = collect_all_weight_layers(model)

    if verbose:
        print(f"\n{'='*60}")
        print(f"  {label}")
        print(f"{'='*60}")

    for layer in layers:
        weights = layer.get_weights()
        if not weights:
            continue

        W  = weights[0]
        b  = weights[1] if len(weights) > 1 else None

        w_torch = torch.from_numpy(W.astype(np.float32))

        if granularity == "per_layer":
            qw_flat, _, _ = piecewise_linear_quant(
                w_torch.reshape(-1),
                bits=bits,
                scale_bits=scale_bits,
                break_point_approach=break_point,
                pw_opt=pw_opt,
                approximate=approximate,
            )
            Wq = qw_flat.reshape(w_torch.shape).numpy()

        else:  # per_channel
            w_flat  = w_torch.reshape(w_torch.shape[0], -1)
            qw_list = []
            for c in range(w_flat.shape[0]):
                qw, _, _ = piecewise_linear_quant(
                    w_flat[c],
                    bits=bits,
                    scale_bits=scale_bits,
                    break_point_approach=break_point,
                    pw_opt=pw_opt,
                    approximate=approximate,
                )
                qw_list.append(qw)
            Wq = torch.stack(qw_list).reshape(w_torch.shape).numpy()

        layer.set_weights([Wq, b] if b is not None else [Wq])

        if verbose:
            is_conv = isinstance(layer, tf.keras.layers.Conv2D)
            print(f"  {layer.name:<22} {'Conv2D' if is_conv else 'Dense':<8} → {bits}-bit ({granularity})")

    if verbose:
        print("\nQuantization complete.")

In [ ]:
file_date  = "20260320_064356"
model_path = os.path.join(main_directory, f"results/ensemble_mfcc_mslfb_resumed_{file_date}.keras")

for bits in [8, 4, 2]:
    for granularity in ["per_layer", "per_channel"]:
        model = load_model(model_path)
        lbl   = f"PWLQ {bits}-bit ({granularity})"
        quantize_model_pwlq(model, bits=bits, granularity=granularity, label=lbl)
        compile_and_evaluate(model, test_ds, label=lbl)
        plot_weight_distribution_comparison(fp32_model, model, label=lbl)

## ─────────────────────────────────────────────────────
## 🔬 10.2 BITSPLIT
## ─────────────────────────────────────────────────────

In [ ]:
import os
import numpy as np
import tensorflow as tf


# ---------------------------------------------------------
# BIT-SPLIT (OFWA) CORE
# ---------------------------------------------------------

def fwa(W, bitwidth):
    """
    Fixed-point Weight Approximation (FWA).

    Minimises || W - alpha * Q ||^2 over integer Q and scale alpha.
    W shape: (out_channels, flattened_kernel_size)
    """
    max_val = 2 ** (bitwidth - 1) - 1

    alpha = np.abs(W).max(axis=1) / max_val
    alpha_old = alpha * 1.1

    while np.linalg.norm(alpha - alpha_old) > 1e-9:
        q = W / alpha[:, None]
        q = np.round(q)
        q = np.clip(q, -max_val, max_val)
        alpha_old = alpha
        alpha = np.sum(W * q, axis=1) / np.sum(q * q, axis=1)

    return q, alpha


def split(Q, bitwidth):
    """
    Bit-Split: decompose integer matrix Q into (bitwidth-1) ternary bit planes.
    Each plane B_k in {-1, 0, +1}. MSB is at index 0 (list reversed at end).

    Note: uses np.int32 instead of the deprecated np.int (removed in NumPy 1.24).
    """
    Q_sign = np.sign(Q)
    Q_abs  = np.abs(Q)
    B_sav  = []

    for _ in range(bitwidth - 1):
        # Extract LSB: equivalent to Q_abs % 2
        B = Q_abs - (Q_abs.astype(np.int32) // 2) * 2
        B *= Q_sign
        B_sav.append(B)
        # Right-shift
        Q_abs = (Q_abs.astype(np.int32) // 2).astype(np.float32)

    return B_sav[::-1]  # MSB first


def get_int_B(B_sav):
    """
    Bit-Stitch: reconstruct fixed-point value from ternary planes.
    q = B0 + B1/2 + B2/4 + ...   (B0 is MSB)
    """
    B_sum = B_sav[0].copy()
    for i in range(1, len(B_sav)):
        B_sum += B_sav[i] / (2 ** i)
    return B_sum


def get_int_B_exclusive(B_sav, bit):
    """
    Bit-Stitch excluding plane `bit`. Used for residual computation.
    """
    B_sum = np.zeros_like(B_sav[0])
    for i in range(len(B_sav)):
        if i == bit:
            continue
        B_sum += B_sav[i] / (2 ** i)
    return B_sum


def get_ot_given_a(W, alpha):
    """
    Optimal ternary update given residual W and scale alpha.
    Implements Eq. (8) from the paper:
      +1 or -1  if |W| >= alpha/2
       0        otherwise
    """
    B = np.sign(W)
    B[np.abs(W) * 2 < alpha[:, None]] = 0
    return B


def ofwa(W, bitwidth, max_epoch=50):
    """
    Optimal Fixed-point Weight Approximation (OFWA).

    Pipeline:
      1. FWA initialisation
      2. Bit-Split
      3. Alternating optimisation of alpha and each bit plane
      4. Bit-Stitch

    Returns
    -------
    B_sav : list of ternary planes
    B_sum : reconstructed fixed-point integer matrix
    alpha : per-channel scale factors
    """
    assert bitwidth >= 3, "OFWA requires bitwidth >= 3"

    Q, alpha = fwa(W, bitwidth)
    B_sav    = split(Q, bitwidth)

    # Reposition decimal point: absorb implicit base 2^(bitwidth-2) into alpha
    # so that get_int_B() works with values in [-1, 1].
    alpha = alpha * (2 ** (bitwidth - 2))

    for _ in range(max_epoch):
        alpha_old = alpha.copy()

        B_sum = get_int_B(B_sav)
        # Least-squares update for alpha (Eq. 4)
        alpha = np.sum(W * B_sum, axis=1) / np.sum(B_sum * B_sum, axis=1)

        if np.linalg.norm(alpha - alpha_old) <= 1e-9:
            break

        # Update each bit plane sequentially (Eq. 5-8)
        for bit in range(bitwidth - 1):
            W_res      = W - get_int_B_exclusive(B_sav, bit) * alpha[:, None]
            B_sav[bit] = get_ot_given_a(W_res * (2 ** bit), alpha)

    B_sum = get_int_B(B_sav)
    return B_sav, B_sum, alpha


# ---------------------------------------------------------
# KERAS KERNEL QUANTIZATION
# ---------------------------------------------------------

def bitsplit_quantize_kernel(kernel, num_bits):
    """
    Apply OFWA to a Keras kernel.

    Conv2D kernel shape: (H, W, Cin, Cout) -> reshape to (Cout, H*W*Cin)
    Dense  kernel shape: (Cin, Cout)        -> transpose to (Cout, Cin)
    """
    original_shape = kernel.shape

    if kernel.ndim == 4:
        # (H, W, Cin, Cout) -> (Cout, H, W, Cin) -> (Cout, H*W*Cin)
        W = np.transpose(kernel, (3, 0, 1, 2))
        W = W.reshape(W.shape[0], -1)

    elif kernel.ndim == 2:
        W = kernel.T  # (Cout, Cin)

    else:
        return kernel  # unknown shape, return unchanged

    _B_sav, B_sum, alpha = ofwa(W, num_bits)

    W_q = B_sum * alpha[:, None]

    if kernel.ndim == 4:
        W_q = W_q.reshape(
            original_shape[3],  # Cout
            original_shape[0],  # H
            original_shape[1],  # W
            original_shape[2],  # Cin
        )
        W_q = np.transpose(W_q, (1, 2, 3, 0))  # back to (H, W, Cin, Cout)
    else:
        W_q = W_q.T  # back to (Cin, Cout)

    return W_q.astype(np.float32)


# ---------------------------------------------------------
# RECURSIVE MODEL QUANTIZATION
# ---------------------------------------------------------

def quantize_all_layers_tf(model, num_bits=8):
    """
    Recursively apply OFWA to Conv2D and Dense layers.
    Recurses into Sequential submodels (e.g. model_mfcc, model_mslfb).
    Biases are always left at full precision.
    """
    assert num_bits >= 3, "Bit-split OFWA requires num_bits >= 3"

    for layer in model.layers:

        # Recurse into Sequential submodels
        if isinstance(layer, tf.keras.Model):
            quantize_all_layers_tf(layer, num_bits)
            continue

        if isinstance(layer, (tf.keras.layers.Conv2D,
                              tf.keras.layers.Dense)):
            weights = layer.get_weights()
            if not weights:
                continue

            W_q = bitsplit_quantize_kernel(weights[0], num_bits)

            if len(weights) == 2:
                layer.set_weights([W_q, weights[1]])  # preserve bias
            else:
                layer.set_weights([W_q])


# ---------------------------------------------------------
# LOAD MODEL, QUANTIZE, EVALUATE
# ---------------------------------------------------------

file_date  = "20260320_064356"
model_path = os.path.join(main_directory, f"results/ensemble_mfcc_mslfb_resumed_{file_date}.keras")

for bits in [8, 4, 3]:
    model = load_model(model_path)
    lbl   = f"BitSplit {bits}-bit (per-channel)"
    quantize_all_layers_tf(model, num_bits=bits)
    compile_and_evaluate(model, test_ds, label=lbl)
    plot_weight_distribution_comparison(fp32_model, model, label=lbl)

## ─────────────────────────────────────────────────────
## 🔬 10.3 ADAROUND
## ─────────────────────────────────────────────────────

In [ ]:
import os
import tensorflow as tf
import numpy as np
from tensorflow.keras.models import load_model


# =========================================================
# SCALE — fixed MSE-optimal, computed once (paper Section 5)
# =========================================================
def compute_initial_scale(w, num_bits):
    qmin = -(2 ** (num_bits - 1))
    qmax =  (2 ** (num_bits - 1)) - 1
    axes = tuple(range(len(w.shape) - 1))

    w_max      = tf.reduce_max(tf.abs(w), axis=axes, keepdims=True) + 1e-8
    scale_init = w_max / qmax

    best_scale = scale_init
    best_mse   = tf.reduce_mean((w - tf.clip_by_value(
                    tf.round(w / scale_init), qmin, qmax) * scale_init) ** 2)

    for alpha in np.linspace(0.7, 1.0, 20):
        s   = scale_init * alpha
        w_q = tf.clip_by_value(tf.round(w / s), qmin, qmax) * s
        mse = tf.reduce_mean((w - w_q) ** 2)
        if mse < best_mse:
            best_mse   = mse
            best_scale = s

    return best_scale, qmin, qmax


# =========================================================
# RECTIFIED SIGMOID h(V) — paper equation (23)
# =========================================================
def rectified_sigmoid(v, gamma=-0.1, zeta=1.1):
    s = tf.sigmoid(v)
    h = s * (zeta - gamma) + gamma
    return tf.clip_by_value(h, 0.0, 1.0)


# =========================================================
# REGULARIZER f_reg — paper equation (24)
# =========================================================
def rounding_regularizer(hv, beta):
    return tf.reduce_sum(1.0 - tf.pow(tf.abs(2.0 * hv - 1.0), beta))


# =========================================================
# COLLECT LAYERS RECURSIVELY
# =========================================================
def collect_quant_layers(model):
    layers = []
    def _collect(layer):
        if isinstance(layer, (tf.keras.layers.Conv2D, tf.keras.layers.Dense)):
            layers.append(layer)
        if hasattr(layer, "layers"):
            for sub in layer.layers:
                _collect(sub)
    for l in model.layers:
        _collect(l)
    return layers


# =========================================================
# CAPTURE LAYER ACTIVATIONS — patches instance not class
# =========================================================
def capture_layer_activations(model, layer, calib_ds):
    """
    Capture input and output activations for a specific layer
    by patching the instance's call method directly.
    Patching the instance (layer.call) instead of the class
    (layer.__class__.call) ensures the hook only fires for
    this specific layer, not all layers of the same type.
    """
    captured = {"inp": [], "out": []}
    original_call = layer.call

    def hooked_call(inputs, *args, **kwargs):
        result = original_call(inputs, *args, **kwargs)
        captured["inp"].append(tf.identity(inputs))
        captured["out"].append(tf.identity(result))
        return result

    layer.call = hooked_call  # patch instance, not class

    for batch in calib_ds:
        x_batch, _ = batch
        model(x_batch, training=False)

    layer.call = original_call  # restore

    return captured["inp"], captured["out"]


# =========================================================
# MANUAL LAYER FORWARD (for Conv2D and Dense)
# =========================================================
def manual_forward(layer, x, w, b):
    if isinstance(layer, tf.keras.layers.Conv2D):
        y = tf.nn.conv2d(x, w, strides=layer.strides, padding=layer.padding.upper())
        if b is not None:
            y = tf.nn.bias_add(y, b)
    elif isinstance(layer, tf.keras.layers.Dense):
        y = tf.matmul(x, w)
        if b is not None:
            y = y + b
    if layer.activation is not None:
        y = layer.activation(y)
    return y


# =========================================================
# ADAROUND — single layer optimization
# =========================================================
def adaround_layer(
    model, layer, calib_ds,
    num_bits=4, num_iters=10000,
    lambda_reg=1.0,
    beta_start=20.0,
    beta_end=2.0,
    warmup=0.2,
    lr=1e-3
):
    weights = layer.get_weights()
    if not weights:
        return

    print(f"\n=== AdaRound: {layer.name} | shape: {weights[0].shape} ===")

    w = tf.constant(weights[0], tf.float32)
    b = tf.constant(weights[1], tf.float32) if len(weights) > 1 else None

    scale, qmin, qmax = compute_initial_scale(w, num_bits)
    w_scaled = w / scale

    # Capture calibration activations
    layer_inputs, layer_outputs = capture_layer_activations(model, layer, calib_ds)

    if not layer_inputs:
        print(f"  WARNING: No activations captured for {layer.name}, skipping.")
        return

    # Baseline naive MSE
    w_naive   = tf.clip_by_value(tf.round(w_scaled), qmin, qmax) * scale
    naive_mse = np.mean([
        tf.reduce_mean((y_fp - manual_forward(layer, x_l, w_naive, b)) ** 2).numpy()
        for x_l, y_fp in zip(layer_inputs, layer_outputs)
    ])
    print(f"  Naive rounding MSE:   {naive_mse:.8f}")

    # Initialize V
    frac         = w_scaled - tf.floor(w_scaled)
    frac_clipped = tf.clip_by_value(frac, 0.001, 0.999)
    gamma, zeta  = -0.1, 1.1
    s_init       = (frac_clipped - gamma) / (zeta - gamma)
    s_init       = tf.clip_by_value(s_init, 0.001, 0.999)
    V            = tf.Variable(tf.math.log(s_init / (1.0 - s_init)), trainable=True)

    optimizer    = tf.keras.optimizers.Adam(lr)
    warmup_iters = int(num_iters * warmup)
    n_batches    = len(layer_inputs)

    for it in range(num_iters):
        progress = max(0.0, (it - warmup_iters) / max(num_iters - warmup_iters, 1))
        beta     = beta_start + (beta_end - beta_start) * progress

        idx  = np.random.randint(0, n_batches)
        x_l  = layer_inputs[idx]
        y_fp = layer_outputs[idx]

        with tf.GradientTape() as tape:
            h_v         = rectified_sigmoid(V)
            w_tilde_int = tf.clip_by_value(tf.floor(w_scaled) + h_v, qmin, qmax)
            w_tilde     = w_tilde_int * scale

            y_q      = manual_forward(layer, x_l, w_tilde, b)
            rec_loss = tf.reduce_mean((y_fp - y_q) ** 2)

            if it >= warmup_iters:
                reg_loss = lambda_reg * rounding_regularizer(h_v, beta)
            else:
                reg_loss = 0.0

            loss = rec_loss + reg_loss

        grads = tape.gradient(loss, [V])
        optimizer.apply_gradients(zip(grads, [V]))

        if it % 1000 == 0 or it == num_iters - 1:
            h_debug = rectified_sigmoid(V)
            pct_up  = tf.reduce_mean(tf.cast(h_debug > 0.5, tf.float32)).numpy() * 100
            print(f"  iter {it:05d} | rec={rec_loss:.2e} | beta={beta:.2f} | >0.5={pct_up:.1f}%")

    # Hard rounding
    h_hard      = tf.cast(rectified_sigmoid(V) >= 0.5, tf.float32)
    w_final_int = tf.clip_by_value(tf.floor(w_scaled) + h_hard, qmin, qmax)
    w_final     = w_final_int * scale

    pct_up = tf.reduce_mean(h_hard).numpy() * 100
    print(f"  Hard rounding: {pct_up:.1f}% rounded up")

    adaround_mse = np.mean([
        tf.reduce_mean((y_fp - manual_forward(layer, x_l, w_final, b)) ** 2).numpy()
        for x_l, y_fp in zip(layer_inputs, layer_outputs)
    ])
    print(f"  AdaRound MSE:         {adaround_mse:.8f}")
    print(f"  MSE improvement:      {naive_mse / max(adaround_mse, 1e-10):.2f}x")

    layer.set_weights([w_final.numpy(), b.numpy()] if b is not None else [w_final.numpy()])


# =========================================================
# APPLY ADAROUND TO FULL MODEL
# =========================================================
def apply_adaround(model, calib_ds, num_bits=4, num_iters=10000):
    quant_layers = collect_quant_layers(model)

    print("Quantizable layers:")
    for i, l in enumerate(quant_layers):
        print(f"  [{i:02d}] {l.name} | shape: {l.get_weights()[0].shape if l.get_weights() else 'N/A'}")

    for layer in quant_layers:
        adaround_layer(model, layer, calib_ds, num_bits=num_bits, num_iters=num_iters)

    print("\nAdaRound complete.")


# =========================================================
# LOAD + RUN
# =========================================================
file_date  = "20260320_064356"
model_path = os.path.join(main_directory, f"results/ensemble_mfcc_mslfb_resumed_{file_date}.keras")

calib_ds = train_ds.unbatch().batch(32).take(1024 // 32)

for bits in [8, 4, 2]:
    model = load_model(model_path)
    lbl   = f"AdaRound {bits}-bit"
    apply_adaround(model, calib_ds, num_bits=bits)
    compile_and_evaluate(model, test_ds, label=lbl)
    plot_weight_distribution_comparison(fp32_model, model, label=lbl)

## ─────────────────────────────────────────────────────
## 🔬 10.4 SUBSETQ
## ─────────────────────────────────────────────────────

In [ ]:
from itertools import combinations
from joblib import Parallel, delayed

# ---------------------------------------------------------
# UNIVERSAL SET & HELPERS
# ---------------------------------------------------------
def gen_universal_set():
    powers = [1.0, 0.5, 0.25, 0.125, 0.0]
    uset   = set()
    for a in powers:
        for b in powers:
            v = a + b
            if v > 0:
                uset.add(round(v, 6))
    uset    = sorted(uset, reverse=True)
    max_val = uset[0]
    uset    = [v / max_val for v in uset]
    return np.array(sorted(set([round(v, 6) for v in uset]), reverse=True))


def find_scale_factor(w_abs, S, max_iter=50, tol=1e-5):
    alpha = 1.0
    for _ in range(max_iter):
        S_scaled  = alpha * S
        qi        = S_scaled[np.argmin(np.abs(w_abs[:, None] - S_scaled[None, :]), axis=1)]
        denom     = np.sum(qi ** 2)
        if denom < 1e-10:
            break
        alpha_new = np.sum(w_abs * qi) / denom
        if abs(alpha_new - alpha) < tol:
            alpha = alpha_new
            break
        alpha = alpha_new
    return alpha


def _find_best_qps_fast(w_flat, combos):
    w_abs  = np.abs(w_flat)
    signs  = np.sign(w_flat)
    signs[signs == 0] = 1

    best_loss = np.inf
    best_wq   = None

    for S in combos:
        S      = np.array(S)
        alpha  = find_scale_factor(w_abs, S)
        S_sc   = alpha * S
        idx    = np.argmin(np.abs(w_abs[:, None] - S_sc[None, :]), axis=1)
        qi     = S_sc[idx]
        wq     = qi * signs
        loss   = np.mean((w_flat - wq) ** 2)
        if loss < best_loss:
            best_loss = loss
            best_wq   = wq

    return best_wq


# ---------------------------------------------------------
# PROCESS ONE CHANNEL — used by joblib
# ---------------------------------------------------------
def _process_channel(w_ch, combos):
    return _find_best_qps_fast(w_ch, combos)


# ---------------------------------------------------------
# PER-CHANNEL SubsetQ — parallelized
# ---------------------------------------------------------
def subset_quantize_channel_wise(W, bits=4, n_jobs=-1):
    Su     = gen_universal_set()
    N      = min(2 ** (bits - 1), len(Su))
    combos = list(combinations(Su, N))
    out_ch = W.shape[-1]
    W_r    = W.reshape(-1, out_ch)

    results = Parallel(n_jobs=n_jobs, prefer='threads')(
        delayed(_process_channel)(W_r[:, c], combos)
        for c in range(out_ch)
    )

    Wq = np.stack(results, axis=1)
    return Wq.reshape(W.shape)


# ---------------------------------------------------------
# PER-LAYER SubsetQ
# ---------------------------------------------------------
def subset_quantize_layer_wise(W, bits=4):
    Su     = gen_universal_set()
    N      = min(2 ** (bits - 1), len(Su))
    combos = list(combinations(Su, N))
    wq     = _find_best_qps_fast(W.flatten(), combos)
    return wq.reshape(W.shape)


# ---------------------------------------------------------
# APPLY TO KERAS MODEL
# ---------------------------------------------------------
def apply_subsetq_keras(model, bits=4, mode="channel", n_jobs=-1):
    assert mode in ("channel", "layer"), "mode must be 'channel' or 'layer'"
    layers = collect_all_weight_layers(model)

    print(f"\n{'='*60}")
    print(f"  SubsetQ {bits}-bit (per-{mode})")
    print(f"{'='*60}")

    for layer in layers:
        weights = layer.get_weights()
        if not weights:
            continue

        W  = weights[0]
        b  = weights[1] if len(weights) > 1 else None
        is_conv = isinstance(layer, tf.keras.layers.Conv2D)

        print(f"  {layer.name:<22} {'Conv2D' if is_conv else 'Dense':<8} "
              f"shape={W.shape}")

        if mode == "channel":
            Wq = subset_quantize_channel_wise(W, bits=bits, n_jobs=n_jobs)
        else:
            Wq = subset_quantize_layer_wise(W, bits=bits)

        layer.set_weights([Wq.astype(np.float32), b] if b is not None
                          else [Wq.astype(np.float32)])

    print("\nQuantization complete.")
    return model

In [ ]:
# ---------------------------------------------------------
# RUN
# ---------------------------------------------------------
file_date  = "20260320_064356"
model_path = os.path.join(main_directory, f"results/ensemble_mfcc_mslfb_resumed_{file_date}.keras")

for mode in ["layer"]:
    for bits in [4, 3, 2]:
        model = load_model(model_path)
        lbl   = f"SubsetQ {bits}-bit (per-{mode})"
        apply_subsetq_keras(model, bits=bits, mode=mode)
        compile_and_evaluate(model, test_ds, label=lbl)
        plot_weight_distribution_comparison(fp32_model, model, label=lbl)

In [ ]:
# ---------------------------------------------------------
# RUN
# ---------------------------------------------------------
file_date  = "20260320_064356"
model_path = os.path.join(main_directory, f"results/ensemble_mfcc_mslfb_resumed_{file_date}.keras")

for mode in ["channel"]:
    for bits in [4, 3, 2]:
        model = load_model(model_path)
        lbl   = f"SubsetQ {bits}-bit (per-{mode})"
        apply_subsetq_keras(model, bits=bits, mode=mode)
        compile_and_evaluate(model, test_ds, label=lbl)
        plot_weight_distribution_comparison(fp32_model, model, label=lbl)

## ─────────────────────────────────────────────────────
## 🔬 10.5 BRECQ
## ─────────────────────────────────────────────────────

### ─────────────────────────────────────────────────────
### 🔬 10.5.1 BRECQ SETUP
### ─────────────────────────────────────────────────────

In [ ]:
import os
import tensorflow as tf
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"


# ==========================================================
# PADDING HELPER
# ==========================================================
def tf_same_pad(x, kernel_size=2):
    h, w       = x.shape[2], x.shape[3]
    pad_h      = (kernel_size - 1) if h % 1 == 0 else 0
    pad_w      = (kernel_size - 1) if w % 1 == 0 else 0
    pad_top    = pad_h // 2;  pad_bottom = pad_h - pad_top
    pad_left   = pad_w // 2;  pad_right  = pad_w - pad_left
    return F.pad(x, (pad_left, pad_right, pad_top, pad_bottom))


# ==========================================================
# MODEL — WITH BN (no folding)
# ==========================================================
class SubModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1,   16, 4, padding=0, bias=True)
        self.bn1   = nn.BatchNorm2d(16,  eps=0.001)
        self.drop1 = nn.Dropout(0.45)
        self.conv2 = nn.Conv2d(16,  32, 4, padding=0, bias=True)
        self.bn2   = nn.BatchNorm2d(32,  eps=0.001)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.drop2 = nn.Dropout(0.45)
        self.conv3 = nn.Conv2d(32,  64, 2, padding=0, bias=True)
        self.bn3   = nn.BatchNorm2d(64,  eps=0.001)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.drop3 = nn.Dropout(0.45)
        self.conv4 = nn.Conv2d(64, 128, 2, padding=0, bias=True)
        self.bn4   = nn.BatchNorm2d(128, eps=0.001)
        self.drop4 = nn.Dropout(0.45)
        self.fc1   = None
        self.drop5 = nn.Dropout(0.45)
        self.fc2   = None
        self.drop6 = nn.Dropout(0.45)
        self.fc3   = None

    def _build_fcs(self, flat_size):
        self.fc1 = nn.Linear(flat_size, 256).to(device)
        self.fc2 = nn.Linear(256, 128).to(device)
        self.fc3 = nn.Linear(128, 12).to(device)

    def forward(self, x):
        x = x.permute(0, 3, 1, 2)
        x = self.bn1(F.relu(self.conv1(x)));  x = self.drop1(x)
        x = self.bn2(F.relu(self.conv2(x)));  x = self.pool1(x);  x = self.drop2(x)
        x = tf_same_pad(x)
        x = self.bn3(F.relu(self.conv3(x)));  x = self.pool2(x);  x = self.drop3(x)
        x = tf_same_pad(x)
        x = self.bn4(F.relu(self.conv4(x)));  x = self.drop4(x)
        x = x.permute(0, 2, 3, 1)
        x = torch.flatten(x, 1)
        if self.fc1 is None:
            self._build_fcs(x.shape[1])
        x = F.relu(self.fc1(x));  x = self.drop5(x)
        x = F.relu(self.fc2(x));  x = self.drop6(x)
        x = F.relu(self.fc3(x))
        return x


class DualBranchEnsemble(nn.Module):
    def __init__(self):
        super().__init__()
        self.branch_mfcc  = SubModel()
        self.branch_mslfb = SubModel()
        self.classifier   = nn.Linear(24, 12)

    def forward(self, x1, x2):
        o1 = self.branch_mfcc(x1)
        o2 = self.branch_mslfb(x2)
        x  = torch.cat([o1, o2], dim=1)
        return F.softmax(self.classifier(x), dim=1)


# ==========================================================
# WRAPPER
# ==========================================================
class TupleInputModel(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.branch_mfcc  = model.branch_mfcc
        self.branch_mslfb = model.branch_mslfb
        self.classifier   = model.classifier

    def forward(self, x):
        x1, x2 = x
        o1  = self.branch_mfcc(x1)
        o2  = self.branch_mslfb(x2)
        out = torch.cat([o1, o2], dim=1)
        return torch.nn.functional.softmax(self.classifier(out), dim=1)


# ==========================================================
# WEIGHT TRANSFER HELPERS
# ==========================================================
def load_conv(tf_layer, torch_layer):
    weights = tf_layer.get_weights()
    w = np.transpose(weights[0], (3, 2, 0, 1))
    torch_layer.weight.data.copy_(torch.tensor(w).float().to(device))
    if len(weights) > 1:
        torch_layer.bias.data.copy_(torch.tensor(weights[1]).float().to(device))

def load_batchnorm(tf_layer, torch_layer):
    gamma, beta, moving_mean, moving_var = tf_layer.get_weights()
    torch_layer.weight.data.copy_(      torch.tensor(gamma).float().to(device))
    torch_layer.bias.data.copy_(        torch.tensor(beta).float().to(device))
    torch_layer.running_mean.data.copy_(torch.tensor(moving_mean).float().to(device))
    torch_layer.running_var.data.copy_( torch.tensor(moving_var).float().to(device))

def load_dense(tf_layer, torch_layer):
    w, b = tf_layer.get_weights()
    torch_layer.weight.data.copy_(torch.tensor(w.T).float().to(device))
    torch_layer.bias.data.copy_(  torch.tensor(b).float().to(device))

def transfer_branch(tf_seq, pt_branch):
    load_conv(     tf_seq.layers[0],  pt_branch.conv1)
    load_batchnorm(tf_seq.layers[1],  pt_branch.bn1)
    load_conv(     tf_seq.layers[3],  pt_branch.conv2)
    load_batchnorm(tf_seq.layers[4],  pt_branch.bn2)
    load_conv(     tf_seq.layers[7],  pt_branch.conv3)
    load_batchnorm(tf_seq.layers[8],  pt_branch.bn3)
    load_conv(     tf_seq.layers[11], pt_branch.conv4)
    load_batchnorm(tf_seq.layers[12], pt_branch.bn4)
    load_dense(    tf_seq.layers[15], pt_branch.fc1)
    load_dense(    tf_seq.layers[17], pt_branch.fc2)
    load_dense(    tf_seq.layers[19], pt_branch.fc3)

In [ ]:
# ==========================================================
# COMPARE OUTPUTS
# ==========================================================
def compare_outputs(tf_model, pt_model, mfcc_h, mfcc_w, mslfb_h, mslfb_w, device):
    x1 = np.random.randn(1, mfcc_h,  mfcc_w,  1).astype(np.float32)
    x2 = np.random.randn(1, mslfb_h, mslfb_w, 1).astype(np.float32)

    tf_out = tf_model([x1, x2], training=False).numpy()

    pt_model.eval()
    with torch.no_grad():
        pt_out = pt_model(
            torch.tensor(x1).to(device),
            torch.tensor(x2).to(device)
        ).cpu().numpy()

    print("Max absolute difference:", np.max(np.abs(tf_out - pt_out)))
    print("TF  output:", tf_out)
    print("PT  output:", pt_out)


# ==========================================================
# DEBUG LAYER BY LAYER — fixed for Conv(relu) → BN → Dropout
# ==========================================================
def debug_layer_by_layer_fixed(pt_model, seq2, mfcc_h, mfcc_w, device):
    x1 = np.random.randn(1, mfcc_h, mfcc_w, 1).astype(np.float32)

    # TF step by step
    t_tf = tf.constant(x1)
    tf_intermediates = {}
    for i, layer in enumerate(seq2.layers):
        t_tf = layer(t_tf, training=False)
        tf_intermediates[i] = t_tf.numpy()
        print(f"TF layer [{i}] {layer.name}: {t_tf.shape}")
    print()

    pt_model.eval()
    with torch.no_grad():
        t = torch.tensor(x1).to(device).permute(0, 3, 1, 2)

        # Conv+ReLU → TF[0]
        t = F.relu(pt_model.branch_mfcc.conv1(t))
        diff = np.max(np.abs(tf_intermediates[0].transpose(0,3,1,2) - t.cpu().numpy()))
        print(f"After conv1+relu  max diff: {diff:.6f}  PT: {tuple(t.shape)}")

        # BN → TF[1]
        t = pt_model.branch_mfcc.bn1(t)
        diff = np.max(np.abs(tf_intermediates[1].transpose(0,3,1,2) - t.cpu().numpy()))
        print(f"After bn1         max diff: {diff:.6f}  PT: {tuple(t.shape)}")
        # TF[2] = Dropout (no-op) — skip

        # Conv+ReLU → TF[3]
        t = F.relu(pt_model.branch_mfcc.conv2(t))
        diff = np.max(np.abs(tf_intermediates[3].transpose(0,3,1,2) - t.cpu().numpy()))
        print(f"After conv2+relu  max diff: {diff:.6f}  PT: {tuple(t.shape)}")

        # BN → TF[4]
        t = pt_model.branch_mfcc.bn2(t)
        diff = np.max(np.abs(tf_intermediates[4].transpose(0,3,1,2) - t.cpu().numpy()))
        print(f"After bn2         max diff: {diff:.6f}  PT: {tuple(t.shape)}")

        # MaxPool → TF[5]
        t = F.max_pool2d(t, 2, 2)
        diff = np.max(np.abs(tf_intermediates[5].transpose(0,3,1,2) - t.cpu().numpy()))
        print(f"After pool1       max diff: {diff:.6f}  PT: {tuple(t.shape)}")
        # TF[6] = Dropout (no-op) — skip

        # Conv+ReLU → TF[7]
        t = tf_same_pad(t)
        t = F.relu(pt_model.branch_mfcc.conv3(t))
        diff = np.max(np.abs(tf_intermediates[7].transpose(0,3,1,2) - t.cpu().numpy()))
        print(f"After conv3+relu  max diff: {diff:.6f}  PT: {tuple(t.shape)}")

        # BN → TF[8]
        t = pt_model.branch_mfcc.bn3(t)
        diff = np.max(np.abs(tf_intermediates[8].transpose(0,3,1,2) - t.cpu().numpy()))
        print(f"After bn3         max diff: {diff:.6f}  PT: {tuple(t.shape)}")

        # MaxPool → TF[9]
        t = F.max_pool2d(t, 2, 2)
        diff = np.max(np.abs(tf_intermediates[9].transpose(0,3,1,2) - t.cpu().numpy()))
        print(f"After pool2       max diff: {diff:.6f}  PT: {tuple(t.shape)}")
        # TF[10] = Dropout (no-op) — skip

        # Conv+ReLU → TF[11]
        t = tf_same_pad(t)
        t = F.relu(pt_model.branch_mfcc.conv4(t))
        diff = np.max(np.abs(tf_intermediates[11].transpose(0,3,1,2) - t.cpu().numpy()))
        print(f"After conv4+relu  max diff: {diff:.6f}  PT: {tuple(t.shape)}")

        # BN → TF[12]
        t = pt_model.branch_mfcc.bn4(t)
        diff = np.max(np.abs(tf_intermediates[12].transpose(0,3,1,2) - t.cpu().numpy()))
        print(f"After bn4         max diff: {diff:.6f}  PT: {tuple(t.shape)}")
        # TF[13] = Dropout (no-op) — skip

        # Flatten → TF[14]
        t = t.permute(0, 2, 3, 1)
        t = torch.flatten(t, 1)
        diff = np.max(np.abs(tf_intermediates[14] - t.cpu().numpy()))
        print(f"After flatten     max diff: {diff:.6f}  PT: {tuple(t.shape)}")

        # Dense+ReLU → TF[15]
        t = F.relu(pt_model.branch_mfcc.fc1(t))
        diff = np.max(np.abs(tf_intermediates[15] - t.cpu().numpy()))
        print(f"After fc1+relu    max diff: {diff:.6f}  PT: {tuple(t.shape)}")
        # TF[16] = Dropout (no-op) — skip

        # Dense+ReLU → TF[17]
        t = F.relu(pt_model.branch_mfcc.fc2(t))
        diff = np.max(np.abs(tf_intermediates[17] - t.cpu().numpy()))
        print(f"After fc2+relu    max diff: {diff:.6f}  PT: {tuple(t.shape)}")
        # TF[18] = Dropout (no-op) — skip

        # Dense+ReLU → TF[19]
        t = F.relu(pt_model.branch_mfcc.fc3(t))
        diff = np.max(np.abs(tf_intermediates[19] - t.cpu().numpy()))
        print(f"After fc3+relu    max diff: {diff:.6f}  PT: {tuple(t.shape)}")

In [ ]:
# ==========================================================
# LOAD TF MODEL + TRANSFER + CALL BOTH FUNCTIONS
# ==========================================================
file_date  = "20260320_064356"
model_path = os.path.join(main_directory, f"results/ensemble_mfcc_mslfb_resumed_{file_date}.keras")
tf_model   = tf.keras.models.load_model(model_path)

mfcc_h,  mfcc_w  = tf_model.input[0].shape[1], tf_model.input[0].shape[2]
mslfb_h, mslfb_w = tf_model.input[1].shape[1], tf_model.input[1].shape[2]
print(f"TF model loaded ✓  MFCC: ({mfcc_h},{mfcc_w})  MSLFB: ({mslfb_h},{mslfb_w})")

seq2 = tf_model.get_layer("sequential")
seq3 = tf_model.get_layer("sequential_1")

fresh_pt = DualBranchEnsemble().to(device)
fresh_pt.eval()
with torch.no_grad():
    _ = fresh_pt.branch_mfcc( torch.randn(1, mfcc_h,  mfcc_w,  1).to(device))
    _ = fresh_pt.branch_mslfb(torch.randn(1, mslfb_h, mslfb_w, 1).to(device))

transfer_branch(seq2, fresh_pt.branch_mfcc)
transfer_branch(seq3, fresh_pt.branch_mslfb)
load_dense(tf_model.get_layer("dense_6"), fresh_pt.classifier)
print("Weights transferred ✓")

compare_outputs(tf_model, fresh_pt, mfcc_h, mfcc_w, mslfb_h, mslfb_w, device)
debug_layer_by_layer_fixed(fresh_pt, seq2, mfcc_h, mfcc_w, device)

### ─────────────────────────────────────────────────────
### 🔬 10.5.2 BRECQ METHODOLOGY
### ─────────────────────────────────────────────────────

In [ ]:
from sklearn.metrics import precision_score, recall_score

# ==========================================================
# EVALUATE FUNCTION
# ==========================================================
def evaluate_quant_model(model, dataset):
    model.eval()
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for (inputs, labels) in dataset:
            x1 = torch.from_numpy(inputs[0].numpy()).float().to(device)
            x2 = torch.from_numpy(inputs[1].numpy()).float().to(device)

            outputs = model((x1, x2))
            preds   = torch.argmax(outputs, dim=1).cpu().numpy()
            targets = torch.argmax(
                torch.from_numpy(labels.numpy()), dim=1
            ).numpy()

            all_preds.append(preds)
            all_labels.append(targets)

    all_preds  = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    accuracy  = (all_preds == all_labels).mean()
    precision = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    recall    = recall_score(   all_labels, all_preds, average='macro', zero_division=0)

    classes = np.unique(all_labels)

    # Balanced accuracy = mean per-class recall
    per_class_recall = [
        np.mean(all_preds[all_labels == c] == c)
        for c in classes
    ]
    balanced_acc = float(np.mean(per_class_recall))

    # Macro F1 = unweighted mean of per-class F1 scores
    per_class_f1 = []
    for c in classes:
        tp = np.sum((all_preds == c) & (all_labels == c))
        fp = np.sum((all_preds == c) & (all_labels != c))
        fn = np.sum((all_preds != c) & (all_labels == c))
        precision_c = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall_c    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1_c        = (2 * precision_c * recall_c / (precision_c + recall_c)
                       if (precision_c + recall_c) > 0 else 0.0)
        per_class_f1.append(f1_c)

    macro_f1 = float(np.mean(per_class_f1))

    return accuracy, precision, recall, balanced_acc, macro_f1


# ==========================================================
# CALIBRATION DATA
# ==========================================================
CALIBRATION_SAMPLES = 1024
calibration_ds = train_ds.take(CALIBRATION_SAMPLES // batch_size)

cali_x1_list = []
cali_x2_list = []
for (inputs, _) in calibration_ds:
    cali_x1_list.append(torch.from_numpy(inputs[0].numpy()).float())
    cali_x2_list.append(torch.from_numpy(inputs[1].numpy()).float())

cali_x1 = torch.cat(cali_x1_list, dim=0).cpu()
cali_x2 = torch.cat(cali_x2_list, dim=0).cpu()


# ==========================================================
# BRECQ HELPERS
# ==========================================================
def brecq_build_and_transfer(tf_model, device):
    """Build a fresh PyTorch model and transfer weights from Keras."""
    mfcc_h,  mfcc_w  = tf_model.input[0].shape[1], tf_model.input[0].shape[2]
    mslfb_h, mslfb_w = tf_model.input[1].shape[1], tf_model.input[1].shape[2]

    fresh_pt = DualBranchEnsemble().to(device)
    fresh_pt.eval()
    with torch.no_grad():
        _ = fresh_pt.branch_mfcc( torch.randn(1, mfcc_h,  mfcc_w,  1).to(device))
        _ = fresh_pt.branch_mslfb(torch.randn(1, mslfb_h, mslfb_w, 1).to(device))

    seq2 = tf_model.get_layer("sequential")
    seq3 = tf_model.get_layer("sequential_1")
    transfer_branch(seq2, fresh_pt.branch_mfcc)
    transfer_branch(seq3, fresh_pt.branch_mslfb)
    load_dense(tf_model.get_layer("dense_6"), fresh_pt.classifier)
    fresh_pt.eval()
    return fresh_pt


def brecq_build_quant_model(wrapped, n_bits, device):
    """Wrap model in QuantModel and fix activation quantizers."""
    weight_quant_params = dict(n_bits=n_bits, symmetric=False, channel_wise=True,  scale_method='max')
    act_quant_params    = dict(n_bits=n_bits, symmetric=False, channel_wise=False, scale_method='max')

    quant_model = QuantModel(wrapped, weight_quant_params, act_quant_params).to(device)
    quant_model.eval()

    for name, module in quant_model.named_modules():
        if isinstance(module, QuantModule):
            module.act_quantizer.delta      = torch.tensor(1.0).to(device)
            module.act_quantizer.zero_point = torch.tensor(0.0).to(device)
            module.act_quantizer.inited     = True
    print("  Activation quantizers set to identity ✓")
    return quant_model


def brecq_init_weight_scales(quant_model, cali_x1, cali_x2, device):
    """Initialize weight quantization scales via calibration forward pass."""
    print("  Initializing weight quantization scales...")
    with torch.no_grad():
        quant_model.set_quant_state(True, False)
        _ = quant_model((cali_x1.to(device), cali_x2.to(device)))
        quant_model.set_quant_state(False, False)

    print(f"\n  {'Layer':<45} {'delta_mean':>12}  {'zp_mean':>10}")
    print(f"  {'-'*70}")
    for name, module in quant_model.named_modules():
        if isinstance(module, QuantModule):
            delta      = module.weight_quantizer.delta
            zp         = module.weight_quantizer.zero_point
            delta_mean = delta.mean().item() if delta is not None else -1
            zp_mean    = zp.mean().item()    if zp    is not None else -1
            flag = " ← SUSPICIOUS" if delta_mean < 1e-6 or delta_mean > 10 else ""
            print(f"  {name:<45} {delta_mean:>12.8f}  {zp_mean:>10.4f}{flag}")


def brecq_reconstruct_layers(quant_model, cali_x1, cali_x2, device):
    """Layer-by-layer reconstruction using BRECQ."""
    print("  Layer-by-layer reconstruction...")
    for name, module in quant_model.named_modules():
        if isinstance(module, QuantModule) and not module.ignore_reconstruction:
            print(f"    layer: {name}")
            layer_reconstruction(
                quant_model, module,
                (cali_x1.to(device), cali_x2.to(device)),
                batch_size=32, iters=1000, weight=0.001,
                opt_mode='mse', asym=False, include_act_func=True,
                b_range=(20, 2), warmup=0.2, act_quant=False,
                lr=4e-5, p=2.0, multi_gpu=False
            )


def brecq_plot_weight_distributions(quant_model, n_bits):
    records = []
    for name, module in quant_model.named_modules():
        if isinstance(module, QuantModule):
            if not (hasattr(module, 'org_weight') and module.org_weight is not None):
                continue

            fp_w = module.org_weight.detach()

            with torch.no_grad():
                q_w = module.weight_quantizer(fp_w)

            records.append((
                name,
                fp_w.cpu().numpy(),
                q_w.cpu().numpy(),
            ))

    print(f"  Layers found: {len(records)}")
    if len(records) == 0:
        print("  WARNING: no layers found — skipping distribution plot.")
        return

    n_plots   = len(records)
    n_cols    = 3
    n_rows    = math.ceil(n_plots / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 3.5))
    axes      = axes.flatten()

    for i, (name, fp_w, q_w) in enumerate(records):
        ax      = axes[i]
        fp_flat = fp_w.flatten()
        q_flat  = q_w.flatten()

        w_min     = min(fp_flat.min(), q_flat.min())
        w_max     = max(fp_flat.max(), q_flat.max())
        bin_edges = np.linspace(w_min, w_max, 60)

        ax.hist(fp_flat, bins=bin_edges, color='steelblue', alpha=0.6,
                edgecolor='none', label='FP32')
        ax.hist(q_flat,  bins=bin_edges, color='tomato',    alpha=0.6,
                edgecolor='none', label='Quantized')

        n_levels = len(np.unique(q_flat))
        ax.annotate(
            f"FP32  n={len(fp_flat)}\nQuant levels: {n_levels}",
            xy=(0.97, 0.95), xycoords='axes fraction',
            fontsize=6.5, ha='right', va='top',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='gray', alpha=0.85)
        )
        ax.set_title(name, fontsize=7.5, pad=4)
        ax.set_xlabel("Weight value", fontsize=7)
        ax.set_ylabel("Count",        fontsize=7)
        ax.tick_params(labelsize=6)
        ax.legend(fontsize=6)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle(
        f"FP32 vs BRECQ {n_bits}-bit Weight Distributions",
        fontsize=13, y=1.01
    )
    plt.tight_layout()
    plt.show()


# ==========================================================
# BRECQ PIPELINE
# ==========================================================
def run_brecq(n_bits):
    print(f"\n{'='*50}")
    print(f"  BRECQ {n_bits}-bit (layer-by-layer)")
    print(f"{'='*50}")

    file_date  = "20260320_064356"
    model_path = os.path.join(main_directory, f"results/ensemble_mfcc_mslfb_resumed_{file_date}.keras")
    tf_model   = tf.keras.models.load_model(model_path)

    # Build and transfer weights
    fresh_pt = brecq_build_and_transfer(tf_model, device)
    del tf_model

    # FP32 sanity check — now unpacks 5 values
    acc_fp32, _, _, _, _ = evaluate_quant_model(TupleInputModel(fresh_pt).to(device), test_ds)
    print(f"  FP32 accuracy: {acc_fp32:.4f}  (should be ~0.8723)")

    # Build quant model
    wrapped     = TupleInputModel(fresh_pt).to(device)
    quant_model = brecq_build_quant_model(wrapped, n_bits, device)

    # Init weight scales
    brecq_init_weight_scales(quant_model, cali_x1, cali_x2, device)

    # Sanity check after scale init — now unpacks 5 values
    quant_model.set_quant_state(True, False)
    quant_model.eval()
    acc_init, _, _, _, _ = evaluate_quant_model(quant_model, test_ds)
    print(f"\n  Accuracy after scale init: {acc_init:.4f}")
    quant_model.set_quant_state(False, False)

    # Layer-by-layer reconstruction
    brecq_reconstruct_layers(quant_model, cali_x1, cali_x2, device)

    # Final evaluation
    quant_model.set_quant_state(True, False)
    quant_model.eval()
    accuracy, precision, recall, balanced_acc, macro_f1 = evaluate_quant_model(
        quant_model, test_ds)

    print(f"\n  BRECQ {n_bits}-bit Results")
    print(f"  {'='*45}")
    print(f"  {'Metric':<22} {'Value':>10}")
    print(f"  {'-'*35}")
    print(f"  {'FP32 Accuracy':<22} {acc_fp32:>10.4f}")
    print(f"  {'After scale init':<22} {acc_init:>10.4f}")
    print(f"  {'After reconstruct':<22} {accuracy:>10.4f}")
    print(f"  {'Balanced Accuracy':<22} {balanced_acc:>10.4f}")
    print(f"  {'Precision':<22} {precision:>10.4f}")
    print(f"  {'Recall':<22} {recall:>10.4f}")
    print(f"  {'Macro F1':<22} {macro_f1:>10.4f}")
    print(f"  {'='*45}")

    # Weight distribution comparison
    brecq_plot_weight_distributions(quant_model, n_bits)

    return accuracy, precision, recall, balanced_acc, macro_f1

In [ ]:
# ==========================================================
# RUN
# ==========================================================
for n_bits in [8, 4, 2]:
    run_brecq(n_bits)

## ─────────────────────────────────────────────────────
## 🔬 10.6 LAPQ
## ─────────────────────────────────────────────────────

In [ ]:
import os
import tensorflow as tf
import numpy as np
from scipy.optimize import minimize

# ---------------------------------------------------------
# UNIFORM SYMMETRIC QUANTIZER (paper equation 1-2)
# ---------------------------------------------------------
def uniform_symmetric_quant_tf(w, bits, clip):
    qmax  = 2 ** (bits - 1) - 1
    scale = clip / qmax
    x     = tf.clip_by_value(w, -clip, clip)
    return tf.round(x / scale) * scale


# ---------------------------------------------------------
# LP NORM OF QUANTIZATION ERROR (paper Phase 1)
# ---------------------------------------------------------
def lp_quant_error(w_fp, bits, clip, p):
    w_q   = uniform_symmetric_quant_tf(
                tf.constant(w_fp, tf.float32), bits, clip)
    error = tf.abs(tf.constant(w_fp, tf.float32) - w_q)
    return tf.reduce_mean(tf.pow(error + 1e-10, p)).numpy()


# ---------------------------------------------------------
# FIND Δp FOR ONE LAYER (Phase 1, per layer per p)
# ---------------------------------------------------------
def find_delta_p(w_fp, bits, p, n_candidates=10):
    w_max      = np.max(np.abs(w_fp)) + 1e-12
    candidates = np.linspace(0.1 * w_max, w_max, n_candidates)
    errors     = [lp_quant_error(w_fp, bits, c, p) for c in candidates]
    return candidates[np.argmin(errors)]


# ---------------------------------------------------------
# COLLECT WEIGHT LAYERS
# ---------------------------------------------------------
def collect_weight_layers(model):
    layers = []
    def _collect(m):
        for layer in m.layers:
            if isinstance(layer, (tf.keras.layers.Conv2D,
                                  tf.keras.layers.Dense)):
                if layer.kernel is not None:
                    layers.append(layer)
            if hasattr(layer, 'layers'):
                _collect(layer)
    _collect(model)
    return list(dict.fromkeys(layers))


# ---------------------------------------------------------
# FAST LOSS EVALUATION
# ---------------------------------------------------------
loss_fn = tf.keras.losses.CategoricalCrossentropy()

@tf.function
def _forward_loss(model, cal_x1_tf, cal_x2_tf, cal_y_tf):
    y_pred = model((cal_x1_tf, cal_x2_tf), training=False)
    return loss_fn(cal_y_tf, y_pred)

def evaluate_loss_fast(model, cal_x1_tf, cal_x2_tf, cal_y_tf):
    return _forward_loss(model, cal_x1_tf, cal_x2_tf, cal_y_tf).numpy()


# ---------------------------------------------------------
# APPLY + EVALUATE THEN RESTORE
# ---------------------------------------------------------
def apply_and_evaluate(model, layers, originals, clips, bits,
                        cal_x1_tf, cal_x2_tf, cal_y_tf):
    for layer, w in zip(layers, originals):
        layer.kernel.assign(w)
    for layer, clip in zip(layers, clips):
        layer.kernel.assign(
            uniform_symmetric_quant_tf(layer.kernel, bits, float(clip))
        )
    loss = evaluate_loss_fast(model, cal_x1_tf, cal_x2_tf, cal_y_tf)
    for layer, w in zip(layers, originals):
        layer.kernel.assign(w)
    return loss


# ---------------------------------------------------------
# PHASE 1 + 2: LAYERWISE Lp INIT WITH QUADRATIC APPROX
# ---------------------------------------------------------
def layerwise_lp_initialization(model, layers, originals,
                                 cal_x1_tf, cal_x2_tf, cal_y_tf, bits):
    p_values = [0.5, 1.0, 2.0, 4.0]

    # Phase 1: find Δp for every layer and every p value
    print("  Phase 1: computing per-layer Δp for each p...")
    delta_ps = []
    for layer in layers:
        w_fp = layer.kernel.numpy()
        deltas_for_layer = [find_delta_p(w_fp, bits, p) for p in p_values]
        delta_ps.append(deltas_for_layer)
        print(f"    {layer.name}: clips = "
              f"{[f'{d:.4f}' for d in deltas_for_layer]}")

    # Phase 2: evaluate task loss at each p
    print("  Phase 2: evaluating task loss at each p (forward passes)...")
    task_losses = []
    for j, p in enumerate(p_values):
        clips_for_p = [delta_ps[i][j] for i in range(len(layers))]
        loss = apply_and_evaluate(
            model, layers, originals, clips_for_p, bits,
            cal_x1_tf, cal_x2_tf, cal_y_tf
        )
        task_losses.append(loss)
        print(f"    p={p:.1f}  →  task loss = {loss:.6f}")

    # Fit quadratic and find optimal p*
    coeffs  = np.polyfit(p_values, task_losses, 2)
    a, b, _ = coeffs

    if a > 0:
        p_opt = -b / (2 * a)
        p_opt = np.clip(p_opt, min(p_values), max(p_values))
        print(f"  Quadratic minimum at p* = {p_opt:.4f}")
    else:
        p_opt = p_values[int(np.argmin(task_losses))]
        print(f"  No quadratic minimum — using best observed p = {p_opt:.4f}")

    # Interpolate Δp* for each layer at optimal p*
    init_clips = []
    for i, layer in enumerate(layers):
        clip_opt = float(np.interp(p_opt, p_values, delta_ps[i]))
        init_clips.append(clip_opt)
        print(f"  {layer.name}: init clip = {clip_opt:.6f}")

    return np.array(init_clips)


# ---------------------------------------------------------
# PHASE 3: JOINT SCIPY OPTIMIZATION
# ---------------------------------------------------------
def joint_optimization(model, layers, originals, init_clips,
                        cal_x1_tf, cal_x2_tf, cal_y_tf,
                        bits, max_iter=100):
    call_count = [0]
    best_loss  = [np.inf]
    no_improve = [0]
    PATIENCE   = 50   # stop if no improvement in 50 steps

    def objective(clips):
        call_count[0] += 1
        clips = np.abs(clips)
        loss  = apply_and_evaluate(
            model, layers, originals, clips, bits,
            cal_x1_tf, cal_x2_tf, cal_y_tf
        )
        if call_count[0] % 10 == 0:
            print(f"    Joint opt step {call_count[0]}: loss={loss:.6f}")

        # Early stopping
        if loss < best_loss[0] - 1e-4:
            best_loss[0]  = loss
            no_improve[0] = 0
        else:
            no_improve[0] += 1

        if no_improve[0] >= PATIENCE:
            print(f"    Early stop at step {call_count[0]} — no improvement for {PATIENCE} steps")
            raise StopIteration

        return loss

    try:
        result = minimize(
            objective,
            init_clips,
            method='Powell',
            options={
                'maxiter': max_iter,
                'ftol':    1e-3,    # looser — Powell with 15 dims doesn't need 1e-4
                'xtol':    1e-3,    # also loosen parameter tolerance
                'disp':    True,
            }
        )
        opt_clips = np.abs(result.x)
    except StopIteration:
        # Return best clips seen so far — Powell doesn't expose current x on interrupt
        # so we re-apply init_clips as fallback (still better than running forever)
        print("  Using best clips found before early stop.")
        opt_clips = np.abs(init_clips)

    return opt_clips


# ---------------------------------------------------------
# FULL LAPQ PIPELINE
# ---------------------------------------------------------
def lapq_quantize_model(model, cal_x1_tf, cal_x2_tf, cal_y_tf, bits=4):
    layers    = collect_weight_layers(model)
    originals = [l.kernel.numpy().copy() for l in layers]

    print(f"Found {len(layers)} quantizable layers")

    print("\nPhase 1+2: Layerwise Lp initialization with task-loss quadratic fit...")
    init_clips = layerwise_lp_initialization(
        model, layers, originals,
        cal_x1_tf, cal_x2_tf, cal_y_tf, bits
    )

    print("\nPhase 3: Joint Powell optimization...")
    opt_clips = joint_optimization(
        model, layers, originals, init_clips,
        cal_x1_tf, cal_x2_tf, cal_y_tf,
        bits, max_iter=100
    )

    print("\nApplying final optimized quantization...")
    for layer, w in zip(layers, originals):
        layer.kernel.assign(w)
    for layer, clip in zip(layers, opt_clips):
        layer.kernel.assign(
            uniform_symmetric_quant_tf(layer.kernel, bits, float(clip))
        )

    print("LAPQ complete.")


# ---------------------------------------------------------
# PRE-LOAD CALIBRATION DATA ONCE
# ---------------------------------------------------------
file_date  = "20260320_064356"
model_path = os.path.join(main_directory, f"results/ensemble_mfcc_mslfb_resumed_{file_date}.keras")

cal_ds = train_ds.unbatch().batch(32).take(512 // 32)

cal_x1_list, cal_x2_list, cal_y_list = [], [], []
for (x1_batch, x2_batch), y_batch in cal_ds:
    cal_x1_list.append(x1_batch.numpy())
    cal_x2_list.append(x2_batch.numpy())
    cal_y_list.append(y_batch.numpy())

cal_x1_tf = tf.constant(np.concatenate(cal_x1_list), dtype=tf.float32)
cal_x2_tf = tf.constant(np.concatenate(cal_x2_list), dtype=tf.float32)
cal_y_tf  = tf.constant(np.concatenate(cal_y_list),  dtype=tf.float32)

print(f"Calibration data cached: {cal_x1_tf.shape[0]} samples ✓")

# ---------------------------------------------------------
# RUN
# ---------------------------------------------------------
for bits in [8, 4, 2]:
    model = load_model(model_path)
    lbl   = f"LAPQ {bits}-bit (per-layer)"
    lapq_quantize_model(model, cal_x1_tf, cal_x2_tf, cal_y_tf, bits=bits)
    compile_and_evaluate(model, test_ds, label=lbl)
    plot_weight_distribution_comparison(fp32_model, model, label=lbl)

In [ ]:
import os
import tensorflow as tf
import numpy as np
from scipy.optimize import minimize


# ---------------------------------------------------------
# UNIFORM SYMMETRIC QUANTIZER — per-channel clips
# clips shape: (out_ch,)  weights shape: (..., out_ch)
# ---------------------------------------------------------
def uniform_symmetric_quant_perchannel_tf(w, bits, clips):
    """
    w     : tf.Tensor (..., out_ch)
    clips : tf.Tensor (out_ch,)
    """
    qmax   = float(2 ** (bits - 1) - 1)
    clips  = tf.cast(clips, tf.float32)
    scales = clips / qmax                        # (out_ch,)
    w_clip = tf.clip_by_value(w, -clips, clips)
    return tf.round(w_clip / scales) * scales


# ---------------------------------------------------------
# LP ERROR FOR A SINGLE CHANNEL VECTOR
# ---------------------------------------------------------
def lp_quant_error_channel(w_ch, bits, clip, p):
    """w_ch: 1-D numpy array"""
    qmax   = 2 ** (bits - 1) - 1
    scale  = clip / qmax
    w_tf   = tf.constant(w_ch, tf.float32)
    w_clip = tf.clip_by_value(w_tf, -clip, clip)
    w_q    = tf.round(w_clip / scale) * scale
    error  = tf.abs(w_tf - w_q)
    return tf.reduce_mean(tf.pow(error + 1e-10, p)).numpy()


# ---------------------------------------------------------
# FIND Δp FOR ONE CHANNEL
# ---------------------------------------------------------
def find_delta_p_channel(w_ch, bits, p, n_candidates=10):
    w_max      = np.max(np.abs(w_ch)) + 1e-12
    candidates = np.linspace(0.1 * w_max, w_max, n_candidates)
    errors     = [lp_quant_error_channel(w_ch, bits, c, p) for c in candidates]
    return candidates[np.argmin(errors)]


# ---------------------------------------------------------
# COLLECT WEIGHT LAYERS
# ---------------------------------------------------------
def collect_weight_layers(model):
    layers = []
    def _collect(m):
        for layer in m.layers:
            if isinstance(layer, (tf.keras.layers.Conv2D,
                                  tf.keras.layers.Dense)):
                if layer.kernel is not None:
                    layers.append(layer)
            if hasattr(layer, 'layers'):
                _collect(layer)
    _collect(model)
    return list(dict.fromkeys(layers))


# ---------------------------------------------------------
# FAST LOSS EVALUATION
# ---------------------------------------------------------
loss_fn = tf.keras.losses.CategoricalCrossentropy()

@tf.function
def _forward_loss(model, cal_x1_tf, cal_x2_tf, cal_y_tf):
    y_pred = model((cal_x1_tf, cal_x2_tf), training=False)
    return loss_fn(cal_y_tf, y_pred)

def evaluate_loss_fast(model, cal_x1_tf, cal_x2_tf, cal_y_tf):
    return _forward_loss(model, cal_x1_tf, cal_x2_tf, cal_y_tf).numpy()


# ---------------------------------------------------------
# APPLY PER-CHANNEL CLIPS + EVALUATE THEN RESTORE
# layer_clips: list of np.ndarray, one (out_ch,) per layer
# ---------------------------------------------------------
def apply_and_evaluate_perchannel(model, layers, originals, layer_clips, bits,
                                   cal_x1_tf, cal_x2_tf, cal_y_tf):
    # Restore originals first
    for layer, w in zip(layers, originals):
        layer.kernel.assign(w)

    # Apply per-channel quantization
    for layer, clips in zip(layers, layer_clips):
        w      = layer.kernel          # (..., out_ch)
        clips_t = tf.constant(clips, dtype=tf.float32)
        layer.kernel.assign(
            uniform_symmetric_quant_perchannel_tf(w, bits, clips_t)
        )

    loss = evaluate_loss_fast(model, cal_x1_tf, cal_x2_tf, cal_y_tf)

    # Restore
    for layer, w in zip(layers, originals):
        layer.kernel.assign(w)

    return loss


# ---------------------------------------------------------
# PHASE 1+2 — PER-CHANNEL Lp INIT WITH QUADRATIC APPROX
# ---------------------------------------------------------
def layerwise_lp_initialization_perchannel(model, layers, originals,
                                            cal_x1_tf, cal_x2_tf, cal_y_tf,
                                            bits):
    p_values = [0.5, 1.0, 2.0, 4.0]

    # Phase 1: per-channel Δp for each layer and each p
    print("  Phase 1: computing per-channel Δp for each p...")
    # delta_ps[layer_idx][p_idx] = np.ndarray (out_ch,)
    delta_ps = []
    for layer in layers:
        w_fp   = layer.kernel.numpy()          # (..., out_ch)
        out_ch = w_fp.shape[-1]
        W_r    = w_fp.reshape(-1, out_ch)      # (fan_in, out_ch)

        layer_deltas = []
        for p in p_values:
            ch_clips = np.array([
                find_delta_p_channel(W_r[:, c], bits, p)
                for c in range(out_ch)
            ], dtype=np.float32)
            layer_deltas.append(ch_clips)

        delta_ps.append(layer_deltas)
        means = [f"{d.mean():.4f}" for d in layer_deltas]
        print(f"    {layer.name} ({out_ch} ch): mean clips per p = {means}")

    # Phase 2: evaluate task loss for each p using its per-channel clips
    print("  Phase 2: evaluating task loss at each p (forward passes)...")
    task_losses = []
    for j, p in enumerate(p_values):
        clips_for_p = [delta_ps[i][j] for i in range(len(layers))]
        loss = apply_and_evaluate_perchannel(
            model, layers, originals, clips_for_p, bits,
            cal_x1_tf, cal_x2_tf, cal_y_tf
        )
        task_losses.append(loss)
        print(f"    p={p:.1f}  →  task loss = {loss:.6f}")

    # Fit quadratic and find optimal p*
    coeffs  = np.polyfit(p_values, task_losses, 2)
    a, b, _ = coeffs

    if a > 0:
        p_opt = float(np.clip(-b / (2 * a), min(p_values), max(p_values)))
        print(f"  Quadratic minimum at p* = {p_opt:.4f}")
    else:
        p_opt = float(p_values[int(np.argmin(task_losses))])
        print(f"  No quadratic minimum — using best observed p = {p_opt:.4f}")

    # Interpolate per-channel clips at p* for each layer
    init_clips = []
    for i, layer in enumerate(layers):
        out_ch    = layers[i].kernel.shape[-1]
        ch_clips  = np.array([
            float(np.interp(p_opt, p_values, [delta_ps[i][j][c] for j in range(len(p_values))]))
            for c in range(out_ch)
        ], dtype=np.float32)
        init_clips.append(ch_clips)
        print(f"  {layer.name}: mean init clip = {ch_clips.mean():.6f}  "
              f"min={ch_clips.min():.6f}  max={ch_clips.max():.6f}")

    return init_clips


# ---------------------------------------------------------
# PHASE 3: JOINT SCIPY OPTIMIZATION — PER-CHANNEL
# The optimisation vector is the concatenation of all
# per-channel clips across all layers, then split back.
# ---------------------------------------------------------
def joint_optimization_perchannel(model, layers, originals, init_clips,
                                   cal_x1_tf, cal_x2_tf, cal_y_tf,
                                   bits, max_iter=100):

    # Build index boundaries to split flat vector back into per-layer arrays
    sizes  = [c.shape[0] for c in init_clips]
    splits = np.cumsum([0] + sizes)
    x0     = np.concatenate(init_clips)

    call_count = [0]
    best_loss  = [np.inf]
    best_x     = [x0.copy()]
    no_improve = [0]
    PATIENCE   = 50

    def objective(x):
        call_count[0] += 1
        x       = np.abs(x)
        # Split flat vector back to per-layer channel clips
        layer_clips = [
            x[splits[i]:splits[i+1]].astype(np.float32)
            for i in range(len(layers))
        ]
        loss = apply_and_evaluate_perchannel(
            model, layers, originals, layer_clips, bits,
            cal_x1_tf, cal_x2_tf, cal_y_tf
        )
        if call_count[0] % 10 == 0:
            print(f"    Joint opt step {call_count[0]}: loss={loss:.6f}")

        if loss < best_loss[0] - 1e-4:
            best_loss[0]  = loss
            best_x[0]     = x.copy()
            no_improve[0] = 0
        else:
            no_improve[0] += 1

        if no_improve[0] >= PATIENCE:
            print(f"    Early stop at step {call_count[0]}")
            raise StopIteration

        return loss

    try:
        result = minimize(
            objective, x0,
            method  = 'Powell',
            options = {'maxiter': max_iter, 'ftol': 1e-3, 'xtol': 1e-3, 'disp': True},
        )
        opt_x = np.abs(result.x)
    except StopIteration:
        print("  Using best clips found before early stop.")
        opt_x = np.abs(best_x[0])

    opt_clips = [
        opt_x[splits[i]:splits[i+1]].astype(np.float32)
        for i in range(len(layers))
    ]
    return opt_clips


# ---------------------------------------------------------
# FULL LAPQ PER-CHANNEL PIPELINE
# ---------------------------------------------------------
def lapq_quantize_model_perchannel(model, cal_x1_tf, cal_x2_tf, cal_y_tf, bits=4):
    layers    = collect_weight_layers(model)
    originals = [l.kernel.numpy().copy() for l in layers]

    total_ch = sum(l.kernel.shape[-1] for l in layers)
    print(f"Found {len(layers)} quantizable layers  |  {total_ch} total channels")

    print("\nPhase 1+2: Per-channel Lp initialization with quadratic fit...")
    init_clips = layerwise_lp_initialization_perchannel(
        model, layers, originals,
        cal_x1_tf, cal_x2_tf, cal_y_tf, bits
    )

    print(f"\nPhase 3: Joint Powell optimization over {total_ch}-dim clip vector...")
    opt_clips = joint_optimization_perchannel(
        model, layers, originals, init_clips,
        cal_x1_tf, cal_x2_tf, cal_y_tf,
        bits, max_iter=100
    )

    print("\nApplying final optimized per-channel quantization...")
    for layer, w in zip(layers, originals):
        layer.kernel.assign(w)
    for layer, clips in zip(layers, opt_clips):
        clips_t = tf.constant(clips, dtype=tf.float32)
        layer.kernel.assign(
            uniform_symmetric_quant_perchannel_tf(layer.kernel, bits, clips_t)
        )
        print(f"  {layer.name:<26} {layer.kernel.shape[-1]:>4} ch | "
              f"clip mean={clips.mean():.4f}  min={clips.min():.4f}  max={clips.max():.4f}")

    print("LAPQ per-channel complete.")


# ---------------------------------------------------------
# CALIBRATION DATA  (same as before)
# ---------------------------------------------------------
file_date  = "20260320_064356"
model_path = os.path.join(main_directory,
                          f"results/ensemble_mfcc_mslfb_resumed_{file_date}.keras")

cal_ds = train_ds.unbatch().batch(32).take(512 // 32)

cal_x1_list, cal_x2_list, cal_y_list = [], [], []
for (x1_batch, x2_batch), y_batch in cal_ds:
    cal_x1_list.append(x1_batch.numpy())
    cal_x2_list.append(x2_batch.numpy())
    cal_y_list.append(y_batch.numpy())

cal_x1_tf = tf.constant(np.concatenate(cal_x1_list), dtype=tf.float32)
cal_x2_tf = tf.constant(np.concatenate(cal_x2_list), dtype=tf.float32)
cal_y_tf  = tf.constant(np.concatenate(cal_y_list),  dtype=tf.float32)

print(f"Calibration data cached: {cal_x1_tf.shape[0]} samples ✓")

# ---------------------------------------------------------
# RUN
# ---------------------------------------------------------
for bits in [8, 4, 2]:
    model = load_model(model_path)
    lbl   = f"LAPQ {bits}-bit (per-channel)"
    lapq_quantize_model_perchannel(model, cal_x1_tf, cal_x2_tf, cal_y_tf, bits=bits)
    compile_and_evaluate(model, test_ds, label=lbl)
    plot_weight_distribution_comparison(fp32_model, model, label=lbl)

## ─────────────────────────────────────────────────────
## 🔬 10.7 QDROP
## ─────────────────────────────────────────────────────

In [ ]:
import copy

# ==========================================================
# DEVICE — CPU fallback if no GPU available
# ==========================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


# ==========================================================
# QDROP CONFIG — scales with bit-width
# ==========================================================
class QDropReconConfig:
    def __init__(self, n_bits=4):
        self.batch_size = 32
        self.b_range    = (20, 2)
        self.warm_up    = 0.2
        self.drop_prob  = 0.5
        self.scale_lr   = 4e-5
        self.keep_gpu   = True
        self.round_mode = 'learned_hard_sigmoid'

        # Scale iters and rounding weight with bit-width
        if n_bits <= 2:
            self.iters  = 5000
            self.weight = 0.1
        elif n_bits <= 4:
            self.iters  = 2000
            self.weight = 0.01
        else:
            self.iters  = 1000
            self.weight = 0.001


# ==========================================================
# CALIBRATION DATA
# ==========================================================
CALIBRATION_SAMPLES = 1024
calibration_ds = train_ds.take(CALIBRATION_SAMPLES // batch_size)

cali_x1_list = []
cali_x2_list = []
for (inputs, _) in calibration_ds:
    cali_x1_list.append(torch.from_numpy(inputs[0].numpy()).float())
    cali_x2_list.append(torch.from_numpy(inputs[1].numpy()).float())

# Keep on CPU — moved to device inside functions
cali_x1 = torch.cat(cali_x1_list, dim=0).cpu()
cali_x2 = torch.cat(cali_x2_list, dim=0).cpu()


# ==========================================================
# QDROP MODEL BUILDER
# ==========================================================
def qdrop_build_quant_model(fresh_pt, n_bits, device):
    import types
    from quantization.quantized_module import QuantizedLayer, QuantizedBlock

    w_qconfig           = types.SimpleNamespace()
    w_qconfig.bit       = n_bits
    w_qconfig.symmetric = n_bits > 2   # asymmetric at 2-bit, symmetric otherwise
    w_qconfig.ch_axis   = 0
    w_qconfig.observer  = 'MinMaxObserver'
    w_qconfig.quantizer = 'AdaRoundFakeQuantize'

    a_qconfig           = types.SimpleNamespace()
    a_qconfig.bit       = n_bits
    a_qconfig.symmetric = False
    a_qconfig.ch_axis   = -1
    a_qconfig.observer  = 'AvgMSEFastObserver'
    a_qconfig.quantizer = 'LSQFakeQuantize'

    def replace_module(module, w_qconfig, a_qconfig, qoutput=True):
        childs = list(module.named_children())
        st, ed = 0, len(childs)
        prev_quantmodule = None
        while st < ed:
            tmp_qoutput = qoutput if st == ed - 1 else True
            name, child = childs[st]
            if isinstance(child, (torch.nn.Conv2d, torch.nn.Linear)):
                setattr(module, name,
                        QuantizedLayer(child, None, w_qconfig, a_qconfig,
                                       qoutput=tmp_qoutput))
                prev_quantmodule = getattr(module, name)
            elif isinstance(child, (torch.nn.ReLU, torch.nn.ReLU6)):
                if prev_quantmodule is not None:
                    prev_quantmodule.activation = child
                    setattr(module, name, StraightThrough())
            else:
                replace_module(child, w_qconfig, a_qconfig, tmp_qoutput)
            st += 1

    model = TupleInputModel(fresh_pt).to(device)
    replace_module(model, w_qconfig, a_qconfig, qoutput=False)
    model.eval()
    return model


# ==========================================================
# QDROP RECONSTRUCTION
# ==========================================================
def qdrop_reconstruct(quant_model, fp_model, cali_x1, cali_x2, device, n_bits):
    """Layer-by-layer QDrop reconstruction."""
    print("  QDrop reconstruction...")

    config    = QDropReconConfig(n_bits=n_bits)
    cali_data = (cali_x1.to(device), cali_x2.to(device))

    enable_quantization(quant_model)

    def recon_model(module, fp_module):
        for name, child in module.named_children():
            if isinstance(child, (QuantizedLayer, QuantizedBlock)):
                print(f"    QDrop: {name}")
                fp_child = getattr(fp_module, name)
                reconstruction(
                    quant_model, fp_model,
                    child, fp_child,
                    cali_data, config
                )
            else:
                try:
                    recon_model(child, getattr(fp_module, name))
                except AttributeError:
                    pass

    recon_model(quant_model, fp_model)
    print("  Reconstruction complete ✓")


# ==========================================================
# QDROP CALIBRATION
# ==========================================================
def qdrop_calibrate(quant_model, cali_x1, cali_x2, device):
    cali_data = (cali_x1.to(device), cali_x2.to(device))

    print("  Calibrating activation quantizers...")
    with torch.no_grad():
        enable_calibration_woquantization(quant_model,
                                          quantizer_type='act_fake_quant')
        for _ in range(4):
            quant_model(cali_data)

        print("  Calibrating weight quantizers...")
        enable_calibration_woquantization(quant_model,
                                          quantizer_type='weight_fake_quant')
        quant_model(cali_data)

    print("  Calibration done ✓")


# ==========================================================
# PLOT
# ==========================================================
def brecq_plot_weight_distributions_qdrop(fp32_weights_dict, quant_model, n_bits):
    records = []
    for name, module in quant_model.named_modules():
        if isinstance(module, (torch.nn.Conv2d, torch.nn.Linear)):
            if name not in fp32_weights_dict:
                continue
            fp_w = fp32_weights_dict[name]
            q_w  = module.weight.detach().cpu().numpy()
            records.append((name, fp_w, q_w))

    if not records:
        print("  WARNING: no layers found for plot.")
        return

    n_plots   = len(records)
    n_cols    = 3
    n_rows    = math.ceil(n_plots / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 3.5))
    axes      = axes.flatten()

    for i, (name, fp_w, q_w) in enumerate(records):
        ax        = axes[i]
        fp_flat   = fp_w.flatten()
        q_flat    = q_w.flatten()
        w_min     = min(fp_flat.min(), q_flat.min())
        w_max     = max(fp_flat.max(), q_flat.max())
        bin_edges = np.linspace(w_min, w_max, 60)

        ax.hist(fp_flat, bins=bin_edges, color='steelblue',
                alpha=0.6, edgecolor='none', label='FP32')
        ax.hist(q_flat,  bins=bin_edges, color='tomato',
                alpha=0.6, edgecolor='none', label='Quantized')

        n_levels = len(np.unique(q_flat))
        ax.annotate(f"FP32 n={len(fp_flat)}\nQuant levels: {n_levels}",
                    xy=(0.97, 0.95), xycoords='axes fraction',
                    fontsize=6.5, ha='right', va='top',
                    bbox=dict(boxstyle='round,pad=0.3', fc='white',
                              ec='gray', alpha=0.85))
        ax.set_title(name, fontsize=7.5, pad=4)
        ax.set_xlabel("Weight value", fontsize=7)
        ax.set_ylabel("Count", fontsize=7)
        ax.tick_params(labelsize=6)
        ax.legend(fontsize=6)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle(f"FP32 vs QDrop {n_bits}-bit Weight Distributions",
                 fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()


# ==========================================================
# QDROP PIPELINE
# ==========================================================
def run_qdrop(n_bits):
    print(f"\n{'='*55}")
    print(f"  QDrop {n_bits}-bit  |  device={device}")
    print(f"{'='*55}")

    file_date  = "20260320_064356"
    model_path = os.path.join(main_directory,
                              f"results/ensemble_mfcc_mslfb_resumed_{file_date}.keras")
    tf_model   = tf.keras.models.load_model(model_path)

    fresh_pt = brecq_build_and_transfer(tf_model, device)
    del tf_model

    # FP32 sanity check
    acc_fp32, _, _, _, _ = evaluate_quant_model(
        TupleInputModel(fresh_pt).to(device), test_ds
    )
    print(f"  FP32 accuracy: {acc_fp32:.4f}  (should be ~0.8723)")

    # Deepcopy BEFORE wrapping — this is the true FP32 reference for reconstruction
    fresh_pt_copy = copy.deepcopy(fresh_pt)

    # Build quantized model from the original
    quant_model = qdrop_build_quant_model(fresh_pt, n_bits, device)
    quant_model = quant_model.to(device)

    # FP model — pure fp32, not a disabled quant graph
    fp_model = TupleInputModel(fresh_pt_copy).to(device)
    fp_model.eval()

    # Capture FP32 weights BEFORE reconstruction for the plot
    fp32_weights_dict = {}
    for name, module in quant_model.named_modules():
        if isinstance(module, (torch.nn.Conv2d, torch.nn.Linear)):
            fp32_weights_dict[name] = module.weight.detach().cpu().numpy().copy()

    # Calibrate
    qdrop_calibrate(quant_model, cali_x1, cali_x2, device)
    quant_model = quant_model.to(device)

    # Accuracy after calibration
    enable_quantization(quant_model)
    quant_model.eval()
    acc_init, _, _, _, _ = evaluate_quant_model(quant_model, test_ds)
    print(f"  Accuracy after calibration: {acc_init:.4f}")

    # QDrop reconstruction
    qdrop_reconstruct(quant_model, fp_model, cali_x1, cali_x2, device, n_bits=n_bits)

    # Final evaluation — all 5 metrics
    enable_quantization(quant_model)
    quant_model.eval()
    accuracy, precision, recall, balanced_acc, macro_f1 = evaluate_quant_model(
        quant_model, test_ds)

    print(f"\n  QDrop {n_bits}-bit Results")
    print(f"  {'='*50}")
    print(f"  {'Metric':<22} {'Value':>10}")
    print(f"  {'-'*35}")
    print(f"  {'FP32 Accuracy':<22} {acc_fp32:>10.4f}")
    print(f"  {'After calibration':<22} {acc_init:>10.4f}")
    print(f"  {'After QDrop':<22} {accuracy:>10.4f}")
    print(f"  {'Balanced Accuracy':<22} {balanced_acc:>10.4f}")
    print(f"  {'Precision':<22} {precision:>10.4f}")
    print(f"  {'Recall':<22} {recall:>10.4f}")
    print(f"  {'Macro F1':<22} {macro_f1:>10.4f}")
    print(f"  {'='*50}")

    brecq_plot_weight_distributions_qdrop(fp32_weights_dict, quant_model, n_bits)

    return accuracy, precision, recall, balanced_acc, macro_f1


In [ ]:
# ==========================================================
# RUN
# ==========================================================
qdrop_results = {}
for n_bits in [8, 4, 2]:
    acc, prec, rec, bal, f1 = run_qdrop(n_bits)
    qdrop_results[n_bits] = (acc, prec, rec, bal, f1)

# ── Summary ───────────────────────────────────────────────
print(f"\n{'='*75}")
print(f"  QDROP FINAL SUMMARY")
print(f"{'='*75}")
print(f"  {'Method':<20} {'Accuracy':>10}  {'Balanced':>10}  {'Precision':>10}  {'Recall':>10}  {'Macro F1':>10}")
print(f"  {'-'*72}")
for n_bits, (acc, prec, rec, bal, f1) in qdrop_results.items():
    print(f"  {'QDrop '+str(n_bits)+'-bit':<20} {acc:>10.4f}  {bal:>10.4f}  {prec:>10.4f}  {rec:>10.4f}  {f1:>10.4f}")

# ─────────────────────────────────────────────────────
# 🎙️ 11. LIVE AUDIO TESTING
# ─────────────────────────────────────────────────────

In [ ]:
import os
import random
import numpy as np
import tensorflow as tf
from IPython.display import Audio, display
from scipy.io import wavfile

# ---------------------------------------------------------
# LOAD TRAINED MODEL
# ---------------------------------------------------------
file_date  = "20260320_064356"
model_path = os.path.join(main_directory, f"results/ensemble_mfcc_mslfb_resumed_{file_date}.keras")
model = tf.keras.models.load_model(model_path)

# Use exact same label order as training — never hardcode
class_names = list(le.classes_)
print("Class names:", class_names)

# ---------------------------------------------------------
# PREPROCESS FUNCTION
# ---------------------------------------------------------
def preprocess_audio_custom(audio_path):
    sr, signal = wavfile.read(audio_path)
    signal = signal.astype(np.float32)

    if len(signal) < audio_length:
        signal = np.pad(signal, (0, audio_length - len(signal)))
    else:
        signal = signal[:audio_length]

    # No noise added — correct for real inference
    # Model was trained on noisy audio but we predict on clean recordings
    mfcc  = MFCC(signal)
    mslfb = MSLFB(signal)

    mfcc  = (mfcc  - np.mean(mfcc))  / (np.std(mfcc)  + 1e-8)
    mslfb = (mslfb - np.mean(mslfb)) / (np.std(mslfb) + 1e-8)

    mfcc  = np.expand_dims(mfcc,  axis=(0, -1))
    mslfb = np.expand_dims(mslfb, axis=(0, -1))
    return signal, sr, mfcc, mslfb

# ---------------------------------------------------------
# PREDICT FUNCTION
# ---------------------------------------------------------
def predict(audio_path):
    signal, sr, mfcc_input, mslfb_input = preprocess_audio_custom(audio_path)
    display(Audio(signal, rate=sr))

    preds      = model.predict([mfcc_input, mslfb_input], verbose=0)
    pred_idx   = np.argmax(preds)
    confidence = np.max(preds)


    print(f"Predicted : {class_names[pred_idx]}")
    print(f"Confidence: {confidence:.3f}")
    print("All scores:")

    for name, score in sorted(zip(class_names, preds[0]), key=lambda x: -x[1]):
        print(f"  {name:<10} {score:.3f}")

    return class_names[pred_idx], confidence

# ---------------------------------------------------------
# PREDICT A SPECIFIC FILE
# ---------------------------------------------------------
predict(os.path.join(main_directory, "yes/004ae714_nohash_0.wav"))

# ---------------------------------------------------------
# PREDICT A RANDOM SAMPLE FROM A COMMAND FOLDER
# ---------------------------------------------------------
def predict_random(command):
    folder     = os.path.join(main_directory, command)
    audio_path = random.choice([
        os.path.join(folder, f)
        for f in os.listdir(folder)
        if f.endswith(".wav")
    ])

    print(f"File: {audio_path}")
    return predict(audio_path)

predict_random("yes")
predict_random("no")
predict_random("stop")

# ---------------------------------------------------------
# BATCH EVALUATE A FOLDER
# Maps folder name to correct class label
# e.g. unknown words folders all map to 'unknown'
# ---------------------------------------------------------
def evaluate_folder(command, n_samples=20):
    folder = os.path.join(main_directory, command)
    files  = [f for f in os.listdir(folder) if f.endswith(".wav")]
    files  = random.sample(files, min(n_samples, len(files)))

    # Map folder name to the label the model uses
    known_commands = {'yes', 'no', 'up', 'down', 'left', 'right', 'on', 'off', 'stop', 'go'}
    expected_label = command if command in known_commands else 'unknown'

    correct = 0
    for f in files:
        pred, _ = predict(os.path.join(folder, f))
        if pred == expected_label:
            correct += 1

    print(f"\n{command} (expected '{expected_label}'): "
          f"{correct}/{len(files)} correct ({100*correct/len(files):.1f}%)")


#evaluate_folder("yes",  n_samples=10)
#evaluate_folder("no",   n_samples=10)
#evaluate_folder("go",   n_samples=10)
#evaluate_folder("down", n_samples=10)
#evaluate_folder("left",  n_samples=10)
#evaluate_folder("up",   n_samples=10)
#evaluate_folder("off",   n_samples=10)
#evaluate_folder("on",    n_samples=10)
#evaluate_folder("right",  n_samples=10)
#evaluate_folder("stop",   n_samples=10)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import math

# ─────────────────────────────────────────────────────────────────────────────
#  Capture activations during a single inference pass
# ─────────────────────────────────────────────────────────────────────────────

def capture_all_activations_inference(model, mfcc_input, mslfb_input):
    """
    Runs one forward pass and captures the output activation of every
    Conv2D and Dense layer in the model.

    Returns
    -------
    dict : { layer_name: np.ndarray (flattened activations) }
    """
    activations = {}

    def _hook_layer(layer):
        original_call = layer.call

        def hooked_call(inputs, *args, **kwargs):
            result = original_call(inputs, *args, **kwargs)
            activations[layer.name] = (
                result.numpy() if hasattr(result, 'numpy') else result
            )
            return result

        layer.call = hooked_call
        return original_call

    def _iter_all_layers(m):
        for layer in m.layers:
            yield layer
            if hasattr(layer, 'layers'):
                yield from _iter_all_layers(layer)

    # Patch all Conv2D and Dense layers
    originals = {}
    for layer in _iter_all_layers(model):
        if isinstance(layer, (tf.keras.layers.Conv2D, tf.keras.layers.Dense)):
            originals[layer.name] = _hook_layer(layer)

    # Single forward pass
    model([mfcc_input, mslfb_input], training=False)

    # Restore original calls
    for layer in _iter_all_layers(model):
        if layer.name in originals:
            layer.call = originals[layer.name]

    # Flatten each activation tensor for plotting
    return {
        name: act.flatten()
        for name, act in activations.items()
    }


# ─────────────────────────────────────────────────────────────────────────────
#  Plot activation distributions
# ─────────────────────────────────────────────────────────────────────────────

def plot_activation_distributions(activations_dict, bins=60,
                                   skew_threshold=0.5, kurt_threshold=1.0,
                                   label=""):
    """
    Plots the activation distribution for each captured layer.
    One subplot per layer, annotated with skew / kurt / signed status.
    """
    from scipy import stats as scipy_stats

    layer_names = list(activations_dict.keys())
    n_plots     = len(layer_names)
    n_cols      = 3
    n_rows      = math.ceil(n_plots / n_cols)

    fig, axes = plt.subplots(n_rows, n_cols,
                              figsize=(n_cols * 5, n_rows * 3.5))
    axes = np.array(axes).flatten()

    for i, layer_name in enumerate(layer_names):
        ax   = axes[i]
        acts = activations_dict[layer_name]

        ax.hist(acts, bins=bins, color='steelblue',
                edgecolor='none', alpha=0.85)

        # Stats
        mu       = acts.mean()
        sigma    = acts.std()
        skewness = float(scipy_stats.skew(acts))
        kurt     = float(scipy_stats.kurtosis(acts))
        is_signed = bool(acts.min() < 0)
        a_min     = float(acts.min())
        a_max     = float(acts.max())

        # Normal fit overlay
        if sigma > 0:
            x     = np.linspace(a_min, a_max, 300)
            pdf   = scipy_stats.norm.pdf(x, mu, sigma)
            scale = len(acts) * (a_max - a_min) / bins
            ax.plot(x, pdf * scale, color='tomato', linewidth=1.5,
                    label='Normal fit')

        normal = abs(skewness) < skew_threshold and abs(kurt) < kurt_threshold
        color  = 'green' if normal else 'red'

        annotation = (
            f"sk={skewness:.2f}  ku={kurt:.2f}\n"
            f"{'signed' if is_signed else 'unsigned'}\n"
            f"min={a_min:.4f}  max={a_max:.4f}\n"
            f"n={len(acts):,}"
        )
        ax.annotate(
            annotation,
            xy=(0.97, 0.95), xycoords='axes fraction',
            fontsize=7, ha='right', va='top', color=color,
            bbox=dict(boxstyle='round,pad=0.3', fc='white',
                      ec=color, alpha=0.85)
        )

        ax.set_title(layer_name, fontsize=8, pad=4)
        ax.set_xlabel("Activation value", fontsize=7)
        ax.set_ylabel("Count", fontsize=7)
        ax.tick_params(labelsize=6)
        ax.legend(fontsize=6)

    for j in range(n_plots, len(axes)):
        axes[j].set_visible(False)

    title = f"Activation Distributions — {label}" if label else "Activation Distributions"
    plt.suptitle(title, fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

    # ── Summary table ─────────────────────────────────────────────────────────
    print(f"\n{'Layer':<30} {'n':>8}  {'min':>9}  {'max':>9}  "
          f"{'mean':>9}  {'std':>9}  {'skew':>7}  {'kurt':>7}  {'signed':>7}")
    print("-" * 105)
    for layer_name, acts in activations_dict.items():
        sk = float(scipy_stats.skew(acts))
        ku = float(scipy_stats.kurtosis(acts))
        print(f"  {layer_name:<28} {len(acts):>8,}  {acts.min():>9.4f}  "
              f"{acts.max():>9.4f}  {acts.mean():>9.4f}  {acts.std():>9.4f}  "
              f"{sk:>7.2f}  {ku:>7.2f}  "
              f"{'yes' if acts.min() < 0 else 'no':>7}")


# ─────────────────────────────────────────────────────────────────────────────
#  RUN — single inference + plot
# ─────────────────────────────────────────────────────────────────────────────

# Pick one audio file to run inference on
audio_path = os.path.join(main_directory, "yes/004ae714_nohash_0.wav")

signal, sr, mfcc_input, mslfb_input = preprocess_audio_custom(audio_path)

# Show prediction
preds      = model.predict([mfcc_input, mslfb_input], verbose=0)
pred_idx   = np.argmax(preds)
confidence = np.max(preds)
print(f"File      : {audio_path}")
print(f"Predicted : {class_names[pred_idx]}  (confidence={confidence:.3f})")

# Capture activations during inference
activations = capture_all_activations_inference(model, mfcc_input, mslfb_input)

# Plot
plot_activation_distributions(
    activations,
    bins            = 60,
    skew_threshold  = 0.5,
    kurt_threshold  = 1.0,
    label           = f"'{class_names[pred_idx]}' — {os.path.basename(audio_path)}",
)

In [ ]:
def capture_and_compare_activations(
    fp32_model,
    quant_model,
    mfcc_input,
    mslfb_input,
    audio_label = "",
    bins        = 60,
):
    """
    Runs inference on both FP32 and quantized model for the same input,
    captures activations of all Conv2D and Dense layers, and plots
    FP32 vs Quantized side by side for each layer.
    Also prints a summary table with the KL divergence between distributions.
    """
    from scipy import stats as scipy_stats
    from scipy.special import kl_div

    def _capture(model):
        activations = {}

        def _iter_all_layers(m):
            for layer in m.layers:
                yield layer
                if hasattr(layer, 'layers'):
                    yield from _iter_all_layers(layer)

        originals = {}
        for layer in _iter_all_layers(model):
            if isinstance(layer, (tf.keras.layers.Conv2D,
                                  tf.keras.layers.Dense)):
                original_call = layer.call
                def make_hook(orig, name):
                    def hooked(inputs, *args, **kwargs):
                        result = orig(inputs, *args, **kwargs)
                        activations[name] = (
                            result.numpy() if hasattr(result, 'numpy')
                            else result
                        ).flatten()
                        return result
                    return hooked
                originals[layer.name] = original_call
                layer.call = make_hook(original_call, layer.name)

        model([mfcc_input, mslfb_input], training=False)

        for layer in _iter_all_layers(model):
            if layer.name in originals:
                layer.call = originals[layer.name]

        return activations

    print("  Capturing FP32 activations...")
    acts_fp32  = _capture(fp32_model)
    print("  Capturing quantized activations...")
    acts_quant = _capture(quant_model)

    # ── Align layers present in both models ───────────────────────────────────
    common_layers = [n for n in acts_fp32 if n in acts_quant]
    n_plots       = len(common_layers)
    n_cols        = 2   # FP32 left, Quant right — but we do them as pairs
    n_rows        = n_plots

    fig, axes = plt.subplots(
        n_rows, 2,
        figsize=(14, n_rows * 3.0),
        squeeze=False
    )

    kl_records = []

    for i, layer_name in enumerate(common_layers):
        fp_acts = acts_fp32[layer_name]
        q_acts  = acts_quant[layer_name]

        # Shared bin range
        a_min = min(fp_acts.min(), q_acts.min())
        a_max = max(fp_acts.max(), q_acts.max())
        edges = np.linspace(a_min, a_max, bins + 1)

        for j, (acts, color, model_label) in enumerate([
            (fp_acts,  'steelblue', 'FP32'),
            (q_acts,   'tomato',    'Quantized'),
        ]):
            ax = axes[i][j]
            ax.hist(acts, bins=edges, color=color,
                    edgecolor='none', alpha=0.85, label=model_label)

            sk = float(scipy_stats.skew(acts))
            ku = float(scipy_stats.kurtosis(acts))
            ax.annotate(
                f"sk={sk:.2f}  ku={ku:.2f}\n"
                f"min={acts.min():.4f}  max={acts.max():.4f}\n"
                f"mean={acts.mean():.4f}  std={acts.std():.4f}",
                xy=(0.97, 0.95), xycoords='axes fraction',
                fontsize=6.5, ha='right', va='top',
                bbox=dict(boxstyle='round,pad=0.25', fc='white',
                          ec=color, alpha=0.85)
            )
            ax.set_title(
                f"{layer_name} — {model_label}", fontsize=8, pad=3)
            ax.set_xlabel("Activation value", fontsize=7)
            ax.set_ylabel("Count", fontsize=7)
            ax.tick_params(labelsize=6)

        # ── KL divergence (symmetric) ──────────────────────────────────────
        # Histogram-based approximation using shared bins
        fp_hist,  _ = np.histogram(fp_acts, bins=edges, density=True)
        q_hist,   _ = np.histogram(q_acts,  bins=edges, density=True)
        eps         = 1e-10
        fp_hist     = fp_hist + eps
        q_hist      = q_hist  + eps
        fp_hist    /= fp_hist.sum()
        q_hist     /= q_hist.sum()
        kl_sym      = float(np.sum(kl_div(fp_hist, q_hist))
                            + np.sum(kl_div(q_hist, fp_hist))) / 2

        # MSE between activation vectors (same size since same input)
        mse = float(np.mean((fp_acts - q_acts) ** 2))
        rel_mse = mse / (np.mean(fp_acts ** 2) + 1e-10)

        kl_records.append({
            'layer':   layer_name,
            'kl_sym':  kl_sym,
            'mse':     mse,
            'rel_mse': rel_mse,
        })

    title = (f"FP32 vs Quantized Activations — {audio_label}"
             if audio_label else "FP32 vs Quantized Activations")
    plt.suptitle(title, fontsize=13, y=1.005)
    plt.tight_layout()
    plt.show()

    # ── Summary table ─────────────────────────────────────────────────────────
    print(f"\n{'Layer':<28} {'KL (sym)':>10}  {'MSE':>12}  {'Rel MSE':>10}  Quality")
    print("-" * 75)
    for rec in kl_records:
        quality = ("✓ good"   if rec['rel_mse'] < 0.01
                   else "⚠ medium" if rec['rel_mse'] < 0.05
                   else "✗ poor")
        print(f"  {rec['layer']:<26} {rec['kl_sym']:>10.5f}  "
              f"{rec['mse']:>12.6f}  {rec['rel_mse']:>10.6f}  {quality}")

    return kl_records

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  RUN — comparação de activações FP32 vs vários métodos de quantização
# ─────────────────────────────────────────────────────────────────────────────

audio_path = os.path.join(main_directory, "yes/004ae714_nohash_0.wav")
signal, sr, mfcc_input, mslfb_input = preprocess_audio_custom(audio_path)

print(f"File      : {audio_path}")
preds      = fp32_model.predict([mfcc_input, mslfb_input], verbose=0)
pred_idx   = np.argmax(preds)
confidence = np.max(preds)
print(f"Predicted : {class_names[pred_idx]}  (confidence={confidence:.3f})")

calib_ds = train_ds.unbatch().batch(32).take(1024 // 32)

# ── Sensitivity analysis — run once, reuse for all methods ───────────────────
df_channels, layer_records, df_act = channel_sensitivity_analysis_with_clipping(
    model_path               = model_path,
    calib_ds                 = calib_ds,
    candidate_bits           = (2, 3, 4, 8),
    clip_percentiles         = (0.95, 0.99, 0.999, 1.0),
    mse_threshold_percentile = 0.3,
    skewness_threshold       = 0.5,
    qdrop_prob               = 0.5,
    qdrop_seed               = 42,
    act_mse_threshold        = 0.01,
    act_clip_percentile      = 0.999,
)

# ─────────────────────────────────────────────────────────────────────────────
#  Method 1 — Uniform 8-bit per-channel  (simplest baseline)
# ─────────────────────────────────────────────────────────────────────────────
model_uniform8 = load_model(model_path)
uniform_quantize_model(model_uniform8, num_bits=8, label="Uniform 8-bit")

kl_uniform8 = capture_and_compare_activations(
    fp32_model  = fp32_model,
    quant_model = model_uniform8,
    mfcc_input  = mfcc_input,
    mslfb_input = mslfb_input,
    audio_label = "Uniform 8-bit — 'yes'",
)

# ─────────────────────────────────────────────────────────────────────────────
#  Method 2 — Uniform 4-bit per-channel  (aggressive baseline)
# ─────────────────────────────────────────────────────────────────────────────
model_uniform4 = load_model(model_path)
uniform_quantize_model(model_uniform4, num_bits=4, label="Uniform 4-bit")

kl_uniform4 = capture_and_compare_activations(
    fp32_model  = fp32_model,
    quant_model = model_uniform4,
    mfcc_input  = mfcc_input,
    mslfb_input = mslfb_input,
    audio_label = "Uniform 4-bit — 'yes'",
)

# ─────────────────────────────────────────────────────────────────────────────
#  Method 3 — Channel sensitivity + clipping + mixed bits (weights only)
# ─────────────────────────────────────────────────────────────────────────────
model_sensitivity = load_model(model_path)
model_sensitivity = quantize_model_with_rounding(
    model              = model_sensitivity,
    layer_records      = layer_records,
    skewness_threshold = 0.5,
    calib_ds           = calib_ds,
    rounding_mode      = 'nearest',   # no greedy overhead for comparison
)

kl_sensitivity = capture_and_compare_activations(
    fp32_model  = fp32_model,
    quant_model = model_sensitivity,
    mfcc_input  = mfcc_input,
    mslfb_input = mslfb_input,
    audio_label = "Sensitivity + Clipping (mixed bits) — 'yes'",
)

# ─────────────────────────────────────────────────────────────────────────────
#  Method 4 — Activações apenas (pesos FP32)
# ─────────────────────────────────────────────────────────────────────────────
model_act_only, _, _ = quantize_activations_only_per_layer(
    model_path      = model_path,
    calib_ds        = calib_ds,
    num_bits        = 8,
    clip_percentile = 0.999,
)

kl_act_only_8 = capture_and_compare_activations(
    fp32_model  = fp32_model,
    quant_model = model_act_only,
    mfcc_input  = mfcc_input,
    mslfb_input = mslfb_input,
    audio_label = "Activações apenas 8-bit — 'yes'",
)

# ── e também a 4-bit para ver o impacto ──────────────────────────────────────
model_act_only_4, _, _ = quantize_activations_only_per_layer(
    model_path      = model_path,
    calib_ds        = calib_ds,
    num_bits        = 4,
    clip_percentile = 0.999,
)

kl_act_only_4 = capture_and_compare_activations(
    fp32_model  = fp32_model,
    quant_model = model_act_only_4,
    mfcc_input  = mfcc_input,
    mslfb_input = mslfb_input,
    audio_label = "Activações apenas 4-bit — 'yes'",
)


# ─────────────────────────────────────────────────────────────────────────────
#  Final comparison table — all methods side by side
# ─────────────────────────────────────────────────────────────────────────────
methods = [
    ("Uniform 8-bit",        kl_uniform8),
    ("Uniform 4-bit",        kl_uniform4),
    ("Sensitivity (nearest)",kl_sensitivity),
    ("Act only 8-bit",        kl_act_only_8),
    ("Act only 4-bit",        kl_act_only_4),
]

print(f"\n{'='*90}")
print(f"  ACTIVATION DISTORTION COMPARISON — all methods — '{class_names[pred_idx]}'")
print(f"{'='*90}")
print(f"  {'Layer':<26}", end="")
for method_name, _ in methods:
    print(f"  {method_name[:18]:>18}", end="")
print()
print(f"  {'':.<26}", end="")
for _ in methods:
    print(f"  {'(Rel MSE)':>18}", end="")
print()
print("-" * 90)

# Collect per-layer rel_mse for each method
all_layers = [rec['layer'] for rec in kl_uniform8]
for layer_name in all_layers:
    print(f"  {layer_name:<26}", end="")
    for _, kl_recs in methods:
        rec = next((r for r in kl_recs if r['layer'] == layer_name), None)
        val = f"{rec['rel_mse']:.5f}" if rec else "—"
        print(f"  {val:>18}", end="")
    print()

print("-" * 90)
print(f"  {'AVERAGE':<26}", end="")
for _, kl_recs in methods:
    avg = np.mean([r['rel_mse'] for r in kl_recs])
    print(f"  {avg:>18.5f}", end="")
print()
print(f"{'='*90}")

# ─────────────────────────────────────────────────────
# 📚 12. REFERENCES
# ─────────────────────────────────────────────────────

<a id="refrences"></a>
## <center>References</center>

[1] Yu, Dong and Li Deng. “Automatic Speech Recognition: A Deep Learning Approach.” (2014).

[2] https://medium.com/analytics-vidhya/understanding-the-mel-spectrogram-fca2afa2ce53

[3] https://medium.com/linagoralabs/computing-mfccs-voice-recognition-features-on-arm-systems-dae45f016eb6

[4] https://haythamfayek.com/2016/04/21/speech-processing-for-machine-learning.html

[5] Uday Kamath, John Liu, James Whitaker. "Deep Learning for NLP and Speech Recognition" (2019)

[6] Jiang, Wenbin et al. “Speech Magnitude Spectrum Reconstruction from MFCCs Using Deep Neural Network.” Chinese Journal of Electronics 27 (2018): 393-398.

[7] Dahl, George E. et al. “Context-Dependent Pre-Trained Deep Neural Networks for Large-Vocabulary Speech Recognition.” IEEE Transactions on Audio, Speech, and Language Processing 20 (2012): 30-42.

[8] Tanveer, Muhammad Hassan et al. “Mel-spectrogram and Deep CNN Based Representation Learning from Bio-Sonar Implementation on UAVs.” 2021 International Conference on Computer, Control and Robotics (ICCCR) (2021): 220-224.

[9] Hönig, Florian et al. “Revising Perceptual Linear Prediction (PLP).” INTERSPEECH (2005).